# Research notebook

Experiments only. The feature engine, `train()` and `infer()` live in `submission.ipynb`;
this notebook imports them from the **converted** script, so every run also proves the
submission still converts.

Workflow: build the 10k cache once (it persists in `cache/`), then test ideas against it
with `cv_cols`. Only proven changes go back into `submission.ipynb`.


## 0. Import the engine (this is also the converter check)

In [2]:
!python -m crunch_convert notebook submission.ipynb _engine.py --override

warning df5a182d: line 164: column 15: global variable: found potential use of global variable `train_data` that will be commented out
warning df5a182d: line 301: column 11: global variable: found potential use of global variable `crunch_tools` that will be commented out


In [3]:
import importlib, sys
import numpy as np, pandas as pd, lightgbm as lgb, math, os, time, gc, json

sys.path.insert(0, ".")
import _engine
importlib.reload(_engine)
from _engine import *          # REGISTRY, StreamFeatures, build_dataset, ts_auc, TSRanker, LGB_PARAMS, ...
from _engine import _safe, _folded_ts_auc_table, _stride_for   # `import *` skips _-prefixed names

print(len(REGISTRY), N_FEATURES, [f.name for f in REGISTRY])   # expect 18 85

Tier 1 registered | features so far: 28
research_step / local_test defined -- see the markdown above for the 4-line workflow.
Tier 1 registered | features so far: 28
research_step / local_test defined -- see the markdown above for the 4-line workflow.
15 111 ['z2', 'e2', 'tail', 'time', 'lmom', 'wass', 'energy', 'mmd', 'entropy', 'bocpd', 'garch', 'argarch', 'tgarch', 'recres', 'hmm']


/Users/balendu/Desktop/structural/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import crunch
crunch_tools = crunch.load_notebook()
train_data, test_data = crunch_tools.load_data()
print(len(list(train_data)), "train series")

loaded crunch tools for module: <module '__main__'>

cli version: 12.0.3
available ram: 16.00 gb
available cpu: 10 core
----
data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test.reduced.parquet (106299 bytes)
data/y_test.reduced.parquet: already exists, file l

## 1. The 10k feature cache

Saved as `.npy` so it can be memory-mapped: on 16 GB the 1.8 GB matrix stays on disk and
only the slices being trained on live in RAM. Build once (~15-25 min on an M4), reuse forever.

In [5]:
CACHE = "cache"
os.makedirs(CACHE, exist_ok=True)
XP = f"{CACHE}/X_all.npy"

if os.path.exists(XP):
    X_all = np.load(XP, mmap_mode="r")
    y_all = np.load(f"{CACHE}/y_all.npy"); t_all = np.load(f"{CACHE}/t_all.npy"); g_all = np.load(f"{CACHE}/g_all.npy")
    feat_names_all = json.load(open(f"{CACHE}/names.json"))
    print("loaded cache:", X_all.shape)
else:
    t0 = time.time()
    X, y, t, _, g = build_dataset(list(train_data), subsample=False)     # full resolution
    feat_names_all = active_feature_names(ACTIVE_FAMILIES)
    assert X.shape[1] == len(feat_names_all)
    np.save(XP, X); np.save(f"{CACHE}/y_all.npy", y)
    np.save(f"{CACHE}/t_all.npy", t); np.save(f"{CACHE}/g_all.npy", g)
    json.dump(feat_names_all, open(f"{CACHE}/names.json", "w"))
    del X; gc.collect()
    X_all = np.load(XP, mmap_mode="r"); y_all, t_all, g_all = y, t, g
    print(f"extracted {X_all.shape} in {(time.time()-t0)/60:.1f} min")

loaded cache: (5036517, 85)


In [6]:
from _engine import _safe
try:
    from _engine import _CondBase, _history, _garch_grid, _SQ2      # present if the vol families are in submission.ipynb
    print("vol helpers imported from the engine")
except ImportError:
    print("vol helpers NOT in the engine -- paste the volatility-families cell instead")

vol helpers imported from the engine


## 2. Folds + the comparison harness

Three folds of series. Each validation fold is split in half: the **stop** half picks the
number of trees, the **score** half is only used for the reported TS-AUC, so a variant is
never graded on the rows it early-stopped on. `cv_cols` runs both halves (6 paired fits).

Only row *indices* are stored, never copies of the matrix -- that is what keeps this inside 16 GB.

In [7]:
P2_FOLDS = 3

def _stride_arr(t):
    s = np.full(t.shape, KEEP_EVERY[-1][1], dtype=np.int32)
    for up_to, stride in reversed(KEEP_EVERY):
        s[t < up_to] = stride
    return s

def _sorted_by_t(mask):
    idx = np.flatnonzero(mask)
    return idx[np.argsort(t_all[idx], kind="stable")]

def build_folds(seed=0):
    uniq = np.unique(g_all)
    fold_u = np.empty(uniq.size, dtype=np.int32)
    for k, part in enumerate(np.array_split(np.random.default_rng(seed).permutation(uniq.size), P2_FOLDS)):
        fold_u[part] = k
    fold_row = fold_u[np.searchsorted(uniq, g_all)]
    half_row = (np.random.default_rng(seed + 1).random(uniq.size) < 0.5)[np.searchsorted(uniq, g_all)]
    sub = (t_all % _stride_arr(t_all)) == 0          # rows extract_series(subsample=True) keeps
    folds = []
    for k in range(P2_FOLDS):
        folds.append(dict(tr=_sorted_by_t((fold_row != k) & sub),
                          st=_sorted_by_t((fold_row == k) & half_row),
                          sc=_sorted_by_t((fold_row == k) & ~half_row)))
        print(f"fold {k}: train {folds[-1]['tr'].size:,} | stop {folds[-1]['st'].size:,} | score {folds[-1]['sc'].size:,}")
    return folds

STRIDE_ALL = _stride_arr(t_all)
FOLD_IDX = build_folds(0)

fold 0: train 1,026,983 | stop 819,962 | score 843,857
fold 1: train 1,020,006 | stop 839,517 | score 851,548
fold 2: train 1,022,021 | stop 827,867 | score 853,766


In [8]:
CMP = {}

import numpy as np, lightgbm as lgb, time, gc

def fit_one(F, cols, extra=None, swap=False, num_rounds=None, patience=100, seeds=(0,)):
    """Train on F['tr'], early-stop on one validation half, score the other. Averages `seeds`."""
    si, ei = (F["sc"], F["st"]) if swap else (F["st"], F["sc"])
    cols = np.asarray(cols)
    names = [BASE_NAMES[i] for i in cols]
    Xtr = np.ascontiguousarray(np.asarray(X_all[F["tr"]])[:, cols])
    Xs  = np.ascontiguousarray(np.asarray(X_all[si])[:, cols])
    Xe  = np.ascontiguousarray(np.asarray(X_all[ei])[:, cols])
    preds, iters = [], []
    for s in seeds:
        params = dict(LGB_PARAMS); params.update(extra or {})
        params.update(objective="binary", metric="binary_logloss",
                      seed=s, bagging_seed=s + 100, feature_fraction_seed=s + 200)
        dtr = lgb.Dataset(Xtr, label=y_all[F["tr"]],
                          weight=STRIDE_ALL[F["tr"]].astype(np.float32), feature_name=names)
        dva = lgb.Dataset(Xs, label=y_all[si], reference=dtr)
        b = lgb.train(params, dtr, num_boost_round=num_rounds or NUM_ROUNDS, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(patience, verbose=False)])
        preds.append(b.predict(Xe, num_iteration=b.best_iteration))
        iters.append(b.best_iteration)
        del dtr, dva, b; gc.collect()
    auc = ts_auc(y_all[ei], t_all[ei], np.mean(preds, axis=0))
    del Xtr, Xs, Xe, preds; gc.collect()
    return auc, int(np.mean(iters))

def cv_cols(label, cols, extra=None, **kw):
    aucs, iters = [], []
    for k, F in enumerate(FOLD_IDX):
        for swap in (False, True):
            t0 = time.time()
            a, it = fit_one(F, cols, extra=extra, swap=swap, **kw)
            aucs.append(a); iters.append(it)
            print(f"  [{label}] fold {k} swap={swap}: {a:.5f} | trees {it} | {time.time()-t0:.0f}s")
    CMP[label] = np.array(aucs)
    print(f"[{label}] {len(cols)} feats | mean {CMP[label].mean():.5f} | mean trees {np.mean(iters):.0f}")

def cmp_table(base=None):
    base = base or next(iter(CMP))
    b = CMP[base]; rows = []
    for lab, a in CMP.items():
        d = a - b
        rows.append(dict(config=lab, mean=a.mean(), delta=d.mean(),
                         se=d.std(ddof=1)/np.sqrt(len(d)) if lab != base else 0.0,
                         better=f"{int((d > 0).sum())}/{len(d)}"))
    return pd.DataFrame(rows).sort_values("mean", ascending=False).round(5)

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
ALL_COLS = list(range(len(feat_names_all)))

**Decision rule for every experiment below:** adopt a change only if `delta > 2*se`
and `better` is 5/6 or 6/6. A pruning change is safe to adopt if `delta > -2*se` (no measurable loss).

## 3. Testing a new feature family

Define the family, register it, extract **only its columns** for the same 10k series,
bolt them onto the cache, rebuild the folds, and compare. `glr2` below is ready to test:
Andrews-Ploberger mean/trimmed-mean aggregation over the unknown break point, plus a
mean-shift GLR (nothing in the current 85 tests for a location change at an unknown k).

In [10]:
class GLRAggFamily(FeatureFamily):
    """Aggregations over the unknown break point k, from two prefix sums of `e`.

    No break: e ~ N(0,1). Break at k (length m, S1 = sum e, S2 = sum e^2):
        variance LLR_k = m/2 * (s2 - 1 - log s2),  s2 = S2/m
        mean     LLR_k = S1^2 / (2m)
    sup over k is the classic test; Andrews & Ploberger (1994) showed mean/exponential
    aggregation has better weighted-average power when k is unknown. bocpd_logodds is
    already the exponential one, so this adds the mean, a top-decile trimmed mean, and
    the mean-shift test (the old z/e families used FIXED windows, not a maximized k)."""
    name = "glr2"
    channels = ("e",)
    cost = "O1"          # stateful (prefix sums) -> runs every step
    version = 1
    MIN_SEG = 8
    TRIM_Q = 0.10

    def names(self):
        return ["glr_var_mean", "glr_var_trimmean", "glr_mean_max", "glr_mean_argmax"]

    def init(self, hist):
        cap = 1025
        self.P2 = np.zeros(cap, dtype=np.float64)
        self.P1 = np.zeros(cap, dtype=np.float64)
        self.K = np.arange(cap, dtype=np.float64)
        self.n = 0

    def _grow(self):
        old = self.P2.size; cap = 2 * old
        for nm in ("P2", "P1"):
            arr = np.zeros(cap, dtype=np.float64); arr[:old] = getattr(self, nm)
            setattr(self, nm, arr)
        self.K = np.arange(cap, dtype=np.float64)

    def update(self, ctx):
        x = ctx.vals["e"]
        n = self.n + 1
        if n >= self.P2.size:
            self._grow()
        self.P2[n] = self.P2[n - 1] + x * x
        self.P1[n] = self.P1[n - 1] + x
        self.n = n
        M = self.MIN_SEG
        if n < M:
            return [0.0, 0.0, 0.0, 0.0]
        nk = n - M + 1
        m = n - self.K[:nk]
        r = np.maximum((self.P2[n] - self.P2[:nk]) / m, 1e-12)
        llr_var = 0.5 * m * (r - 1.0 - np.log(r))
        s1 = self.P1[n] - self.P1[:nk]
        llr_mean = (s1 * s1) / (2.0 * m)
        ntop = max(1, int(round(self.TRIM_Q * nk)))
        top = np.partition(llr_var, nk - ntop)[nk - ntop:] if ntop < nk else llr_var
        j = int(np.argmax(llr_mean))
        return [_safe(float(llr_var.mean())), _safe(float(top.mean())),
                _safe(float(llr_mean[j])), math.log(m[j])]


if not any(f.name == "glr2" for f in REGISTRY):
    REGISTRY.append(GLRAggFamily())
print([f.name for f in REGISTRY])

['z2', 'e2', 'tail', 'time', 'lmom', 'wass', 'energy', 'mmd', 'entropy', 'bocpd', 'garch', 'argarch', 'tgarch', 'recres', 'hmm', 'glr2']


In [11]:
NEW = "glr2"
NEW_PATH = f"{CACHE}/new_{NEW}.npz"
new_names = active_feature_names({NEW})

if os.path.exists(NEW_PATH) and np.load(NEW_PATH)["X"].shape[1] != len(new_names):
    print(f"{NEW_PATH} has the wrong width -- discarding and re-extracting")
    os.remove(NEW_PATH)

if os.path.exists(NEW_PATH):
    Z = np.load(NEW_PATH); Xn = Z["X"]; new_names = [str(s) for s in Z["names"]]
else:
    t0 = time.time()
    # backend="fork": the family was registered in THIS process. loky workers re-import
    # _engine fresh, never see it, and silently return 0 columns.
    Xn, yn, tn, _, gn = build_dataset(list(train_data), subsample=False, active_families={NEW},
                                      n_jobs=8, backend="fork")
    assert Xn.shape[1] == len(new_names), f"extracted {Xn.shape[1]} columns, expected {len(new_names)}"
    assert np.array_equal(yn, y_all) and np.array_equal(tn, t_all) and np.array_equal(gn, g_all), \
        "row order differs from the cache"
    np.savez(NEW_PATH, X=Xn, names=np.array(new_names))
    print(f"{Xn.shape} in {(time.time()-t0)/60:.1f} min")
print(new_names)

['glr_var_mean', 'glr_var_trimmean', 'glr_mean_max', 'glr_mean_argmax']


**Univariate first** (1 min): strength alone, and correlation with the strongest existing
feature. Low TS-AUC + low correlation is still worth testing jointly -- `bocpd_runlen_log`
scored 0.524 alone and ended up 4th by model gain.

In [ ]:
# bolt the new columns onto the cache (in memory only; rerun cell 1 to get back to 85)
if new_names[0] not in feat_names_all:
    X_all = np.hstack([np.asarray(X_all), Xn])
    feat_names_all = list(feat_names_all) + new_names
    del Xn; gc.collect()
FOLD_IDX = build_folds(0)
idx = {n: i for i, n in enumerate(feat_names_all)}
BASE_COLS = [i for i, n in enumerate(feat_names_all) if n not in new_names]
NEW_COLS  = [idx[n] for n in new_names]
MEAN_COLS = [idx[n] for n in ("glr_mean_max", "glr_mean_argmax")]
AGG_COLS  = [idx[n] for n in ("glr_var_mean", "glr_var_trimmean")]
print(X_all.shape, len(BASE_COLS), len(NEW_COLS))

In [ ]:
cv_cols("base", BASE_COLS)

In [ ]:
cv_cols("+glr2 (all 4)", BASE_COLS + NEW_COLS)

In [ ]:
cmp_table("base")

If the family wins, split it -- the two halves test different ideas (aggregation vs mean-shift):

In [ ]:
cv_cols("+agg only", BASE_COLS + AGG_COLS)
cv_cols("+meanshift only", BASE_COLS + MEAN_COLS)
cmp_table("base")

## 3b. Literature families: `ctm`, `newma`, `bocpd2`

`glr2` is rejected (+0.00106, se 0.00117, 4/6 -- fails the `delta > 2*se` rule) and is dropped
from `REGISTRY` in the next cell. The three candidates are `lit_families.py`, pasted verbatim.

Extraction uses `backend="fork"`: the classes exist only in this kernel, fork workers inherit
them, while loky workers re-import `_engine` and silently return 0 columns.
The combined matrix is written to `cache/X_all_lit.npy` and memory-mapped, like the 85-column cache.
Run the univariate cell before the hstack cell (the hstack cell frees `Xl`).

In [13]:
"""Three new families, each from a different line of the online change-detection literature.
All assumption-light: they test the online stream against the historical calibration only
through ranks / PIT values, or not at all.

  ctm     -- conformal test martingales + e-detectors (Vovk et al. 2021 "Retrain or not
             retrain"; Shin, Ramdas & Rinaldo 2023 "E-detectors"). O(1)/step.
  newma   -- NEWMA: two EWMAs of random Fourier features at different rates (Keriven et al.).
             Model-free, no window, no reference sample. O(d)/step.
  bocpd2  -- Bayesian single-change filter with UNKNOWN mean and variance (Normal-Inverse-Gamma),
             unlike `bocpd` which fixes the mean at 0. O(t)/step.
"""
from scipy.special import ndtr as _ndtr, gammaln as _gammaln2

_EPSG = (0.3, 0.5, 0.8)                 # power-betting exponents for the e-detectors


class ConformalMartingaleFamily(FeatureFamily):
    """Testing by betting on conformal p-values.

    Under "no break" the PIT value of each new point against the historical sample is
    Uniform(0,1). Two p-value streams are used:
        p_loc   = u                        (location / tail shifts)
        p_scale = 2*min(F(e), 1-F(e))      (two-sided: small when |e| is large)
    For each stream we keep three statistics, all O(1):
        jumper -- Vovk's Simple Jumper martingale, a mixture over betting slopes
                  eps in {-1,0,1} with jump rate J (bets adapt without tuning)
        sr     -- Shiryaev-Roberts e-detector:  R_t = (1 + R_{t-1}) * e_t
        cusum  -- CUSUM-style e-detector:       C_t = max(1, C_{t-1}) * e_t
    with e_t a mixture of two-sided power bets, mean exactly 1 under the null:
        e_t = mean over g of  (g/2) * (p^(g-1) + (1-p)^(g-1)),
    which is large whenever p lands near either end of [0,1].
    SR and CUSUM restart at every step, so they need no break-point search at all.
    """
    name = "ctm"
    channels = ("e",)
    cost = "O1"
    version = 1
    JUMP = 0.01

    def names(self):
        return ["ctm_jump_loc", "ctm_sr_loc", "ctm_cusum_loc",
                "ctm_jump_scale", "ctm_sr_scale", "ctm_cusum_scale"]

    def init(self, hist):
        self.w = [[1 / 3.0, 1 / 3.0, 1 / 3.0], [1 / 3.0, 1 / 3.0, 1 / 3.0]]   # jumper weights
        self.wlog = [0.0, 0.0]                        # log wealth (overflow guard)
        self.sr = [-math.inf, -math.inf]              # log Shiryaev-Roberts (R_0 = 0)
        self.cu = [0.0, 0.0]                          # log CUSUM (C_0 = 1)

    def update(self, ctx):
        out = []
        p_loc = min(max(ctx.u, 1e-6), 1 - 1e-6)
        pe = float(_ndtr(ctx.vals["e"]))
        p_scale = min(max(2.0 * min(pe, 1.0 - pe), 1e-6), 1 - 1e-6)
        for s, p in enumerate((p_loc, p_scale)):
            J = self.JUMP
            a, b, c = self.w[s]
            m = J * (a + b + c) / 3.0
            q = 1.0 - J
            d = p - 0.5
            a = (q * a + m) * (1.0 - d)                      # eps = -1
            b = (q * b + m)                                  # eps =  0
            c = (q * c + m) * (1.0 + d)                      # eps = +1
            tot = a + b + c
            if tot <= 0.0:
                tot = 1e-300
            self.w[s] = [a / tot, b / tot, c / tot]
            self.wlog[s] += math.log(tot)

            e = 0.0
            for g in _EPSG:
                e += 0.5 * g * (p ** (g - 1.0) + (1.0 - p) ** (g - 1.0))
            e /= len(_EPSG)
            le = math.log(max(e, 1e-300))
            r = self.sr[s]
            self.sr[s] = (le if r == -math.inf else math.log1p(math.exp(min(r, 700.0))) + le
                          if r < 700.0 else r + le)           # log((1 + R) * e)
            self.cu[s] = max(0.0, self.cu[s]) + le            # log(max(1, C) * e)
            out += [_safe(self.wlog[s]), _safe(self.sr[s]), _safe(self.cu[s])]
        return out


class NEWMAFamily(FeatureFamily):
    """NEWMA (Keriven, Garreau & Poli): compare two exponentially weighted moving averages
    of the same random-feature embedding, one fast and one slow. Their difference is small
    while the distribution is stationary and jumps when it changes -- a kernel two-sample
    statistic with no window, no stored sample and no reference set. O(d) per step.
    Random Fourier features are seeded, so the features are deterministic across runs."""
    name = "newma"
    channels = ("e",)
    cost = "O1"
    version = 1
    D = 32                      # random features
    SIGMA = 1.0                 # RBF bandwidth (channel is unit-scale by construction)
    RATES = ((0.05, 0.005), (0.15, 0.015), (0.40, 0.04))    # (fast, slow) pairs

    def names(self):
        return ["newma_%d" % i for i in range(len(self.RATES))]

    def init(self, hist):
        rng = np.random.default_rng(12345)                   # fixed -> deterministic
        self.om = rng.normal(0.0, 1.0 / self.SIGMA, self.D)
        self.b = rng.uniform(0.0, 2 * math.pi, self.D)
        self.scale = math.sqrt(2.0 / self.D)
        self.zf = [np.zeros(self.D) for _ in self.RATES]
        self.zs = [np.zeros(self.D) for _ in self.RATES]
        self.n = 0

    def update(self, ctx):
        x = ctx.vals["e"]
        phi = self.scale * np.cos(self.om * x + self.b)
        self.n += 1
        out = []
        for i, (lf, ls) in enumerate(self.RATES):
            self.zf[i] += lf * (phi - self.zf[i])
            self.zs[i] += ls * (phi - self.zs[i])
            d = self.zf[i] - self.zs[i]
            out.append(_safe(math.log(float(np.dot(d, d)) + 1e-12)))
        return out


class BOCPDMeanVarFamily(FeatureFamily):
    """Bayesian single-change filter with BOTH mean and variance unknown (Normal-Inverse-Gamma
    prior), vectorized over every candidate break point. `bocpd` fixes the mean at 0, so it is
    blind to a pure location shift; this one is not. Posterior predictive is Student-t."""
    name = "bocpd2"
    channels = ("e",)
    cost = "O1"
    version = 1
    HAZARD = 1.0 / 500.0
    MU0, KAPPA0, A0, B0 = 0.0, 1.0, 2.0, 1.0

    def names(self):
        return ["bocpd2_logodds", "bocpd2_runlen_log", "bocpd2_mushift"]

    def init(self, hist):
        cap = 1024
        self.lw = np.zeros(cap)
        self.mu = np.zeros(cap); self.ka = np.zeros(cap)
        self.al = np.zeros(cap); self.be = np.zeros(cap)
        self.k = 0
        h = self.HAZARD
        self.c_keep = -math.log1p(-h)
        self.c_new = math.log(h) - math.log1p(-h)

    def _grow(self):
        old = self.lw.size; cap = 2 * old
        for nm in ("lw", "mu", "ka", "al", "be"):
            arr = np.zeros(cap); arr[:old] = getattr(self, nm)
            setattr(self, nm, arr)

    def update(self, ctx):
        x = ctx.vals["e"]
        k = self.k
        if k + 1 >= self.lw.size:
            self._grow()
        lw = self.lw
        if k:
            lw[:k] += self.c_keep
        lw[k] = self.c_new
        self.mu[k], self.ka[k], self.al[k], self.be[k] = self.MU0, self.KAPPA0, self.A0, self.B0
        k += 1
        self.k = k
        mu, ka, al, be = self.mu[:k], self.ka[:k], self.al[:k], self.be[:k]
        # Student-t predictive: df = 2a, loc = mu, scale^2 = b(k+1)/(a k)
        s2 = be * (ka + 1.0) / (al * ka)
        df = 2.0 * al
        z2 = (x - mu) ** 2 / s2
        logpred = (_gammaln2(al + 0.5) - _gammaln2(al) - 0.5 * np.log(math.pi * df * s2)
                   - (al + 0.5) * np.log1p(z2 / df))
        lw[:k] += logpred - (-0.5 * math.log(2 * math.pi) - 0.5 * x * x)
        # NIG update
        d = x - mu
        self.be[:k] = be + 0.5 * ka * d * d / (ka + 1.0)
        self.mu[:k] = mu + d / (ka + 1.0)
        self.ka[:k] = ka + 1.0
        self.al[:k] = al + 0.5
        L = float(lw[:k].max())
        wts = np.exp(lw[:k] - L)
        tot = float(wts.sum())
        logodds = L + math.log(tot)
        wts /= tot
        runlen = float(np.dot(wts, np.arange(k, 0, -1, dtype=np.float64)))
        mushift = float(np.dot(wts, self.mu[:k]))
        return [_safe(logodds), math.log(runlen), _safe(mushift)]


# ---- registration. REGISTRY is _engine's own list object: mutate it in place, never rebind.
REGISTRY[:] = [f for f in REGISTRY if f.name != "glr2"]     # glr2 rejected: +0.00106, se 0.00117, 4/6
for cls in (ConformalMartingaleFamily, NEWMAFamily, BOCPDMeanVarFamily):
    if not any(f.name == cls.name for f in REGISTRY):
        REGISTRY.append(cls())
LIT = {"ctm", "newma", "bocpd2"}
print([f.name for f in REGISTRY])

['z2', 'e2', 'tail', 'time', 'lmom', 'wass', 'energy', 'mmd', 'entropy', 'bocpd', 'garch', 'argarch', 'tgarch', 'recres', 'hmm', 'ctm', 'newma', 'bocpd2']


In [14]:
# smoke test: each family alone through StreamFeatures on 3 synthetic series
# (variance break, mean+variance break, no break), heavy-tailed like the real data
_rng = np.random.default_rng(7)
_series = []
for i in range(3):
    xh = _rng.standard_t(5, 800)
    xo = _rng.standard_t(5, 400)
    if i == 0:
        xo[200:] *= 1.8
    elif i == 1:
        xo[150:] = xo[150:] * 1.3 + 0.8
    _series.append((xh, xo))

for fam in [f for f in REGISTRY if f.name in LIT]:
    names = fam.names()
    rows, t0 = [], time.time()
    for xh, xo in _series:
        fx = StreamFeatures(xh, active_families={fam.name})
        assert fx.names == names, (fam.name, fx.names)
        for v in xo:
            r = fx.update(v)
            assert len(r) == len(names), f"{fam.name}: update() returned {len(r)} values, names() has {len(names)}"
            rows.append(r)
    R = np.asarray(rows, dtype=np.float64)
    bad = [names[c] for c in np.unique(np.nonzero(~np.isfinite(R))[1])]
    assert not bad, f"{fam.name}: non-finite values in {bad}"
    const = [names[c] for c in range(R.shape[1]) if R[:, c].std() == 0]
    assert not const, f"{fam.name}: constant column(s) {const}"
    print(f"  {fam.name:7s} OK: {len(names)} columns x {R.shape[0]} rows, all finite, none constant "
          f"| {(time.time()-t0)/R.shape[0]*1e6:.0f} us/row incl. driver")
del _series, rows, R; gc.collect()

  ctm     OK: 6 columns x 1200 rows, all finite, none constant | 15 us/row incl. driver
  newma   OK: 3 columns x 1200 rows, all finite, none constant | 15 us/row incl. driver
  bocpd2  OK: 3 columns x 1200 rows, all finite, none constant | 26 us/row incl. driver


4554

In [15]:
LIT_PATH = f"{CACHE}/new_lit.npz"
lit_names = active_feature_names(LIT)                          # REGISTRY order: ctm, newma, bocpd2
# names/shape are checked, the family BODIES are not: after editing a family, delete
# cache/new_lit.npz and cache/X_all_lit.npy by hand.

if os.path.exists(LIT_PATH):
    Z = np.load(LIT_PATH); Xl = Z["X"]; saved = [str(s) for s in Z["names"]]; del Z
    if saved != lit_names or Xl.shape != (len(y_all), len(lit_names)):
        print(f"{LIT_PATH} is stale ({Xl.shape}, {saved}) -- discarding and re-extracting")
        del Xl; gc.collect(); os.remove(LIT_PATH)
    else:
        print(f"loaded {LIT_PATH}: {Xl.shape}")

if not os.path.exists(LIT_PATH):
    t0 = time.time()
    # backend="fork": these classes exist only in this kernel. Fork workers inherit them;
    # loky workers re-import _engine, never see them, and return 0 columns.
    Xl, yl, tl, _, gl = build_dataset(list(train_data), subsample=False, active_families=LIT,
                                      n_jobs=8, backend="fork")
    lit_secs = time.time() - t0
    assert Xl.shape[1] == len(lit_names), f"extracted {Xl.shape[1]} columns, expected {len(lit_names)}"
    assert np.array_equal(yl, y_all) and np.array_equal(tl, t_all) and np.array_equal(gl, g_all), \
        "row order differs from the cache"
    del yl, tl, gl, _; gc.collect()
    np.savez(LIT_PATH, X=Xl, names=np.array(lit_names))
    print(f"extracted {Xl.shape} in {lit_secs/60:.1f} min ({lit_secs/len(y_all)*1e6*8:.1f} us/row/worker)")
print(lit_names)

loaded cache/new_lit.npz: (5036517, 12)
['ctm_jump_loc', 'ctm_sr_loc', 'ctm_cusum_loc', 'ctm_jump_scale', 'ctm_sr_scale', 'ctm_cusum_scale', 'newma_0', 'newma_1', 'newma_2', 'bocpd2_logodds', 'bocpd2_runlen_log', 'bocpd2_mushift']


In [16]:
# independent of whatever X_all currently holds (in-memory glr2 copy, the lit memmap, or
# anything rebound interactively): the reference column is read straight from the 85-column
# cache on disk, and Xl is reloaded from cache/new_lit.npz if the hstack cell already freed it
if "Xl" not in globals():
    with np.load(LIT_PATH) as Z:
        Xl = Z["X"]
_base = np.load(XP, mmap_mode="r")
_ref = np.asarray(_base[:, json.load(open(f"{CACHE}/names.json")).index("bocpd_logodds")], dtype=np.float64)
del _base

_rk = TSRanker(t_all)                                          # rank once, reuse for all 12 columns + the reference
tab = _folded_ts_auc_table(Xl, y_all, t_all, lit_names, ranker=_rk)
tab.insert(1, "family", tab.feature.map({nm: f.name for f in REGISTRY for nm in f.names()}))
tab["corr_with_bocpd"] = [abs(np.corrcoef(Xl[:, i].astype(np.float64), _ref)[0, 1]) for i in range(len(lit_names))]
print("reference: bocpd_logodds =", round(_rk.auc_from_ranks(y_all, _rk.ranks(_ref)), 5))
del _rk, _ref; gc.collect()
tab.sort_values("ts_auc_abs", ascending=False).round(4)

KeyboardInterrupt: 

In [17]:
# bolt the lit columns onto the clean 85-column base. np.hstack would hold a ~2 GB copy in
# RAM for the rest of the session, so the combined matrix is written to disk in row chunks
# and reopened memory-mapped -- X_all stays a memmap. Run the univariate cell BEFORE this
# one: Xl is freed here (afterwards the same data is X_all[:, LIT_COLS]).
XL_PATH = f"{CACHE}/X_all_lit.npy"
XL_NAMES = f"{CACHE}/names_lit.json"
base_names = json.load(open(f"{CACHE}/names.json"))          # always the clean 85, whatever ran above
X_base = np.load(XP, mmap_mode="r")
shape = (X_base.shape[0], X_base.shape[1] + len(lit_names))

ok = (os.path.exists(XL_PATH) and os.path.exists(XL_NAMES)
      and json.load(open(XL_NAMES)) == base_names + lit_names
      and np.load(XL_PATH, mmap_mode="r").shape == shape)
if not ok:
    t0 = time.time()
    nb = X_base.shape[1]
    out = np.lib.format.open_memmap(XL_PATH + ".part", mode="w+", dtype=np.float32, shape=shape)
    CH = 250_000                                               # ~100 MB per chunk
    for s in range(0, shape[0], CH):
        out[s:s + CH, :nb] = X_base[s:s + CH]
        out[s:s + CH, nb:] = Xl[s:s + CH]
    out.flush(); del out; gc.collect()
    json.dump(base_names + lit_names, open(XL_NAMES, "w"))
    os.replace(XL_PATH + ".part", XL_PATH)
    print(f"wrote {XL_PATH} {shape} in {time.time()-t0:.0f}s")
del X_base
globals().pop("Xl", None); gc.collect()

X_all = np.load(XL_PATH, mmap_mode="r")                      # replaces any in-memory X_all from the glr2 section
feat_names_all = base_names + lit_names
gc.collect()

FOLD_IDX = build_folds(0)
idx = {n: i for i, n in enumerate(feat_names_all)}
_fam = {f.name: f.names() for f in REGISTRY if f.name in LIT}
BASE_COLS   = list(range(len(base_names)))
CTM_COLS    = [idx[n] for n in _fam["ctm"]]
NEWMA_COLS  = [idx[n] for n in _fam["newma"]]
BOCPD2_COLS = [idx[n] for n in _fam["bocpd2"]]
LIT_COLS    = CTM_COLS + NEWMA_COLS + BOCPD2_COLS
print(type(X_all).__name__, X_all.shape, "| base", len(BASE_COLS), "| ctm", len(CTM_COLS),
      "| newma", len(NEWMA_COLS), "| bocpd2", len(BOCPD2_COLS))

fold 0: train 1,026,983 | stop 819,962 | score 843,857
fold 1: train 1,020,006 | stop 839,517 | score 851,548
fold 2: train 1,022,021 | stop 827,867 | score 853,766
memmap (5036517, 97) | base 85 | ctm 6 | newma 3 | bocpd2 3


In [ ]:
cv_cols("base", BASE_COLS)

In [ ]:
cv_cols("+lit (all 3)", BASE_COLS + LIT_COLS)

In [ ]:
cv_cols("+ctm", BASE_COLS + CTM_COLS)

In [ ]:
cv_cols("+newma", BASE_COLS + NEWMA_COLS)

In [ ]:
cv_cols("+bocpd2", BASE_COLS + BOCPD2_COLS)

In [ ]:
cmp_table("base")

In [ ]:
# free everything this round allocated and go back to the 85-column cache
for _v in ("Xl", "Xn", "lit_names", "new_names", "CORR", "UNIQ", "_Xs", "_ref", "_rk", "tab"):
    globals().pop(_v, None)

X_all = np.load(f"{CACHE}/X_all.npy", mmap_mode="r")
feat_names_all = json.load(open(f"{CACHE}/names.json"))
REGISTRY[:] = [f for f in REGISTRY if f.name not in ("glr2", "ctm", "newma", "bocpd2")]

for k in ("+glr2 (all 4)", "+agg only", "+meanshift only",
          "+lit (all 3)", "+ctm", "+newma", "+bocpd2", "base"):
    CMP.pop(k, None)

import gc; gc.collect()
FOLD_IDX = build_folds(0)
print(X_all.shape, len(feat_names_all), [f.name for f in REGISTRY], list(CMP))

In [18]:
class DependenceFamily(FeatureFamily):
    """Detects a change in the DEPENDENCE structure, which nothing else in the bank sees.

    The `e` channel is the residual after whitening with the AR(p) fitted on the historical
    segment, so under "no break" it is serially uncorrelated: rho_k ~ N(0, 1/m). If the break
    changes the autocorrelation (white noise -> AR(1)) while leaving the marginal variance
    alone, every variance/distribution feature stays flat and only this one moves. Equivalently:
    var(e_t - e_{t-1}) = 2*var(e)*(1 - rho_1), so this is the differenced-channel signal.

    Per window: rho_1, rho_2 as z-scores (rho*sqrt(m)), and Ljung-Box style Q = m*(rho_1^2+rho_2^2),
    which fires for either sign. Plus two EWMA rho_1 for faster reaction after a break."""
    name = "dep"
    channels = ("e",)
    cost = "O1"
    version = 1
    EW_ALPHAS = (0.02, 0.08)

    def names(self):
        out = []
        for W in WINDOWS:
            out += ["dep_rho1_w%d" % W, "dep_rho2_w%d" % W, "dep_q_w%d" % W]
        out += ["dep_rho1_ew%g" % a for a in self.EW_ALPHAS]
        return out

    def init(self, hist):
        self.s11 = [0.0] * len(self.EW_ALPHAS)     # EWMA of e_t * e_{t-1}
        self.s00 = [0.0] * len(self.EW_ALPHAS)     # EWMA of e_t^2
        self.prev = 0.0
        self.n = 0
        self.cap = max(WINDOWS)                    # own ring buffer: no channel plumbing needed
        self.buf = np.zeros(self.cap, dtype=np.float64)

    def _window(self, W):
        m = W if self.n > W else self.n
        i = (self.n - 1) % self.cap
        s = i - m + 1
        if s >= 0:
            return self.buf[s:i + 1]
        return np.concatenate([self.buf[s % self.cap:], self.buf[:i + 1]])

    def update(self, ctx):
        x = ctx.vals["e"]
        self.n += 1
        self.buf[(self.n - 1) % self.cap] = x
        out = []
        for W in WINDOWS:
            w = self._window(W)
            m = w.size
            if m < 12:
                out += [0.0, 0.0, 0.0]
                continue
            s0 = float(np.dot(w, w))
            if s0 <= 0.0:
                out += [0.0, 0.0, 0.0]
                continue
            r1 = float(np.dot(w[1:], w[:-1])) / s0
            r2 = float(np.dot(w[2:], w[:-2])) / s0
            rm = math.sqrt(m)
            out += [_safe(r1 * rm), _safe(r2 * rm), _safe(m * (r1 * r1 + r2 * r2))]

        p = self.prev
        self.prev = x
        for i, a in enumerate(self.EW_ALPHAS):
            self.s11[i] += a * (x * p - self.s11[i])
            self.s00[i] += a * (x * x - self.s00[i])
            r = self.s11[i] / self.s00[i] if self.s00[i] > 1e-12 else 0.0
            out.append(_safe(r * math.sqrt(min(self.n, 2.0 / a))))
        return out


if not any(f.name == "dep" for f in REGISTRY):
    REGISTRY.append(DependenceFamily())
print([f.name for f in REGISTRY])

['z2', 'e2', 'tail', 'time', 'lmom', 'wass', 'energy', 'mmd', 'entropy', 'bocpd', 'garch', 'argarch', 'tgarch', 'recres', 'hmm', 'ctm', 'newma', 'bocpd2', 'dep']


In [ ]:
import os, time
DEP_PATH = f"{CACHE}/new_dep.npz"
if os.path.exists(DEP_PATH):
    Z = np.load(DEP_PATH); Xd = Z["X"]; dep_names = [str(s) for s in Z["names"]]
else:
    t0 = time.time()
    Xd, yn, tn, _, gn = build_dataset(list(train_data), subsample=False, active_families={"dep"})
    assert np.array_equal(yn, y_all) and np.array_equal(tn, t_all) and np.array_equal(gn, g_all), \
        "row order differs from the cache"
    dep_names = active_feature_names({"dep"})
    np.savez(DEP_PATH, X=Xd, names=np.array(dep_names))
    print(f"{Xd.shape} in {(time.time()-t0)/60:.1f} min")
print(dep_names)

In [ ]:
tab = _folded_ts_auc_table(Xd, y_all, t_all, dep_names)
ref = np.asarray(X_all[:, feat_names_all.index("bocpd_logodds")], dtype=np.float64)
tab["corr_with_bocpd"] = [abs(np.corrcoef(Xd[:, i].astype(np.float64), ref)[0, 1]) for i in range(len(dep_names))]
print("reference: bocpd_logodds =", round(ts_auc(y_all, t_all, ref), 5))
tab.sort_values("ts_auc_abs", ascending=False).round(4)

In [ ]:
print(Xd.shape, len(dep_names), dep_names)
print([f.name for f in REGISTRY])
print(len(active_feature_names({"dep"})))

In [ ]:
cv_cols("base", BASE_COLS)

In [ ]:
import gc
if dep_names[0] not in feat_names_all:
    X_all = np.hstack([np.asarray(X_all), Xd])
    feat_names_all = list(feat_names_all) + dep_names
    del Xd; gc.collect()

FOLD_IDX = build_folds(0)
idx = {n: i for i, n in enumerate(feat_names_all)}
BASE_COLS = [i for i, n in enumerate(feat_names_all) if n not in dep_names]
DEP_COLS = [idx[n] for n in dep_names]
print(X_all.shape, "| base:", len(BASE_COLS), "| dep:", len(DEP_COLS))

In [ ]:
cv_cols("+dep", BASE_COLS + DEP_COLS)

In [ ]:
cmp_table("base")

In [20]:
class LongWindowFamily(FeatureFamily):
    """The scale statistics that win at W=256, extended to 512 and 1024.

    Scale features improve monotonically with window (e2_w32 .550 < w64 .556 < w256 .560),
    while SHAPE features go the other way (lmom_lkurt_e_w32 .559 > w64 > w256), so this
    extends only the scale side. Both statistics are exact O(1) from running sums of e and
    e^2, so the longer windows cost nothing. Also adds a slower EWMA, since ew0.02 beat ew0.08."""
    name = "long"
    channels = ("e",)
    cost = "O1"
    version = 1
    WINS = (512, 1024)
    EW = (0.005,)

    def names(self):
        out = []
        for W in self.WINS:
            out += ["e2_w%d" % W, "ftest_e_w%d" % W]
        out += ["e2_ew%g" % a for a in self.EW]
        return out

    def init(self, hist):
        self.cap = max(self.WINS)
        self.buf = np.zeros(self.cap, dtype=np.float64)
        self.n = 0
        self.ew = [0.0] * len(self.EW)
        r = hist.etab if hasattr(hist, "etab") else None
        self.vref = float(np.var(np.asarray(r)[1:-1], ddof=1)) if r is not None else 1.0
        self.nref = (len(r) - 2) if r is not None else 1000

    def _sums(self, W):
        m = W if self.n > W else self.n
        i = (self.n - 1) % self.cap
        s = i - m + 1
        w = self.buf[s:i + 1] if s >= 0 else np.concatenate([self.buf[s % self.cap:], self.buf[:i + 1]])
        return w.sum(), float(np.dot(w, w)), m

    def update(self, ctx):
        x = ctx.vals["e"]
        self.n += 1
        self.buf[(self.n - 1) % self.cap] = x
        out = []
        for W in self.WINS:
            s1, s2, m = self._sums(W)
            if m < 16:
                out += [0.0, 0.0]
                continue
            out.append(_safe((s2 / m - 1.0) / math.sqrt(2.0) * math.sqrt(m)))
            v = max((s2 - s1 * s1 / m) / (m - 1), 1e-12)
            se = math.sqrt(2.0 / (m - 1) + 2.0 / max(self.nref - 1, 2))
            out.append(_safe(math.log(v / max(self.vref, 1e-12)) / se))
        e2 = (x * x - 1.0) * 0.7071067811865476
        for i, a in enumerate(self.EW):
            self.ew[i] += a * (e2 - self.ew[i])
            out.append(_safe(self.ew[i] * math.sqrt(min(self.n, 2.0 / a))))
        return out


if not any(f.name == "long" for f in REGISTRY):
    REGISTRY.append(LongWindowFamily())
long_names = active_feature_names({"long"})
print([f.name for f in REGISTRY])
print(len(long_names), long_names)          # expect 5

['z2', 'e2', 'tail', 'time', 'lmom', 'wass', 'energy', 'mmd', 'entropy', 'bocpd', 'garch', 'argarch', 'tgarch', 'recres', 'hmm', 'ctm', 'newma', 'bocpd2', 'dep', 'long']
5 ['e2_w512', 'ftest_e_w512', 'e2_w1024', 'ftest_e_w1024', 'e2_ew0.005']


In [22]:
fam_of = {nm: f.name for f in REGISTRY for nm in f.names()}
PROD = [i for i, n in enumerate(BASE_NAMES) if n in fam_of and fam_of[n] in
        {"z2", "e2", "tail", "time", "ftest", "bartlett", "levene", "lmom",
         "sigtuk", "wass", "energy", "mmd", "entropy", "bocpd"}]
assert len(PROD) == 73, len(PROD)

NameError: name 'BASE_NAMES' is not defined

In [23]:
import numpy as np
_rng = np.random.default_rng(0)
_xh = _rng.standard_t(6, 2000)
_xo = _rng.standard_t(6, 900); _xo[450:] *= 1.6
_fx = StreamFeatures(_xh, active_families={"long"})
_R = np.array([_fx.update(v) for v in _xo])
print("shape:", _R.shape, "| finite:", np.isfinite(_R).all())
print("names:", _fx.names)
print("pre-break  mean:", _R[350:450].mean(axis=0).round(2))
print("post-break mean:", _R[800:900].mean(axis=0).round(2))
assert _R.shape[1] == len(long_names)
del _rng, _xh, _xo, _fx, _R

shape: (900, 5) | finite: True
names: ['e2_w512', 'ftest_e_w512', 'e2_w1024', 'ftest_e_w1024', 'e2_ew0.005']
pre-break  mean: [1.27 1.26 1.27 1.26 0.74]
post-break mean: [14.21  9.24 11.75  7.98 15.12]


NOW I CAN TEST FEATURES AFTER DEFINING A CLASS AND IT WILL HAPPEN FULLY ON ITS OWN, EZ PZ

In [24]:
# %% [markdown]
# ## test_family on the 111-feature base
# Replaces the ONE-CALL FAMILY TEST cell's `test_family` / `keep_family`. Same call:
#     test_family(MyFamily)              -> smoke test, extraction (cached), univariate table, CV verdict
#     keep_family("myfam")               -> adopt it: every later test runs on base111 + myfam
#     forget("myfam")                    -> free a rejected family's RAM (replaces reset_bank())
#
# The base is the shipped model after pruning: 58 cache columns (bucket 3 removed) +
# garch, argarch, tgarch, recres, hmm (gjr and wavvar removed) = 111.
# Folds use seed 11 and seeds (0, 1, 2) by default, so the base111 result from the wavvar run
# (b11 `noW`, the identical matrix) is reused and the first test costs one CV, not two.
# Every CV result is saved to cache/family_results.json and reused on re-runs.
#
# Needs from the notebook: section 0 (engine: REGISTRY, StreamFeatures, active_feature_names,
# build_dataset), section 1 (cache + train_data), section 2 (build_folds, ts_auc, STRIDE_ALL, CMP),
# and `_cv`, `_stack`, `_folded_ts_auc_table` from the ONE-CALL FAMILY TEST cell.
# Do NOT call the old reset_bank() afterwards: it resets folds to seed 0 and PROD to the old bank.

# %% [1] base111 bank
import os, gc, json, time
import numpy as np, pandas as pd

SHIPPED = ["z2", "e2", "tail", "time", "ftest", "bartlett", "levene", "lmom", "sigtuk",
           "wass", "energy", "mmd", "entropy", "bocpd",
           "garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]
ALL_EXTRA = ["garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]   # live in cache/new_<fam>.npz
BANK_EXTRA = ["garch", "argarch", "tgarch", "recres", "hmm"]                    # same order as the b11 noW run
B3_ALL = [f"{s}_e_w{W}" for s in ("ftest", "bartlett", "levene", "sigtuk", "lmom_lscale")
          for W in (32, 64, 256)]

FAM_FOLD_SEED, FAM_SEEDS = 11, (0, 1, 2)
FAM_RES_PATH = f"{CACHE}/family_results.json"

FAM_NAMES = {f.name: list(f.names()) for f in REGISTRY if f.name in SHIPPED}
assert set(FAM_NAMES) == set(SHIPPED), f"REGISTRY lacks {set(SHIPPED) - set(FAM_NAMES)} -- re-run section 0"

X_all = np.load(f"{CACHE}/X_all.npy", mmap_mode="r")
BASE_NAMES = json.load(open(f"{CACHE}/names.json"))
assert len(BASE_NAMES) == X_all.shape[1] and X_all.shape[0] == y_all.size
cache_names = [n for fam in SHIPPED if fam not in ALL_EXTRA for n in FAM_NAMES[fam]]
assert not [n for n in cache_names if n not in BASE_NAMES]

PROD = [BASE_NAMES.index(n) for n in cache_names if n not in B3_ALL]   # _stack() reads this global
PROD_NAMES = [BASE_NAMES[i] for i in PROD]
assert len(PROD) == 58, len(PROD)


def load_shipped(fam):
    want = FAM_NAMES[fam]
    with np.load(f"{CACHE}/new_{fam}.npz") as Z:
        nm = [str(s) for s in Z["names"]]
        miss = [n for n in want if n not in nm]
        assert not miss, f"{fam}: cached extraction lacks {miss} -- re-extract it with force=True"
        X = np.asarray(Z["X"], dtype=np.float32)
    X = np.ascontiguousarray(X[:, [nm.index(n) for n in want]])
    assert X.shape[0] == y_all.size and np.isfinite(X).all(), fam
    return X, want


EXTRA = {}          # family -> (array, names): the bank families plus anything tested since
KEPT111 = []        # families adopted on top of base111, in order
for fam in BANK_EXTRA:
    EXTRA[fam] = load_shipped(fam)
gc.collect()

FOLD_IDX = build_folds(FAM_FOLD_SEED)
FRES = json.load(open(FAM_RES_PATH)) if os.path.exists(FAM_RES_PATH) else {}


def _save():
    with open(FAM_RES_PATH + ".tmp", "w") as f:
        json.dump(FRES, f, indent=1)
    os.replace(FAM_RES_PATH + ".tmp", FAM_RES_PATH)


# the b11 run's noW is base111, column for column, with the same folds and seeds: reuse it
if "base111" not in FRES and os.path.exists(f"{CACHE}/b11_results.json"):
    r = json.load(open(f"{CACHE}/b11_results.json")).get("noW")
    if r and r["fold_seed"] == FAM_FOLD_SEED and r["seeds"] == list(FAM_SEEDS) and r["n_feats"] == 111:
        FRES["base111"] = dict(aucs=r["aucs"], names_hash=None, fold_seed=FAM_FOLD_SEED,
                               seeds=list(FAM_SEEDS), n_feats=111, note="reused b11 noW")
        _save()
        print("base111: reusing the b11 noW result")


def _bank(extra_fams=()):
    fams = BANK_EXTRA + KEPT111 + list(extra_fams)
    return [EXTRA[f][0] for f in fams], PROD_NAMES + [n for f in fams for n in EXTRA[f][1]]


def _base_label():
    return "base111" + "".join(f"+{k}" for k in KEPT111)


def _names_hash(names):
    return hash(tuple(names)) & 0xFFFFFFFF


def _cv_cached(label, arrays, names, seeds, refresh=False):
    r = FRES.get(label)
    ok = (r is not None and not refresh and r["fold_seed"] == FAM_FOLD_SEED and r["seeds"] == list(seeds)
          and r["n_feats"] == len(names) and r["names_hash"] in (None, _names_hash(names)))
    if ok:
        CMP[label] = np.array(r["aucs"])
        print(f"[{label}] cached | mean {CMP[label].mean():.5f}")
        return CMP[label]
    aucs = _cv(label, arrays, names, seeds=seeds)
    FRES[label] = dict(aucs=[float(a) for a in aucs], names_hash=_names_hash(names), fold_seed=FAM_FOLD_SEED,
                       seeds=list(seeds), n_feats=len(names))
    _save()
    return CMP[label]


_a, _n = _bank()
assert len(_n) == 111 and len(set(_n)) == 111
del _a, _n
print(f"base111 ready | PROD {len(PROD)} + {', '.join(BANK_EXTRA)} | folds seed {FAM_FOLD_SEED} | "
      f"seeds {FAM_SEEDS} | cached: {list(FRES)}")


# %% [2] test_family / keep_family / forget
def test_family(cls, run_cv=True, seeds=FAM_SEEDS, force=False, ref="e2_w256"):
    """Paste a family class, call this, get a verdict against base111 (+ anything kept since)."""
    name = cls.name
    t_start = time.time()
    assert name not in BANK_EXTRA, f"{name} is already in base111"
    assert name not in KEPT111, f"{name} is already kept"

    # 1. register
    REGISTRY[:] = [f for f in REGISTRY if f.name != name]
    REGISTRY.append(cls())
    names = active_feature_names({name})
    assert names, f"{name}: registry returned no feature names"
    clash = set(names) & set(_bank()[1])
    assert not clash, f"{name}: column names already in the bank: {sorted(clash)[:5]}"
    print(f"== {name}: {len(names)} features -> {names}")

    # 2. smoke test on synthetic series
    rng = np.random.default_rng(0)
    rows = []
    for k in range(3):
        xh = rng.standard_t(6, int(rng.integers(500, 2500)))
        xo = rng.standard_t(6, 900)
        xo[450:] *= (1.6 if k else 1.0)
        fx = StreamFeatures(xh, active_families={name})
        rows.append(np.array([fx.update(v) for v in xo]))
    R = np.vstack(rows)
    assert R.shape[1] == len(names), f"{name}: emits {R.shape[1]} values, names() says {len(names)}"
    assert np.isfinite(R).all(), f"{name}: non-finite output"
    dead = [names[i] for i in range(R.shape[1]) if R[:, i].std() == 0]
    print("   smoke test OK" + (f" | constant on synthetic data: {dead}" if dead else ""))

    # 3. extract over the cached series (or load)
    path = f"{CACHE}/new_{name}.npz"
    if os.path.exists(path) and not force:
        with np.load(path) as Z:
            got = [str(s) for s in Z["names"]]
            assert got == names, f"{path} holds different columns; pass force=True"
            Xn = np.ascontiguousarray(np.asarray(Z["X"], dtype=np.float32))
        print(f"   loaded cache {Xn.shape}")
    else:
        t0 = time.time()
        Xn, yn, tn, _, gn = build_dataset(list(train_data), subsample=False,
                                          active_families={name}, backend="fork")
        assert Xn.shape[1] == len(names), f"{name}: got {Xn.shape[1]} columns, expected {len(names)}"
        assert np.array_equal(yn, y_all) and np.array_equal(tn, t_all) and np.array_equal(gn, g_all), \
            "row order differs from the cache"
        assert np.isfinite(Xn).all(), "non-finite values in extraction"
        np.savez(path, X=Xn, names=np.array(names))
        Xn = np.ascontiguousarray(Xn, dtype=np.float32)
        print(f"   extracted {Xn.shape} in {(time.time() - t0) / 60:.1f} min")
    EXTRA[name] = (Xn, names)

    # 4. univariate
    tab = _folded_ts_auc_table(Xn, y_all, t_all, names)
    rv = np.asarray(X_all[:, BASE_NAMES.index(ref)], dtype=np.float64)
    tab["corr_with_" + ref] = [abs(np.corrcoef(Xn[:, i].astype(np.float64), rv)[0, 1]) for i in range(len(names))]
    print(f"   reference {ref} = {ts_auc(y_all, t_all, rv):.5f}")
    display(tab.sort_values("ts_auc_abs", ascending=False).round(4))
    del rv; gc.collect()
    if not run_cv:
        return tab

    # 5. CV: current bank vs current bank + this family (base result reused when cached)
    base_label = _base_label()
    cand_label = f"{base_label}+{name}"
    base_arrays, base_names = _bank()
    _cv_cached(base_label, base_arrays, base_names, seeds)
    cand_arrays, cand_names = _bank([name])
    _cv_cached(cand_label, cand_arrays, cand_names, seeds, refresh=force)

    # 6. verdict (unchanged rule: delta > 2*se and better on >= 5/6)
    d = CMP[cand_label] - CMP[base_label]
    se = d.std(ddof=1) / np.sqrt(d.size)
    better = int((d > 0).sum())
    keep = d.mean() > 2 * se and better >= d.size - 1
    print(f"\n{name} on {base_label} ({len(base_names)} feats): delta {d.mean():+.5f} | se {se:.5f} | "
          f"{d.mean() / se if se else 0:+.1f} sigma | better {better}/{d.size}")
    print("VERDICT:", f"KEEP  -> keep_family('{name}')" if keep else f"REJECT -> forget('{name}')")
    print(f"(total {time.time() - t_start:.0f}s)")
    return tab


def keep_family(name):
    """Adopt a tested family: every later test_family runs on base111 + the kept families."""
    assert name in EXTRA and name not in BANK_EXTRA, f"{name}: test it first"
    cand_label = f"{_base_label()}+{name}"
    assert cand_label in FRES, f"{name}: no CV result on the current base ({cand_label}) -- run test_family first"
    if name not in KEPT111:
        KEPT111.append(name)
    print(f"kept: {KEPT111} | base is now {_base_label()} ({len(_bank()[1])} feats); "
          f"its CV result is the '{cand_label}' run, reused as-is")


def forget(name):
    """Free a rejected family's array (replaces reset_bank for this workflow)."""
    assert name not in BANK_EXTRA and name not in KEPT111, f"{name} is part of the bank"
    EXTRA.pop(name, None)
    REGISTRY[:] = [f for f in REGISTRY if f.name != name]
    gc.collect()
    print(f"forgot {name} | bank: {_base_label()}")

AssertionError: REGISTRY lacks {'wavvar', 'ftest', 'gjr', 'bartlett', 'levene', 'sigtuk'} -- re-run section 0

In [ ]:
for k in [k for k in CMP if k.startswith("base")]:
    CMP.pop(k)
print("kept:", KEPT, "| cached:", list(CMP))

In [ ]:
test_family(LongWindowFamily)

In [ ]:
"""Two families from the survey's two most-cited non-moment lines of work.

  dratio  -- direct density-ratio estimation (uLSIF / RuLSIF; Kawahara & Sugiyama 2009,
             Liu, Yamada, Collier & Sugiyama, Neural Networks 2013). Estimates the RATIO
             p_test(x)/p_ref(x) with a fixed Gaussian basis and reads off the Pearson
             divergence. Sensitive to any change in distributional SHAPE, including ones
             that leave mean and variance untouched -- which no moment or scale test sees.
             Closed form, O(k) per step with incremental window sums.

  rlen    -- the FULL Bayesian online change-point filter (Adams & MacKay 2007): the
             run-length posterior with a hazard rate, allowing repeated changes, rather
             than the single-change variant in `bocpd`. The features are properties of
             that posterior -- expected/MAP run length, its entropy, P(short run) -- i.e.
             "how long has the process looked stationary", not "did it change".
"""
from scipy.special import gammaln as _gl3


class DensityRatioFamily(FeatureFamily):
    """uLSIF: fit w(x) = sum_j a_j K(x, c_j) to approximate p_test/p_ref by least squares.

    With a FIXED basis and the historical sample as the reference, the Gram matrix
    H_ij = E_ref[K_i K_j] is constant, so a = (H + lam I)^-1 h where h_i = E_test[K_i] is
    just a mean of basis functions over the test window. The Pearson divergence estimate
    PE = h.a/2 - 1/2 is then O(k) per step given running sums of the basis vector.
    Windows are exact (add the new phi, drop the one leaving); the EWMA versions need no
    buffer at all and react faster.
    """
    name = "dratio"
    channels = ("e",)
    cost = "O1"
    version = 1
    CENTERS = (-2.5, -1.75, -1.0, -0.5, 0.0, 0.5, 1.0, 1.75, 2.5)
    SIGMA = 0.7
    LAM = 0.05
    EW = (0.02, 0.005)

    def names(self):
        return (["dratio_pe_w%d" % W for W in WINDOWS]
                + ["dratio_pe_ew%g" % a for a in self.EW])

    def init(self, hist):
        c = np.asarray(self.CENTERS, dtype=np.float64)
        self.c = c
        self.k = c.size
        s2 = 2.0 * self.SIGMA ** 2
        ref = np.asarray(hist.etab, dtype=np.float64)[1:-1]         # historical PIT-normal grid
        P = np.exp(-(ref[:, None] - c[None, :]) ** 2 / s2)          # basis on the reference
        H = (P.T @ P) / max(ref.size, 1)
        self.Hinv = np.linalg.inv(H + self.LAM * np.eye(self.k))
        self.s2 = s2
        self.cap = max(WINDOWS)
        self.buf = np.zeros((self.cap, self.k), dtype=np.float64)
        self.sums = {W: np.zeros(self.k) for W in WINDOWS}
        self.ew = [np.zeros(self.k) for _ in self.EW]
        self.n = 0

    def _pe(self, h):
        a = self.Hinv @ h
        return float(h @ a) * 0.5 - 0.5

    def update(self, ctx):
        x = ctx.vals["e"]
        phi = np.exp(-(x - self.c) ** 2 / self.s2)
        i = self.n % self.cap
        self.n += 1
        out = []
        for W in WINDOWS:
            s = self.sums[W]
            s += phi
            if self.n > W:                                   # drop the one leaving the window
                s -= self.buf[(self.n - 1 - W) % self.cap]
            m = W if self.n > W else self.n
            out.append(_safe(self._pe(s / m) * math.sqrt(m)) if m >= 16 else 0.0)
        self.buf[i] = phi
        for j, a in enumerate(self.EW):
            self.ew[j] += a * (phi - self.ew[j])
            out.append(_safe(self._pe(self.ew[j]) * math.sqrt(min(self.n, 2.0 / a))))
        return out


class RunLengthFamily(FeatureFamily):
    """Adams & MacKay (2007) BOCPD: posterior over the run length since the last change.

    Unlike `bocpd` (one change, ever), this allows repeated changes via a constant hazard,
    so the filter tracks "how long the process has looked stationary". Under no break the
    run length grows with t; a break collapses it. The features read the posterior itself:
    expected and MAP run length (log), its entropy, and the mass on short runs -- quantities
    that are not monotone functions of any variance statistic.
    """
    name = "rlen"
    channels = ("e",)
    cost = "O1"
    version = 1
    HAZARD = 1.0 / 250.0
    MU0, KAPPA0, A0, B0 = 0.0, 1.0, 2.0, 1.0
    SHORT = 32

    def names(self):
        return ["rlen_mean_log", "rlen_map_log", "rlen_entropy", "rlen_p_short"]

    def init(self, hist):
        cap = 1024
        self.w = np.zeros(cap); self.w[0] = 1.0            # run-length posterior
        self.mu = np.zeros(cap); self.ka = np.zeros(cap)
        self.al = np.zeros(cap); self.be = np.zeros(cap)
        self.mu[0], self.ka[0], self.al[0], self.be[0] = self.MU0, self.KAPPA0, self.A0, self.B0
        self.k = 1

    def _grow(self):
        old = self.w.size; cap = 2 * old
        for nm in ("w", "mu", "ka", "al", "be"):
            arr = np.zeros(cap); arr[:old] = getattr(self, nm)
            setattr(self, nm, arr)

    def update(self, ctx):
        x = ctx.vals["e"]
        k = self.k
        if k + 1 >= self.w.size:
            self._grow()
        mu, ka, al, be = self.mu[:k], self.ka[:k], self.al[:k], self.be[:k]
        s2 = be * (ka + 1.0) / (al * ka)
        df = 2.0 * al
        pred = np.exp(_gl3(al + 0.5) - _gl3(al) - 0.5 * np.log(math.pi * df * s2)
                      - (al + 0.5) * np.log1p((x - mu) ** 2 / (s2 * df)))
        h = self.HAZARD
        w = self.w[:k]
        growth = w * pred * (1.0 - h)
        cp = float((w * pred * h).sum())
        # posterior update of each surviving hypothesis, then shift by one
        d = x - mu
        nbe = be + 0.5 * ka * d * d / (ka + 1.0)
        nmu = mu + d / (ka + 1.0)
        nka = ka + 1.0
        nal = al + 0.5
        self.w[1:k + 1] = growth
        self.mu[1:k + 1] = nmu; self.ka[1:k + 1] = nka
        self.al[1:k + 1] = nal; self.be[1:k + 1] = nbe
        self.w[0] = cp
        self.mu[0], self.ka[0], self.al[0], self.be[0] = self.MU0, self.KAPPA0, self.A0, self.B0
        k += 1
        self.k = k
        w = self.w[:k]
        tot = float(w.sum())
        if tot <= 0.0:
            w[:] = 0.0; w[0] = 1.0; tot = 1.0
        w /= tot
        r = np.arange(k, dtype=np.float64)
        mean_r = float(w @ r)
        map_r = float(np.argmax(w))
        ent = float(-(w[w > 0] * np.log(w[w > 0])).sum())
        p_short = float(w[:min(self.SHORT, k)].sum())
        return [math.log1p(mean_r), math.log1p(map_r), _safe(ent), _safe(p_short)]


In [ ]:
test_family(RunLengthFamily)

In [ ]:
reset_bank()

## 4. Feature pruning

Two different questions:
* dropping **columns** within a family only simplifies the model (the family is computed either way);
* dropping a whole **family** is what actually saves extraction and inference time.

Correlations are computed **after removing each t's mean**, because TS-AUC only compares
series within a time step and shared drift otherwise inflates everything.

In [ ]:
import joblib
art = joblib.load("resources/model.joblib")
gain = pd.Series(art["booster"].feature_importance("gain"), index=art["features"])
fam_of = {nm: f.name for f in REGISTRY for nm in f.names()}
rank = (pd.DataFrame({"feature": feat_names_all})
          .assign(family=lambda d: d.feature.map(fam_of), gain=lambda d: d.feature.map(gain).fillna(0.0))
          .sort_values(["family", "gain"], ascending=[True, False]))
rank["rank_in_family"] = rank.groupby("family").cumcount() + 1
display(rank.groupby("family").gain.sum().sort_values(ascending=False).to_frame("family_gain").round(0))

In [ ]:
_rs = np.random.default_rng(0)
_sel = np.sort(_rs.choice(X_all.shape[0], size=min(400_000, X_all.shape[0]), replace=False))
_Xs = np.asarray(X_all[_sel], dtype=np.float64); _ts = t_all[_sel]
_, _inv = np.unique(_ts, return_inverse=True); _cnt = np.bincount(_inv).astype(np.float64)
for j in range(_Xs.shape[1]):
    _Xs[:, j] -= (np.bincount(_inv, weights=_Xs[:, j]) / _cnt)[_inv]     # remove each t's mean
_sd = _Xs.std(axis=0); _sd[_sd == 0] = 1.0
CORR = pd.DataFrame(np.corrcoef((_Xs / _sd).T), index=feat_names_all, columns=feat_names_all)
del _Xs; gc.collect()

ordered = rank.sort_values("gain", ascending=False).feature.tolist()
rows, chosen = [], []
for f in ordered:
    mc = max((abs(CORR.loc[f, c]) for c in chosen), default=0.0)
    rows.append(dict(feature=f, family=fam_of[f], gain=float(rank.set_index("feature").gain[f]),
                     max_corr_with_better=mc,
                     closest=max(chosen, key=lambda c: abs(CORR.loc[f, c])) if chosen else ""))
    chosen.append(f)
UNIQ = pd.DataFrame(rows)
display(UNIQ.round(3).head(30))
print("low gain but distinctive (|corr| < 0.5 with everything stronger):")
display(UNIQ[UNIQ.max_corr_with_better < 0.5].round(3))

In [ ]:
def top_k_per_family(k, always=("time", "bocpd")):
    keep = rank[(rank.rank_in_family <= k) | rank.family.isin(always)]
    return [idx[n] for n in keep.feature]

def greedy_keep(thresh, always=("time", "bocpd")):
    keep = []
    for f in ordered:
        if fam_of[f] in always or all(abs(CORR.loc[f, c]) < thresh for c in keep):
            keep.append(f)
    return [idx[n] for n in keep]

def drop_families(drop):
    return [i for i, n in enumerate(feat_names_all) if fam_of[n] not in drop]

for th in (0.99, 0.95, 0.90, 0.80):
    print(f"greedy {th}: {len(greedy_keep(th))} features")
print("top-1 per family:", len(top_k_per_family(1)), "| top-2:", len(top_k_per_family(2)))

In [ ]:
cv_cols("top-2 per family", top_k_per_family(2))

In [ ]:
cv_cols("greedy 0.90", greedy_keep(0.90))

In [ ]:
# the one that actually buys runtime: drop whole families (edit using family_gain above)
cv_cols("drop weak families", drop_families({"mmd", "energy", "sigtuk", "ansari"}))
cmp_table("base")

## 5. LightGBM tuning

Trial 0 is always the current `LGB_PARAMS`, so every number is a delta against what is
submitted today. The best trial of ~30 is optimistically biased -- always re-check the top
2-3 with `cv_cols` (6 paired fits) before adopting.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
TUNE_COLS = ALL_COLS       # set to your chosen feature subset first

def objective(trial):
    extra = dict(
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.1, log=True),
        num_leaves=trial.suggest_int("num_leaves", 7, 127, log=True),
        min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 100, 5000, log=True),
        feature_fraction=trial.suggest_float("feature_fraction", 0.3, 1.0),
        bagging_fraction=trial.suggest_float("bagging_fraction", 0.5, 1.0),
        lambda_l2=trial.suggest_float("lambda_l2", 0.01, 100.0, log=True),
        extra_trees=trial.suggest_categorical("extra_trees", [False, True]),
    )
    aucs, iters, t0 = [], [], time.time()
    for k, F in enumerate(FOLD_IDX):
        a, it = fit_one(F, np.asarray(TUNE_COLS), extra=extra)
        aucs.append(a); iters.append(it)
        trial.report(float(np.mean(aucs)), k)
        if trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr("fold_aucs", [round(a, 5) for a in aucs])
    trial.set_user_attr("mean_trees", int(np.mean(iters)))
    print(f"trial {trial.number}: {np.mean(aucs):.5f} | trees {int(np.mean(iters))} | {(time.time()-t0)/60:.1f} min | {extra}")
    return float(np.mean(aucs))

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0),
                            storage=f"sqlite:///{CACHE}/optuna.db", study_name="lgb", load_if_exists=True)
study.enqueue_trial({k: LGB_PARAMS[k] for k in ("learning_rate", "num_leaves", "min_data_in_leaf",
                                                "feature_fraction", "bagging_fraction", "lambda_l2")}
                    | {"extra_trees": LGB_PARAMS.get("extra_trees", False)})

In [ ]:
study.optimize(objective, n_trials=40, timeout=3 * 3600)

In [ ]:
df = study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs"))
df = df[df.state == "COMPLETE"].sort_values("value", ascending=False)
df["delta_vs_trial0"] = df.value - df.loc[df.number == 0, "value"].item()
df.head(8)

In [ ]:
# confirm the top trials on the same folds, both halves (6 paired fits each)
for n in (int(df.number.iloc[0]), int(df.number.iloc[1])):
    cv_cols(f"trial{n}", TUNE_COLS, extra=dict(study.trials[n].params))
cmp_table("base (85)")

## 6. Post-processing (`SMOOTH_ALPHA` / `RUNNING_MAX`)

No retraining needed: replay `infer()`'s post-processing on the hold-out scores that
`train()` saves next to the model. Last time every setting lost, so both stay off.

In [ ]:
def replay_postprocess(p, t, g, smooth_alpha=0.0, running_max=False, use_stride=True):
    df = pd.DataFrame({"g": g, "t": t, "p": p}).sort_values(["g", "t"], kind="mergesort")
    if use_stride:
        strides = np.array([_stride_for(int(x)) for x in range(int(df.t.max()) + 1)])
        df["p"] = df.p.where((df.t.values % strides[df.t.values]) == 0)
        df["p"] = df.groupby("g").p.ffill()
    if smooth_alpha > 0:
        df["p"] = df.groupby("g").p.transform(lambda s: s.ewm(alpha=smooth_alpha, adjust=False).mean())
    if running_max:
        df["p"] = df.groupby("g").p.cummax()
    return df.p.clip(0.0, 1.0).reindex(range(len(p))).values

H = np.load("resources/holdout_scores.npz")
hp, hy, ht, hg = H["p"], H["y"], H["t"], H["g"]
print("raw            :", round(ts_auc(hy, ht, hp), 5))
print("with infer stride:", round(ts_auc(hy, ht, replay_postprocess(hp, ht, hg)), 5))
rows = [dict(RUNNING_MAX=rm, SMOOTH_ALPHA=a,
             ts_auc=ts_auc(hy, ht, replay_postprocess(hp, ht, hg, smooth_alpha=a, running_max=rm)))
        for rm in (False, True) for a in (0.0, 0.05, 0.1, 0.2, 0.3)]
pd.DataFrame(rows).sort_values("ts_auc", ascending=False).round(5)

## 7. Promoting a result to `submission.ipynb`

1. New family: paste the class into section 1 (after `BOCPDVarianceFamily`) and add
   `REGISTRY.append(...)` inside a `# @crunch/keep:on` block, guarded by the `if not any(...)` check.
2. New `LGB_PARAMS`: edit the `LGB_PARAMS.update(...)` block in section 4 (already inside a keep block).
3. Dropped families: remove their `REGISTRY.extend([...])` entries.
4. Re-run `python tools/check_submission.py submission.ipynb` -- it must print `ALL CHECKS PASSED`.
5. Retrain (`train(train_data, "resources")`), then upload the notebook + `resources/model.joblib`
   and create a run with **No training**.

In [ ]:
import pickle
items = [(d[0], d[1], np.asarray(list(d[2])), d[3]) for d in list(train_data)[:100]]
pickle.dump(items, open("/tmp/items.pkl", "wb"))

In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb, time, gc

def cv_ensemble(cols, seeds=(0, 1, 2, 3, 4), sizes=(1, 2, 3, 5), patience=100):
    cols = np.asarray(cols)
    names = [feat_names_all[i] for i in cols]
    out = {f"ens{k}_prob": [] for k in sizes} | {f"ens{k}_logit": [] for k in sizes if k > 1}
    for fi, F in enumerate(FOLD_IDX):
        for swap in (False, True):
            si, ei = (F["sc"], F["st"]) if swap else (F["st"], F["sc"])
            Xtr = np.ascontiguousarray(X_all[F["tr"]][:, cols])
            Xs  = np.ascontiguousarray(X_all[si][:, cols])
            Xe  = np.ascontiguousarray(X_all[ei][:, cols])
            preds = []
            for s in seeds:
                p = dict(LGB_PARAMS)
                p.update(objective="binary", metric="binary_logloss",
                         seed=s, bagging_seed=s + 100, feature_fraction_seed=s + 200)
                dtr = lgb.Dataset(Xtr, label=y_all[F["tr"]],
                                  weight=STRIDE_ALL[F["tr"]].astype(np.float32), feature_name=names)
                dva = lgb.Dataset(Xs, label=y_all[si], reference=dtr)
                t0 = time.time()
                b = lgb.train(p, dtr, num_boost_round=NUM_ROUNDS, valid_sets=[dva],
                              callbacks=[lgb.early_stopping(patience, verbose=False)])
                preds.append(b.predict(Xe, num_iteration=b.best_iteration))
                print(f"  fold {fi} swap={swap} seed {s}: trees {b.best_iteration} | {time.time()-t0:.0f}s")
                del dtr, dva, b; gc.collect()
            P = np.array(preds)
            for k in sizes:
                m = P[:k].mean(axis=0)
                out[f"ens{k}_prob"].append(ts_auc(y_all[ei], t_all[ei], m))
                if k > 1:
                    lg = np.log(np.clip(P[:k], 1e-9, 1 - 1e-9) / (1 - np.clip(P[:k], 1e-9, 1 - 1e-9)))
                    out[f"ens{k}_logit"].append(ts_auc(y_all[ei], t_all[ei], lg.mean(axis=0)))
            del Xtr, Xs, Xe, preds, P; gc.collect()
    for k, v in out.items():
        CMP[k] = np.array(v)
    return cmp_table("ens1_prob")

cv_ensemble(list(range(len(feat_names_all))))

In [ ]:
d = CMP["ens5_prob"] - CMP["ens3_prob"]
print(np.round(d, 5), f"| mean {d.mean():+.5f} | se {d.std(ddof=1)/np.sqrt(len(d)):.5f}")

7 WEIGHTED CV TEST


In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb, time, gc

def metric_weights(idx, alpha=1.0, clip=(0.05, 20.0)):
    """Row weights so each t-slice's total weight tracks the TS-AUC weight n_pos(t)*n_neg(t).
    alpha=0 -> stride only (current behaviour); alpha=1 -> full match."""
    st = STRIDE_ALL[idx].astype(np.float64)
    if alpha == 0.0:
        return st.astype(np.float32)
    tt, y = t_all[idx], y_all[idx].astype(np.float64)
    T = int(tt.max()) + 1
    npos = np.bincount(tt, weights=st * y, minlength=T)          # full-resolution counts
    nneg = np.bincount(tt, weights=st * (1.0 - y), minlength=T)
    denom = np.bincount(tt, weights=st, minlength=T)
    slice_w = np.divide(npos * nneg, np.maximum(denom, 1e-9))
    slice_w = np.where(denom > 0, slice_w, 0.0)
    m = slice_w[slice_w > 0].mean()
    r = np.where(slice_w > 0, slice_w / m, 1.0) ** alpha         # relative, mean ~1
    w = st * np.clip(r[tt], *clip)
    return (w / w.mean()).astype(np.float32)

def cv_weighted(label, cols, alpha, extra=None, patience=100):
    cols = np.asarray(cols); names = [feat_names_all[i] for i in cols]
    aucs = []
    for fi, F in enumerate(FOLD_IDX):
        for swap in (False, True):
            si, ei = (F["sc"], F["st"]) if swap else (F["st"], F["sc"])
            params = dict(LGB_PARAMS); params.update(extra or {})
            params.update(objective="binary", metric="binary_logloss")
            Xtr = np.ascontiguousarray(X_all[F["tr"]][:, cols])
            dtr = lgb.Dataset(Xtr, label=y_all[F["tr"]],
                              weight=metric_weights(F["tr"], alpha), feature_name=names)
            dva = lgb.Dataset(np.ascontiguousarray(X_all[si][:, cols]), label=y_all[si], reference=dtr)
            t0 = time.time()
            b = lgb.train(params, dtr, num_boost_round=NUM_ROUNDS, valid_sets=[dva],
                          callbacks=[lgb.early_stopping(patience, verbose=False)])
            del Xtr, dtr, dva; gc.collect()
            Xe = np.ascontiguousarray(X_all[ei][:, cols])
            aucs.append(ts_auc(y_all[ei], t_all[ei], b.predict(Xe, num_iteration=b.best_iteration)))
            print(f"  [{label}] fold {fi} swap={swap}: {aucs[-1]:.5f} | trees {b.best_iteration} | {time.time()-t0:.0f}s")
            del Xe, b; gc.collect()
    CMP[label] = np.array(aucs)
    print(f"[{label}] mean {CMP[label].mean():.5f}")

ALL = list(range(len(feat_names_all)))
# sanity: what the weights actually do
for a in (0.5, 1.0):
    w = metric_weights(FOLD_IDX[0]["tr"], a)
    print(f"alpha={a}: mean {w.mean():.2f} | p1 {np.percentile(w,1):.2f} | p99 {np.percentile(w,99):.2f} | max {w.max():.2f}")

In [ ]:
cv_weighted("w_alpha0", ALL, 0.0)     # = today's model, the baseline

In [ ]:
cv_weighted("w_alpha1", ALL, 1.0)

In [ ]:
cv_weighted("w_alpha0.5", ALL, 0.5)

In [ ]:
cmp_table("w_alpha0")

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

H = np.load("resources/holdout_scores.npz")
p, y, t = H["p"], H["y"], H["t"]

uniq = np.unique(t)
rows = []
for tv in uniq:
    m = t == tv
    yy = y[m]
    npos, nneg = int(yy.sum()), int((1 - yy).sum())
    if npos and nneg:
        rows.append((tv, roc_auc_score(yy, p[m]), npos * nneg, npos + nneg))
D = pd.DataFrame(rows, columns=["t", "auc", "w", "n"])
D["share"] = D.w / D.w.sum()

overall = (D.auc * D.w).sum() / D.w.sum()
print(f"weighted TS-AUC {overall:.5f} | slices {len(D)}")
for lo, hi in [(0, 32), (32, 128), (128, 512), (512, 10**9)]:
    b = D[(D.t >= lo) & (D.t < hi)]
    if len(b):
        print(f"  t {lo:>4}-{hi if hi < 10**8 else 'inf':>4}: auc {(b.auc*b.w).sum()/b.w.sum():.4f} "
              f"| {100*b.share.sum():5.1f}% of the metric | {len(b)} slices")

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax[0].plot(D.t, D.auc, lw=0.6, alpha=0.5)
ax[0].plot(D.t, D.auc.rolling(25, center=True, min_periods=5).mean(), lw=2)
ax[0].axhline(0.5, color="k", lw=0.5); ax[0].axhline(overall, color="r", ls="--", lw=1)
ax[0].set_ylabel("TS-AUC at t"); ax[0].set_ylim(0.4, 0.8)
ax[1].plot(D.t, D.share.cumsum(), lw=2)
ax[1].set_ylabel("cumulative share of metric"); ax[1].set_xlabel("t")
plt.tight_layout(); plt.show()

In [ ]:
fam_of = {nm: f.name for f in REGISTRY for nm in f.names()}
FAMS = sorted({fam_of[n] for n in BASE_NAMES})
ALL = list(range(len(BASE_NAMES)))
print(len(FAMS), FAMS)

cv_cols("full (85)", ALL)                      # baseline, once
for fam in FAMS:
    cols = [i for i, n in enumerate(BASE_NAMES) if fam_of[n] != fam]
    cv_cols(f"-{fam}", cols)
cmp_table("full (85)")

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

H = np.load("resources/holdout_scores.npz")
p, y, t, g = H["p"], H["y"], H["t"], H["g"]

# break time per series
uniq, inv = np.unique(g, return_inverse=True)
BIG = np.iinfo(np.int64).max
tau = np.full(uniq.size, BIG, dtype=np.int64)
m = y == 1
np.minimum.at(tau, inv[m], t[m].astype(np.int64))
age = np.where(tau[inv] < BIG, t.astype(np.int64) - tau[inv], -1)      # -1 = never breaks

AGE_BINS = [(0, 8), (8, 32), (32, 128), (128, 256), (256, 512), (512, 10**9)]

# (a) AUC by AGE, pooling over t: positives of a given age vs ALL negatives in that slice
rows = []
for lo, hi in AGE_BINS:
    num = den = 0.0
    for tv in np.unique(t):
        s = t == tv
        pos = s & (age >= lo) & (age < hi)
        neg = s & (y == 0)
        npos, nneg = int(pos.sum()), int(neg.sum())
        if npos and nneg:
            sc = np.concatenate([p[pos], p[neg]])
            lb = np.concatenate([np.ones(npos), np.zeros(nneg)])
            w = npos * nneg
            num += roc_auc_score(lb, sc) * w; den += w
    rows.append((f"{lo}-{hi if hi < 10**8 else 'inf'}", num / den if den else np.nan, den))
A = pd.DataFrame(rows, columns=["age", "auc", "weight"]); A["share"] = A.weight / A.weight.sum()
print("AUC by break age (pooled over t):"); print(A.round(4).to_string(index=False))

# (b) the decisive 2D view: does AUC fall with AGE at fixed t, or with t at fixed age?
T_BINS = [(0, 64), (64, 256), (256, 512), (512, 10**9)]
grid = pd.DataFrame(index=[f"{a}-{b if b<10**8 else 'inf'}" for a, b in AGE_BINS],
                    columns=[f"t {a}-{b if b<10**8 else 'inf'}" for a, b in T_BINS], dtype=float)
for ti, (tlo, thi) in enumerate(T_BINS):
    for ai, (alo, ahi) in enumerate(AGE_BINS):
        num = den = 0.0
        for tv in np.unique(t[(t >= tlo) & (t < thi)]):
            s = t == tv
            pos = s & (age >= alo) & (age < ahi); neg = s & (y == 0)
            npos, nneg = int(pos.sum()), int(neg.sum())
            if npos and nneg:
                sc = np.concatenate([p[pos], p[neg]]); lb = np.concatenate([np.ones(npos), np.zeros(nneg)])
                num += roc_auc_score(lb, sc) * npos * nneg; den += npos * nneg
        grid.iloc[ai, ti] = num / den if den else np.nan
print("\nAUC by age (rows) x t (cols):"); print(grid.round(3).to_string())

# (c) do the features themselves decay with age? (this is your hypothesis, directly)
POS = (age >= 0)
chk = ["e2_w256", "e2_w32", "e2_ew0.02", "e2_exp", "bocpd_logodds", "ftest_e_w256", "tail3_w64"]
idx = {n: i for i, n in enumerate(BASE_NAMES)}
sub = np.sort(np.random.default_rng(0).choice(np.flatnonzero(POS), size=min(200_000, POS.sum()), replace=False))
F = np.asarray(X_all[sub][:, [idx[c] for c in chk]], dtype=np.float64)
ab = np.digitize(age[sub], [8, 32, 128, 256, 512])
print("\nmean feature value for POSITIVE rows, by age bucket:")
print(pd.DataFrame(F, columns=chk).groupby(ab).mean().round(2).to_string())

# (d) coverage: how much data exists where the AUC drops?
cov = pd.DataFrame({"t": t, "y": y}).assign(b=lambda d: np.digitize(d.t, [64, 256, 512])) \
        .groupby("b").agg(rows=("y", "size"), pos_rate=("y", "mean"), series=("y", lambda s: len(s)))
print("\nhold-out rows by t bucket:"); print(cov.round(3).to_string())
tr = FOLD_IDX[0]["tr"]
print("training rows by t bucket:", np.bincount(np.digitize(t_all[tr], [64, 256, 512])))

In [ ]:
fam_of = {nm: f.name for f in REGISTRY for nm in f.names()}
ALL = list(range(len(BASE_NAMES)))

def without(drop):
    drop = set(drop)
    return [i for i, n in enumerate(BASE_NAMES) if fam_of[n] not in drop]

D3 = {"mood", "ansari", "iqr"}
D4 = D3 | {"fligner"}
D5 = D4 | {""}
print(len(ALL), len(without(D3)), len(without(D4)))

In [ ]:
cv_cols("base (85)", ALL)

In [ ]:
cv_cols("-3 (mood,ansari,iqr)", without(D3))

In [ ]:
cv_cols("-4 (+fligner)", without(D4))

In [ ]:
cmp_table("base (85)")

In [ ]:
n_ser = pd.DataFrame({"t": t, "g": g}).groupby("t").g.nunique()
print(D.assign(series=D.t.map(n_ser)).tail(20).round(3).to_string(index=False))

In [ ]:
fam_of = {nm: f.name for f in REGISTRY for nm in f.names()}
without = lambda drop: [i for i, n in enumerate(BASE_NAMES) if fam_of[n] not in set(drop)]
ALL = list(range(len(BASE_NAMES)))

FOLD_IDX = build_folds(7)          # fresh split: independent of everything above
CMP.clear()
cv_cols("base85 x3", ALL, seeds=(0, 1, 2))

In [ ]:
cv_cols("-4 x3", without({"mood","ansari","iqr","fligner"}), seeds=(0, 1, 2))

In [ ]:
cv_cols("-6 x3", without({"mood","ansari","iqr","fligner","sigtuk","mmd"}), seeds=(0, 1, 2))

In [ ]:
cmp_table("base85 x3")

In [ ]:
fam_of = {nm: f.name for f in REGISTRY for nm in f.names()}
without = lambda drop: [i for i, n in enumerate(BASE_NAMES) if fam_of[n] not in set(drop)]

DROP = {"mood", "ansari", "iqr", "fligner"}       # or the -6 set if it survived
TUNE_COLS = without(DROP)
TUNE_NAMES = [BASE_NAMES[i] for i in TUNE_COLS]
print(len(TUNE_COLS), "features |", sorted({fam_of[n] for n in TUNE_NAMES}))

In [ ]:
import optuna, time
optuna.logging.set_verbosity(optuna.logging.WARNING)

BEST = dict(learning_rate=0.037212, num_leaves=81, min_data_in_leaf=1860,
            feature_fraction=0.871187, bagging_fraction=0.641272,
            lambda_l2=43.84279, extra_trees=True)

def objective(trial):
    extra = dict(
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.07, log=True),
        num_leaves=trial.suggest_int("num_leaves", 31, 200, log=True),
        min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 600, 6000, log=True),
        feature_fraction=trial.suggest_float("feature_fraction", 0.5, 1.0),
        bagging_fraction=trial.suggest_float("bagging_fraction", 0.45, 0.9),
        lambda_l2=trial.suggest_float("lambda_l2", 5.0, 200.0, log=True),
        extra_trees=trial.suggest_categorical("extra_trees", [True, False]),
        min_gain_to_split=trial.suggest_float("min_gain_to_split", 0.0, 0.5),
    )
    aucs, iters, t0 = [], [], time.time()
    for k, F in enumerate(FOLD_IDX):
        a, it = fit_one(F, TUNE_COLS, extra=extra, swap=False, seeds=(0, 1, 2))
        aucs.append(a); iters.append(it)
        trial.report(float(np.mean(aucs)), k)
        if trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr("fold_aucs", [round(a, 5) for a in aucs])
    trial.set_user_attr("mean_trees", int(np.mean(iters)))
    print(f"trial {trial.number}: {np.mean(aucs):.5f} | trees {int(np.mean(iters))} | "
          f"{(time.time()-t0)/60:.1f} min | {extra}")
    return float(np.mean(aucs))

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=0, n_startup_trials=8),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=6, n_warmup_steps=0),
    storage=f"sqlite:///{CACHE}/optuna_v2.db", study_name="lgb_ens_reduced", load_if_exists=True)
study.enqueue_trial(BEST | {"min_gain_to_split": 0.0})     # trial 0 = what you ship today

In [ ]:
study.optimize(objective, n_trials=30, timeout=3 * 3600)

In [ ]:
df = study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs"))
df = df[df.state == "COMPLETE"].sort_values("value", ascending=False)
df["delta_vs_trial0"] = df.value - df.loc[df.number == 0, "value"].item()
df.head(8)

In [ ]:
FOLD_IDX = build_folds(11)
CMP.clear()
cv_cols("trial0 (current)", TUNE_COLS, extra=BEST | {"min_gain_to_split": 0.0}, seeds=(0, 1, 2))

In [ ]:
top = [int(n) for n in df.number.head(2)]
for n in top:
    cv_cols(f"trial{n}", TUNE_COLS, extra=dict(study.trials[n].params), seeds=(0, 1, 2))
cmp_table("trial0 (current)")

In [ ]:
import re, numpy as np, gc

def build_composites(cols_names):
    """Derived columns from the cached matrix: cross-window contrasts, ratios, squashes."""
    idx = {n: i for i, n in enumerate(cols_names)}
    Xb = np.asarray(X_all)                      # (N, 73) mmap slice per column below
    feats, names = [], []

    def add(name, v):
        names.append(name); feats.append(np.asarray(v, dtype=np.float32))

    # 1. cross-window contrasts of the same statistic (fast vs slow response)
    stems = {}
    for n in cols_names:
        m = re.match(r"(.+)_w(\d+)$", n)
        if m:
            stems.setdefault(m.group(1), []).append((int(m.group(2)), n))
    for stem, lst in stems.items():
        lst.sort()
        if len(lst) >= 2:
            a, b = Xb[:, idx[lst[0][1]]], Xb[:, idx[lst[-1][1]]]
            add(f"cx_{stem}_fast_minus_slow", a - b)
            add(f"cx_{stem}_ratio", a / (np.abs(b) + 1.0))

    # 2. EWMA contrasts (fast vs slow decay) on the e2 channel
    for f, s in (("e2_ew0.08", "e2_ew0.02"),):
        if f in idx and s in idx:
            add("cx_e2_ew_fast_minus_slow", Xb[:, idx[f]] - Xb[:, idx[s]])

    # 3. squashes of heavy-tailed detectors (trees split on order, but interactions see scale)
    for n in ("bocpd_logodds", "e2_exp", "e2_w256"):
        if n in idx:
            add(f"cx_tanh_{n}", np.tanh(Xb[:, idx[n]] / 10.0))

    # 4. scale-vs-shape and scale-vs-tail contrasts (same window, different mechanism)
    pairs = [("e2_w256", "lmom_lkurt_e_w256"), ("e2_w64", "lmom_lkurt_e_w64"),
             ("e2_w64", "tail3_w64"), ("e2_w256", "entropy_e_w256")]
    for a, b in pairs:
        if a in idx and b in idx:
            add(f"cx_{a}_minus_{b}", Xb[:, idx[a]] - Xb[:, idx[b]])

    # 5. normalize the detector by how long it has had to accumulate
    if "bocpd_logodds" in idx and "t_log" in idx:
        add("cx_bocpd_per_logt", Xb[:, idx["bocpd_logodds"]] / (Xb[:, idx["t_log"]] + 1.0))

    C = np.column_stack(feats)
    del Xb; gc.collect()
    return C, names

COMP, comp_names = build_composites(BASE_NAMES)
print(COMP.shape, comp_names[:8], "...")
assert np.isfinite(COMP).all()

In [ ]:
EXTRA["comp"] = (COMP, comp_names)
_cv("base", [], BASE_NAMES)
_cv("+comp", [COMP], BASE_NAMES + comp_names)

In [ ]:
cmp_table("base")

In [ ]:
class DistanceFamily(FeatureFamily):
    """AD / CvM / KS / JS / Hellinger / chi-square between the recent window and the historical
    reference. v2: cost "O1" -- v1 was declared "sorted", so the driver fed its private buffer only
    every 4th/16th point after t=128 and every window silently spanned 4-16x more real time.
    The O(W log W) computation is throttled internally on the same schedule instead."""
    name = "dist"
    channels = ("e",)
    cost = "O1"
    version = 2
    WINS = (64, 256)
    NBIN = 12

    def names(self):
        return ["dist_%s_w%d" % (s, W) for W in self.WINS
                for s in ("ad", "cvm", "ks", "js", "hell", "chi2")]

    def init(self, hist):
        ref = np.sort(np.asarray(hist.etab, dtype=np.float64)[1:-1])
        self.ref, self.nref = ref, ref.size
        self.edges = np.quantile(ref, np.arange(1, self.NBIN) / self.NBIN)
        self.p0 = 1.0 / self.NBIN
        self.cap = max(self.WINS)
        self.buf = np.zeros(self.cap, dtype=np.float64)
        self.n = 0
        self._last = [0.0] * (6 * len(self.WINS))

    def _window(self, W):
        m = W if self.n > W else self.n
        i = (self.n - 1) % self.cap
        s = i - m + 1
        return self.buf[s:i + 1] if s >= 0 else np.concatenate([self.buf[s % self.cap:], self.buf[:i + 1]])

    def update(self, ctx):
        self.buf[self.n % self.cap] = ctx.vals["e"]           # EVERY point goes into the buffer
        self.n += 1
        t = ctx.t
        stride = 1 if t < 128 else (4 if t < 512 else 16)
        if t % stride:
            return self._last                                  # hold, but the buffer stays exact
        out = []
        for W in self.WINS:
            w = self._window(W)
            m = w.size
            if m < 16:
                out += [0.0] * 6
                continue
            ws = np.sort(w)
            F = np.clip((np.searchsorted(self.ref, ws, "right") + 0.5) / (self.nref + 1.0), 1e-6, 1 - 1e-6)
            i1 = 2.0 * np.arange(1, m + 1) - 1.0
            ad = -m - float(np.sum(i1 * (np.log(F) + np.log(1.0 - F[::-1])))) / m
            cvm = 1.0 / (12.0 * m) + float(np.sum((F - i1 / (2.0 * m)) ** 2))
            k = np.arange(1, m + 1) / m
            ks = float(max(np.max(k - F), np.max(F - (k - 1.0 / m)))) * math.sqrt(m)
            cnt = np.bincount(np.searchsorted(self.edges, w, "right"), minlength=self.NBIN).astype(np.float64)
            p = cnt / m
            mmix = 0.5 * (p + self.p0)
            nz = p > 0
            js = float(0.5 * (np.sum(p[nz] * np.log(p[nz] / mmix[nz]))
                              + np.sum(self.p0 * np.log(self.p0 / mmix))))
            hell = float(np.sqrt(max(1.0 - np.sum(np.sqrt(p * self.p0)), 0.0)))
            chi2 = float(np.sum((p - self.p0) ** 2) / self.p0) * m
            out += [_safe(ad), _safe(cvm * m), _safe(ks), _safe(js * m), _safe(hell * math.sqrt(m)), _safe(chi2)]
        self._last = out
        return out

In [ ]:
print("KEPT:", KEPT)
print("baseline label will be:", "base" + (f" (+{'+'.join(KEPT)})" if KEPT else ""))
print("columns:", len(PROD) + sum(EXTRA[k][0].shape[1] for k in KEPT))    # 109

In [ ]:
for k in [k for k in CMP if k.startswith("base")]:
    CMP.pop(k)

In [ ]:
import numpy as np, gc, os

# 0. the broken dist cache must go
p = f"{CACHE}/new_dist.npz"
if os.path.exists(p):
    os.remove(p); print("removed stale", p)

# 1. restore the shipped families into the baseline
def load_fam(name, drop=()):
    Z = np.load(f"{CACHE}/new_{name}.npz")
    nm = [str(s) for s in Z["names"]]
    keep = [i for i, n in enumerate(nm) if n not in set(drop)]
    X = np.ascontiguousarray(np.asarray(Z["X"], dtype=np.float32)[:, keep]); del Z; gc.collect()
    return X, [nm[i] for i in keep]

EXTRA["garch"] = load_fam("garch", drop=("garch_persist",))
EXTRA["tgarch"] = load_fam("tgarch")
EXTRA["argarch"] = load_fam("argarch")
_nj = [str(s) for s in np.load(f"{CACHE}/new_gjr.npz")["names"]]
EXTRA["gjr"] = load_fam("gjr", drop=[n for n in _nj if "asym" in n or n.endswith("_gamma")])
for f in ("garch", "tgarch", "argarch", "gjr"):
    keep_family(f)

# 2. verify, and drop any cached baseline
n_base = len(PROD) + sum(EXTRA[k][0].shape[1] for k in KEPT)
print("KEPT:", KEPT, "| baseline columns:", n_base)
assert n_base == 109, n_base
assert _stack(FOLD_IDX[0]["st"][:3], [EXTRA[k][0] for k in KEPT]).shape[1] == 109, "_stack is not using PROD"
for k in [k for k in CMP if k.startswith("base")]:
    CMP.pop(k)

# 3. test the FIXED dist, re-extracting
test_family(DistanceFamily, force=True)

In [ ]:
reset_bank()

: 

In [ ]:
import numpy as np, json, gc
X_all = np.load(f"{CACHE}/X_all.npy", mmap_mode="r")
BASE_NAMES = json.load(open(f"{CACHE}/names.json"))
feat_names_all = list(BASE_NAMES)
y_all = np.load(f"{CACHE}/y_all.npy"); t_all = np.load(f"{CACHE}/t_all.npy"); g_all = np.load(f"{CACHE}/g_all.npy")
STRIDE_ALL = _stride_arr(t_all)
FOLD_IDX = build_folds(11)

fam_of = {nm: f.name for f in REGISTRY for nm in f.names()}
DROPPED = {"mood", "ansari", "iqr", "fligner"}
PROD = [i for i, n in enumerate(BASE_NAMES) if fam_of.get(n) not in DROPPED]
print(len(PROD), "production features")      # expect 73

In [ ]:
PROD = [i for i, n in enumerate(BASE_NAMES) if fam_of.get(n, "DROPPED") not in DROPPED and n in fam_of]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def logreg_scores(F, C=0.1, n_sub=300_000, seed=0):
    """Fit on THIS fold's training rows only; return the score for every row we need."""
    sub = np.sort(np.random.default_rng(seed).choice(F["tr"], size=min(n_sub, F["tr"].size), replace=False))
    Xs = np.asarray(X_all[sub])[:, PROD].astype(np.float64)
    sc = StandardScaler().fit(Xs)
    clf = LogisticRegression(max_iter=1000, C=C).fit(sc.transform(Xs), y_all[sub],
                                                     sample_weight=STRIDE_ALL[sub])
    del Xs; gc.collect()
    out = {}
    for k in ("tr", "st", "sc"):
        Z = np.asarray(X_all[F[k]])[:, PROD].astype(np.float64)
        out[k] = clf.decision_function(sc.transform(Z)).astype(np.float32)[:, None]
        del Z; gc.collect()
    return out

F = FOLD_IDX[0]
S = logreg_scores(F)
print("logreg alone, TS-AUC:", round(ts_auc(y_all[F["sc"]], t_all[F["sc"]], S["sc"].ravel()), 5))
print("reference: LightGBM on 73 features is ~0.615")

In [ ]:
import lightgbm as lgb, time
def cv_with_logreg(label, seeds=(0,1,2)):
    aucs = []
    for fi, F in enumerate(FOLD_IDX):
        S = logreg_scores(F)                      # refit per fold: no leakage
        for swap in (False, True):
            si_k, ei_k = ("sc", "st") if swap else ("st", "sc")
            si, ei = F[si_k], F[ei_k]
            mk = lambda rows, key: np.ascontiguousarray(
                np.hstack([np.asarray(X_all[rows])[:, PROD], S[key]]))
            names = [BASE_NAMES[i] for i in PROD] + ["logreg_score"]
            Xtr, Xs, Xe = mk(F["tr"], "tr"), mk(si, si_k), mk(ei, ei_k)
            preds = []
            for s in seeds:
                p = dict(LGB_PARAMS); p.update(objective="binary", metric="binary_logloss",
                                               seed=s, bagging_seed=s+100, feature_fraction_seed=s+200)
                dtr = lgb.Dataset(Xtr, label=y_all[F["tr"]],
                                  weight=STRIDE_ALL[F["tr"]].astype(np.float32), feature_name=names)
                b = lgb.train(p, dtr, num_boost_round=NUM_ROUNDS,
                              valid_sets=[lgb.Dataset(Xs, label=y_all[si], reference=dtr)],
                              callbacks=[lgb.early_stopping(100, verbose=False)])
                preds.append(b.predict(Xe, num_iteration=b.best_iteration)); del dtr, b; gc.collect()
            aucs.append(ts_auc(y_all[ei], t_all[ei], np.mean(preds, axis=0)))
            print(f"  [{label}] fold {fi} swap={swap}: {aucs[-1]:.5f}")
            del Xtr, Xs, Xe, preds; gc.collect()
        del S; gc.collect()
    CMP[label] = np.array(aucs); print(f"[{label}] mean {CMP[label].mean():.5f}")

cv_cols("prod73", PROD, seeds=(0,1,2))
cv_with_logreg("prod73 + logreg")
cmp_table("prod73")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

WEIGHTS = [0.5, 0.6, 0.7, 0.8, 0.9]          # weight on LightGBM

def per_t(y, t, p, min_n=20):
    rows = []
    for tv in np.unique(t):
        m = t == tv; yy = y[m]
        if yy.sum() and (1 - yy).sum() and m.sum() >= min_n:
            rows.append((tv, roc_auc_score(yy, p[m]), int(yy.sum()) * int((1 - yy).sum())))
    return pd.DataFrame(rows, columns=["t", "auc", "w"])

base = per_t(y_s, t_s, p_lgb).rename(columns={"auc": "lgb"})
M = base.merge(per_t(y_s, t_s, p_log).rename(columns={"auc": "log"})[["t", "log"]], on="t")
for w in WEIGHTS:
    col = f"w{w:.1f}"
    M = M.merge(per_t(y_s, t_s, w * p_lgb + (1 - w) * p_log).rename(columns={"auc": col})[["t", col]], on="t")

# overall, and by t bucket
print("overall TS-AUC")
print(f"  lgb    {ts_auc(y_s, t_s, p_lgb):.5f}")
print(f"  logreg {ts_auc(y_s, t_s, p_log):.5f}")
for w in WEIGHTS:
    b = ts_auc(y_s, t_s, w * p_lgb + (1 - w) * p_log)
    print(f"  {w:.1f}/{1-w:.1f}  {b:.5f}   ({b - ts_auc(y_s, t_s, p_lgb):+.5f} vs lgb)")

BUCKETS = [(0, 64), (64, 128), (128, 256), (256, 512), (512, 10**9)]
rows = []
for lo, hi in BUCKETS:
    b = M[(M.t >= lo) & (M.t < hi)]
    if not len(b):
        continue
    f = lambda c: (b[c] * b.w).sum() / b.w.sum()
    rows.append(dict(bucket=f"{lo}-{hi if hi<10**8 else 'inf'}", share=100*b.w.sum()/M.w.sum(),
                     lgb=f("lgb"), logreg=f("log"), **{f"w{w:.1f}": f(f"w{w:.1f}") for w in WEIGHTS}))
print()
print(pd.DataFrame(rows).round(4).to_string(index=False))

# five panels, one per weight
fig, axes = plt.subplots(len(WEIGHTS), 1, figsize=(11, 3 * len(WEIGHTS)), sharex=True, sharey=True)
sm = lambda s: s.rolling(25, center=True, min_periods=5).mean()
for ax, w in zip(axes, WEIGHTS):
    ax.plot(M.t, sm(M.lgb), lw=1.3, color="tab:blue", label="LightGBM")
    ax.plot(M.t, sm(M["log"]), lw=1.0, color="tab:gray", alpha=0.8, label="logistic")
    ax.plot(M.t, sm(M[f"w{w:.1f}"]), lw=1.8, color="tab:red", label=f"blend {w:.1f}/{1-w:.1f}")
    d = (M[f"w{w:.1f}"] * M.w).sum() / M.w.sum() - (M.lgb * M.w).sum() / M.w.sum()
    ax.set_title(f"LightGBM {w:.0%} + logistic {1-w:.0%}   (weighted delta {d:+.5f})", fontsize=10)
    ax.axhline(0.5, color="k", lw=0.5); ax.set_ylabel("TS-AUC")
    ax.legend(fontsize=8, loc="lower right")
axes[-1].set_xlabel("t")
plt.tight_layout(); plt.show()

# where does the blend beat lgb, weighted by metric mass?
print("\nshare of metric weight where blend > lgb:")
for w in WEIGHTS:
    m = M[f"w{w:.1f}"] > M.lgb
    print(f"  {w:.1f}/{1-w:.1f}: {100*M.w[m].sum()/M.w.sum():5.1f}%")

In [ ]:
import numpy as np, itertools, pandas as pd

def blend_t(a0, ainf, T, p_l=p_lgb, p_g=p_log, t=t_s):
    a = ainf + (a0 - ainf) * np.exp(-t / T)
    return (1.0 - a) * p_l + a * p_g          # a = weight on LOGISTIC

base = ts_auc(y_s, t_s, p_lgb)
best_fixed = max((ts_auc(y_s, t_s, (1-a) * p_lgb + a * p_log), a) for a in np.arange(0, 0.55, 0.05))
print(f"lgb {base:.5f} | best fixed a={best_fixed[1]:.2f} -> {best_fixed[0]:.5f} ({best_fixed[0]-base:+.5f})")

rows = []
for a0, ainf, T in itertools.product([0.4, 0.5, 0.6, 0.7], [0.0, 0.15, 0.25, 0.35], [30, 60, 120, 250]):
    if ainf > a0:
        continue
    s = ts_auc(y_s, t_s, blend_t(a0, ainf, T))
    rows.append(dict(a0=a0, ainf=ainf, T=T, ts_auc=s, vs_lgb=s - base, vs_fixed=s - best_fixed[0]))
R = pd.DataFrame(rows).sort_values("ts_auc", ascending=False)
print(R.head(10).round(5).to_string(index=False))
print("\npure exponential decay to zero (your version):")
print(R[R.ainf == 0.0].head(4).round(5).to_string(index=False))

In [ ]:
import numpy as np, lightgbm as lgb, gc
from scipy.special import expit

def cv_logreg_blend(label, a=0.4, seeds=(0, 1, 2)):
    """LightGBM (3 seeds) blended with a per-fold logistic model: (1-a)*lgb + a*logreg."""
    aucs = []
    nm = [BASE_NAMES[i] for i in PROD]
    for fi, F in enumerate(FOLD_IDX):
        S = logreg_scores(F)                       # refit on this fold's training rows only
        for swap in (False, True):
            si_k, ei_k = ("sc", "st") if swap else ("st", "sc")
            si, ei = F[si_k], F[ei_k]
            Xtr = np.ascontiguousarray(np.asarray(X_all[F["tr"]])[:, PROD])
            Xs  = np.ascontiguousarray(np.asarray(X_all[si])[:, PROD])
            Xe  = np.ascontiguousarray(np.asarray(X_all[ei])[:, PROD])
            pr = []
            for s in seeds:
                p = dict(LGB_PARAMS); p.update(objective="binary", metric="binary_logloss",
                                               seed=s, bagging_seed=s+100, feature_fraction_seed=s+200)
                dtr = lgb.Dataset(Xtr, label=y_all[F["tr"]],
                                  weight=STRIDE_ALL[F["tr"]].astype(np.float32), feature_name=nm)
                b = lgb.train(p, dtr, num_boost_round=NUM_ROUNDS,
                              valid_sets=[lgb.Dataset(Xs, label=y_all[si], reference=dtr)],
                              callbacks=[lgb.early_stopping(100, verbose=False)])
                pr.append(b.predict(Xe, num_iteration=b.best_iteration)); del dtr, b; gc.collect()
            pl = np.mean(pr, axis=0)
            pg = expit(S[ei_k].ravel())
            aucs.append(ts_auc(y_all[ei], t_all[ei], (1 - a) * pl + a * pg))
            print(f"  [{label}] fold {fi} swap={swap}: {aucs[-1]:.5f}")
            del Xtr, Xs, Xe, pr; gc.collect()
        del S; gc.collect()
    CMP[label] = np.array(aucs)
    print(f"[{label}] mean {CMP[label].mean():.5f}")

FOLD_IDX = build_folds(11); CMP.clear()
cv_cols("prod73", PROD, seeds=(0, 1, 2))

In [ ]:
cv_logreg_blend("prod73 + logreg a=0.4", a=0.4)

In [ ]:
cmp_table("prod73")

In [ ]:
import torch, torch.nn as nn, numpy as np, gc
from sklearn.preprocessing import StandardScaler

def fit_mlp(F, hidden=(128, 64), epochs=10, bs=8192, lr=1e-3, wd=1e-4, seed=0):
    torch.manual_seed(seed)
    dev = "mps" if torch.backends.mps.is_available() else "cpu"
    sub = F["tr"]
    sc = StandardScaler().fit(np.asarray(X_all[sub[::5]])[:, PROD])
    net = nn.Sequential(nn.Linear(len(PROD), hidden[0]), nn.ReLU(), nn.Dropout(0.2),
                        nn.Linear(hidden[0], hidden[1]), nn.ReLU(), nn.Dropout(0.2),
                        nn.Linear(hidden[1], 1)).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
    lossf = nn.BCEWithLogitsLoss(reduction="none")
    rng = np.random.default_rng(seed)
    for ep in range(epochs):
        perm = rng.permutation(sub.size)
        tot = 0.0
        for i in range(0, sub.size, bs):
            rows = np.sort(sub[perm[i:i+bs]])
            xb = torch.tensor(sc.transform(np.asarray(X_all[rows])[:, PROD]), dtype=torch.float32, device=dev)
            yb = torch.tensor(y_all[rows], dtype=torch.float32, device=dev)[:, None]
            wb = torch.tensor(STRIDE_ALL[rows], dtype=torch.float32, device=dev)[:, None]
            opt.zero_grad()
            l = (lossf(net(xb), yb) * wb).mean()
            l.backward(); opt.step(); tot += float(l) * rows.size
        print(f"    epoch {ep}: loss {tot/sub.size:.5f}")
    net.eval()

    def score(rows):
        out = []
        with torch.no_grad():
            for i in range(0, rows.size, 100_000):
                r = rows[i:i+100_000]
                xb = torch.tensor(sc.transform(np.asarray(X_all[r])[:, PROD]), dtype=torch.float32, device=dev)
                out.append(torch.sigmoid(net(xb)).cpu().numpy().ravel())
        return np.concatenate(out)
    return score

F = FOLD_IDX[0]
sc_fn = fit_mlp(F)
p_mlp = sc_fn(F["sc"])
print("MLP alone :", round(ts_auc(y_all[F["sc"]], t_all[F["sc"]], p_mlp), 5))
print("LightGBM  :", round(ts_auc(y_all[F["sc"]], t_all[F["sc"]], p_lgb), 5))
print("logistic  :", round(ts_auc(y_all[F["sc"]], t_all[F["sc"]], p_log), 5))
print("spearman vs lgb:", pd.Series(p_mlp).corr(pd.Series(p_lgb), method="spearman").round(4))

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# needs from the previous cells: p_lgb, p_mlp, p_log (optional), y_s, t_s
ei = FOLD_IDX[0]["sc"]
y_s, t_s = y_all[ei], t_all[ei]

def per_t(y, t, p, min_n=20):
    rows = []
    for tv in np.unique(t):
        m = t == tv; yy = y[m]
        if yy.sum() and (1 - yy).sum() and m.sum() >= min_n:
            rows.append((tv, roc_auc_score(yy, p[m]), int(yy.sum()) * int((1 - yy).sum())))
    return pd.DataFrame(rows, columns=["t", "auc", "w"])

series = {"lgb": p_lgb, "mlp": p_mlp}
if "p_log" in dir():
    series["log"] = p_log

M = per_t(y_s, t_s, series["lgb"]).rename(columns={"auc": "lgb"})
for k, p in series.items():
    if k != "lgb":
        M = M.merge(per_t(y_s, t_s, p).rename(columns={"auc": k})[["t", k]], on="t")
for a in (0.2, 0.3, 0.4):
    col = f"bl{a:.1f}"
    M = M.merge(per_t(y_s, t_s, (1 - a) * p_lgb + a * p_mlp).rename(columns={"auc": col})[["t", col]], on="t")

wavg = lambda d, c: (d[c] * d.w).sum() / d.w.sum()
print("overall TS-AUC")
for k in series:
    print(f"  {k:4s} {ts_auc(y_s, t_s, series[k]):.5f}")
for a in (0.2, 0.3, 0.4):
    v = ts_auc(y_s, t_s, (1 - a) * p_lgb + a * p_mlp)
    print(f"  lgb+mlp {1-a:.1f}/{a:.1f}: {v:.5f}  ({v - ts_auc(y_s, t_s, p_lgb):+.5f})")

print("\nby t bucket")
rows = []
for lo, hi in [(0, 64), (64, 128), (128, 256), (256, 512), (512, 10**9)]:
    b = M[(M.t >= lo) & (M.t < hi)]
    if len(b):
        rows.append(dict(bucket=f"{lo}-{hi if hi<10**8 else 'inf'}", share=100*b.w.sum()/M.w.sum(),
                         **{c: wavg(b, c) for c in M.columns if c not in ("t", "w")}))
print(pd.DataFrame(rows).round(4).to_string(index=False))

print("\nspearman with lgb:", {k: round(pd.Series(p).corr(pd.Series(p_lgb), method='spearman'), 4)
                              for k, p in series.items() if k != "lgb"})

sm = lambda s: s.rolling(25, center=True, min_periods=5).mean()
fig, ax = plt.subplots(3, 1, figsize=(11, 10), sharex=True)

# 1. levels
for k, c in [("lgb", "tab:blue"), ("mlp", "tab:green"), ("log", "tab:gray")]:
    if k in M:
        ax[0].plot(M.t, sm(M[k]), lw=1.6, color=c, label={"lgb": "LightGBM", "mlp": "MLP", "log": "logistic"}[k])
ax[0].plot(M.t, sm(M["bl0.3"]), lw=1.4, color="tab:red", ls="--", label="blend 0.7/0.3")
ax[0].axhline(0.5, color="k", lw=0.5); ax[0].set_ylabel("TS-AUC at t"); ax[0].legend(fontsize=8)

# 2. differences vs lgb -- the turning-point test
for k, c in [("mlp", "tab:green"), ("log", "tab:gray")]:
    if k in M:
        ax[1].plot(M.t, sm(M[k] - M.lgb), lw=1.6, color=c, label=f"{k} − lgb")
ax[1].plot(M.t, sm(M["bl0.3"] - M.lgb), lw=1.4, color="tab:red", ls="--", label="blend − lgb")
ax[1].axhline(0, color="k", lw=0.5); ax[1].set_ylabel("difference"); ax[1].legend(fontsize=8)

# 3. z-scored curves: same shape or different?
for k, c in [("lgb", "tab:blue"), ("mlp", "tab:green"), ("log", "tab:gray")]:
    if k in M:
        s = sm(M[k])
        ax[2].plot(M.t, (s - s.mean()) / s.std(), lw=1.4, color=c, label=k)
ax[2].axhline(0, color="k", lw=0.5); ax[2].set_ylabel("z-scored curve"); ax[2].set_xlabel("t"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

# how similar are the shapes, numerically?
print("\ncorrelation of per-t AUC curves (shape similarity):")
for k in ("mlp", "log"):
    if k in M:
        print(f"  {k} vs lgb: {M[k].corr(M.lgb):.3f}")

In [ ]:
class RawMeanFamily(FeatureFamily):
    """Location tests on the RAW series, before PIT and before AR whitening.

    Every location family in the original bank (z, e, welch, mwu, contrast, slope) worked on
    the PIT-transformed value or the AR(3) residual, where a level shift is partly absorbed by
    the AR fit and compressed by the rank transform. This reads ctx.zr = (x - mu_hist)/sd_hist
    directly, so a mean break arrives undiminished.

    Per window: the standardized mean (z-test vs the historical mean) and a Welch-style t that
    also uses the window's own variance, so it is not fooled when the variance moves at the same
    time. Plus one-sided CUSUM detectors for persistent drift, and an expanding-mean z-test."""
    name = "rawmu"
    channels = ("z",)
    cost = "O1"
    version = 1
    WINS = (32, 128, 512)
    DRIFTS = (0.25, 0.5)

    def names(self):
        out = []
        for W in self.WINS:
            out += ["rawmu_z_w%d" % W, "rawmu_t_w%d" % W]
        out += ["rawmu_z_exp", "rawmu_ew0.02"]
        out += ["rawmu_cusum_up%g" % d for d in self.DRIFTS]
        out += ["rawmu_cusum_dn%g" % d for d in self.DRIFTS]
        return out

    def init(self, hist):
        self.cap = max(self.WINS)
        self.buf = np.zeros(self.cap, dtype=np.float64)
        self.n = 0
        self.s1 = 0.0
        self.ew = 0.0
        self.up = [0.0] * len(self.DRIFTS)
        self.dn = [0.0] * len(self.DRIFTS)

    def _window(self, W):
        m = W if self.n > W else self.n
        i = (self.n - 1) % self.cap
        s = i - m + 1
        return self.buf[s:i + 1] if s >= 0 else np.concatenate([self.buf[s % self.cap:], self.buf[:i + 1]])

    def update(self, ctx):
        r = float(ctx.zr)                      # (x - mu_hist) / sd_hist
        self.n += 1
        self.buf[(self.n - 1) % self.cap] = r
        self.s1 += r
        self.ew += 0.02 * (r - self.ew)

        out = []
        for W in self.WINS:
            w = self._window(W)
            m = w.size
            if m < 8:
                out += [0.0, 0.0]
                continue
            mean = float(w.mean())
            out.append(_safe(mean * math.sqrt(m)))
            se = math.sqrt(max(float(w.var(ddof=1)), 1e-12) / m)
            out.append(_safe(mean / se))
        out.append(_safe(self.s1 / self.n * math.sqrt(self.n)))
        out.append(_safe(self.ew * math.sqrt(min(self.n, 100.0))))
        for j, d in enumerate(self.DRIFTS):
            self.up[j] = max(0.0, self.up[j] + r - d)
            self.dn[j] = max(0.0, self.dn[j] - r - d)
        out += [_safe(v) for v in self.up] + [_safe(v) for v in self.dn]
        return out


if not any(f.name == "rawmu" for f in REGISTRY):
    REGISTRY.append(RawMeanFamily())
print([f.name for f in REGISTRY])

In [ ]:
test_family(RawMeanFamily)

In [ ]:
reset_bank()

In [ ]:
# =====================================================================
#  SUBSPACE BLEND: members trained on disjoint feature groups
#  prints (a) solo TS-AUC per member, (b) every blending rule, (c) verdict
# =====================================================================
import numpy as np, pandas as pd, lightgbm as lgb, itertools, time, gc
from scipy.optimize import nnls

fam_of = {nm: f.name for f in REGISTRY for nm in f.names()}

GROUPS = {                                   # 'time' is added to every member
    "scale": {"e2", "z2", "ftest", "bartlett", "levene"},
    "shape": {"lmom", "entropy", "wass", "energy", "mmd", "tail"},
    "seq":   {"sigtuk", "bocpd"},
}
def cols_for(fams):
    keep = set(fams) | {"time"}
    return [i for i, n in enumerate(BASE_NAMES) if fam_of.get(n) in keep]

MEMBERS = {k: cols_for(v) for k, v in GROUPS.items()}
MEMBERS["full"] = [i for i, n in enumerate(BASE_NAMES) if n in fam_of]      # the 73 you ship
for k, c in MEMBERS.items():
    print(f"{k:6s}: {len(c):3d} features")
assert len({tuple(c) for c in MEMBERS.values()}) == len(MEMBERS)

def fit_member(F, cols, si, ei, seeds=(0, 1, 2)):
    cols = np.asarray(cols); nm = [BASE_NAMES[i] for i in cols]
    Xtr = np.ascontiguousarray(np.asarray(X_all[F["tr"]])[:, cols])
    Xs  = np.ascontiguousarray(np.asarray(X_all[si])[:, cols])
    Xe  = np.ascontiguousarray(np.asarray(X_all[ei])[:, cols])
    pr = []
    for s in seeds:
        p = dict(LGB_PARAMS); p.update(objective="binary", metric="binary_logloss",
                                       seed=s, bagging_seed=s + 100, feature_fraction_seed=s + 200)
        dtr = lgb.Dataset(Xtr, label=y_all[F["tr"]],
                          weight=STRIDE_ALL[F["tr"]].astype(np.float32), feature_name=nm)
        b = lgb.train(p, dtr, num_boost_round=NUM_ROUNDS,
                      valid_sets=[lgb.Dataset(Xs, label=y_all[si], reference=dtr)],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        pr.append(b.predict(Xe, num_iteration=b.best_iteration))
        del dtr, b; gc.collect()
    del Xtr, Xs, Xe; gc.collect()
    return np.mean(pr, axis=0)

# ---- collect per-fold-half predictions once, then evaluate every rule offline
P, meta = [], []
for fi, F in enumerate(FOLD_IDX):
    for swap in (False, True):
        si, ei = (F["sc"], F["st"]) if swap else (F["st"], F["sc"])
        t0 = time.time()
        preds = {k: fit_member(F, c, si, ei) for k, c in MEMBERS.items()}
        P.append(preds); meta.append((ei, y_all[ei], t_all[ei]))
        solo = {k: ts_auc(y_all[ei], t_all[ei], v) for k, v in preds.items()}
        print(f"fold {fi} swap={swap} | " + " | ".join(f"{k} {v:.5f}" for k, v in solo.items())
              + f" | {time.time()-t0:.0f}s")

def score(rule):
    return np.array([ts_auc(y, t, rule(pr)) for pr, (_, y, t) in zip(P, meta)])

# ---- (a) solo
print("\nSOLO TS-AUC (mean over 6 fold-halves)")
solo = {k: score(lambda pr, k=k: pr[k]) for k in MEMBERS}
for k, v in sorted(solo.items(), key=lambda kv: -kv[1].mean()):
    print(f"  {k:6s} {v.mean():.5f}")

# ---- pairwise decorrelation (the thing that decides whether blending can work)
print("\nspearman with 'full' (per fold-half mean)")
for k in MEMBERS:
    if k != "full":
        r = np.mean([pd.Series(pr[k]).corr(pd.Series(pr["full"]), method="spearman") for pr in P])
        print(f"  {k:6s} {r:.4f}")

# ---- (b) blending rules
RES = {"full (baseline)": solo["full"]}
subs = [k for k in MEMBERS if k != "full"]
for r in range(2, len(subs) + 1):
    for combo in itertools.combinations(subs, r):
        RES["equal " + "+".join(combo)] = score(lambda pr, c=combo: np.mean([pr[k] for k in c], axis=0))
for r in range(1, len(subs) + 1):
    for combo in itertools.combinations(subs, r):
        for wf in (0.5, 0.7):
            lab = f"full{wf:.1f}+" + "+".join(combo)
            RES[lab] = score(lambda pr, c=combo, wf=wf:
                             wf * pr["full"] + (1 - wf) * np.mean([pr[k] for k in c], axis=0))

# fitted non-negative weights, chosen on the OTHER half of each fold (no self-selection)
keys = list(MEMBERS)
def fitted(pr_fit, y_f, pr_ap):
    A = np.column_stack([pr_fit[k] for k in keys])
    w, _ = nnls(A, y_f.astype(np.float64))
    w = w / (w.sum() + 1e-12)
    return np.column_stack([pr_ap[k] for k in keys]) @ w, w
fit_scores, fit_w = [], []
for i in range(0, len(P), 2):                       # halves of the same fold: fit on one, apply to other
    for a, b in ((i, i + 1), (i + 1, i)):
        p_b, w = fitted(P[a], meta[a][1], P[b])
        fit_scores.append(ts_auc(meta[b][1], meta[b][2], p_b)); fit_w.append(w)
RES["fitted nnls"] = np.array(fit_scores)
print("\nmean fitted weights:", dict(zip(keys, np.round(np.mean(fit_w, axis=0), 3))))

# ---- (c) verdict
base = RES["full (baseline)"]
rows = []
for lab, v in RES.items():
    d = v - base
    se = d.std(ddof=1) / np.sqrt(len(d)) if lab != "full (baseline)" else 0.0
    rows.append(dict(config=lab, mean=v.mean(), delta=d.mean(), se=se,
                     sigma=(d.mean() / se if se else 0.0), better=f"{int((d > 0).sum())}/{len(d)}"))
T = pd.DataFrame(rows).sort_values("mean", ascending=False).round(5)
print("\n" + T.to_string(index=False))
win = T[(T.delta > 2 * T.se) & (T.better.str.split("/").str[0].astype(int) >= 5)]
print("\nVERDICT:", ("KEEP -> " + win.iloc[0].config) if len(win) else "REJECT (nothing clears 2 sigma with 5/6)")

In [ ]:
class GarchFamily(FeatureFamily):
    """GARCH(1,1) refits the NULL MODEL rather than adding another statistic.

    Every feature in the bank treats the historical segment as iid: a quiet stretch followed by
    a turbulent one looks like a variance break to e2/ftest/bocpd even when nothing changed.
    GARCH explains that clustering instead of flagging it:
        h_t = omega + alpha*eps_{t-1}^2 + beta*h_{t-1};   nu_t = eps_t / sqrt(h_t)
    Parameters are fit by MLE on the historical segment with variance targeting (omega pinned so
    the model's unconditional variance matches the sample), leaving a stable 2-D search.
    Under no break nu is iid with unit variance EVEN IF volatility clusters.

    v2: fits on the full ORDERED history (hist.x_series) -- v1 read hist.tail_x, which holds only
    the last AR_P values, so every fit ran on 3 points. Also warms the filter through the history
    so the first online step starts at the conditional variance, not the unconditional one."""
    name = "garch"
    channels = ("z",)
    cost = "O1"
    version = 2
    WINS = (32, 128, 512)
    EW = (0.02,)

    @staticmethod
    def _nll(al, be, x2, v):
        om = v * (1.0 - al - be)
        if om <= 1e-12:
            return 1e18
        h, s = v, 0.0
        for e2 in x2:
            s += math.log(h) + e2 / h
            h = om + al * e2 + be * h
            if h <= 1e-12:
                return 1e18
        return s

    def _fit(self, x):
        v = max(float(np.var(x)) if x.size > 8 else 1.0, 1e-8)
        x2 = (np.asarray(x, dtype=np.float64) ** 2)[-600:]
        best, bp = None, (0.0, 0.0)
        for al in (0.03, 0.06, 0.10, 0.15, 0.22, 0.30):
            for be in (0.0, 0.40, 0.65, 0.80, 0.88, 0.93):
                if al + be > 0.99:
                    continue
                f = self._nll(al, be, x2, v)
                if np.isfinite(f) and (best is None or f < best):
                    best, bp = f, (al, be)
        al, be = bp
        for step in (0.04, 0.015):
            for da in (-step, 0.0, step):
                for db in (-step, 0.0, step):
                    a2, b2 = al + da, be + db
                    if a2 < 0 or b2 < 0 or a2 + b2 > 0.99:
                        continue
                    f = self._nll(a2, b2, x2, v)
                    if f < best:
                        best, al, be = f, a2, b2
        return float(v * (1.0 - al - be)), float(al), float(be), float(v)

    def names(self):
        out = []
        for W in self.WINS:
            out += ["garch_nu2_w%d" % W, "garch_gap_w%d" % W]
        out += ["garch_nu2_ew%g" % a for a in self.EW]
        return out + ["garch_logh", "garch_persist", "garch_nu2_exp"]

    def init(self, hist):
        xs = getattr(hist, "x_series", None)
        assert xs is not None and xs.size >= 50, \
            "HistoricalRef.x_series missing -- run the HistoricalRef patch cell first"
        self.mu = float(hist.mu)
        self.sd = float(hist.sd) if hist.sd > 0 else 1.0
        eps_h = np.asarray(xs, dtype=np.float64) - self.mu
        self.om, self.al, self.be, self.v0 = self._fit(eps_h)          # fit on the REAL history
        h = self.v0
        for e in eps_h:                                                # warm the filter through it
            h = self.om + self.al * e * e + self.be * h
        self.h = max(h, 1e-12)
        self.persist = self.al + self.be
        self.cap = max(self.WINS)
        self.nu2 = np.zeros(self.cap, dtype=np.float64)
        self.gap = np.zeros(self.cap, dtype=np.float64)
        self.n = 0
        self.s1 = 0.0
        self.ew = [0.0] * len(self.EW)

    def _win(self, a, W):
        m = W if self.n > W else self.n
        i = (self.n - 1) % self.cap
        s = i - m + 1
        return a[s:i + 1] if s >= 0 else np.concatenate([a[s % self.cap:], a[:i + 1]])

    def update(self, ctx):
        eps = float(ctx.zr) * self.sd
        h = max(self.h, 1e-12)
        nu2 = min(eps * eps / h, 100.0)
        self.h = self.om + self.al * eps * eps + self.be * h
        i = self.n % self.cap
        self.n += 1
        self.nu2[i] = (nu2 - 1.0) * 0.7071067811865476
        self.gap[i] = math.log(max(eps * eps, 1e-12) / h) if eps != 0.0 else 0.0
        self.s1 += self.nu2[i]

        out = []
        for W in self.WINS:
            a = self._win(self.nu2, W); b = self._win(self.gap, W)
            m = a.size
            if m < 8:
                out += [0.0, 0.0]
                continue
            out.append(_safe(float(a.mean()) * math.sqrt(m)))
            out.append(_safe(float(b.mean()) * math.sqrt(m)))
        for j, al in enumerate(self.EW):
            self.ew[j] += al * (self.nu2[i] - self.ew[j])
            out.append(_safe(self.ew[j] * math.sqrt(min(self.n, 2.0 / al))))
        out.append(_safe(math.log(max(self.h, 1e-12) / max(self.v0, 1e-12))))
        out.append(_safe(self.persist))
        out.append(_safe(self.s1 / self.n * math.sqrt(self.n)))
        return out


import os
_p = f"{CACHE}/new_garch.npz"
if os.path.exists(_p):
    os.remove(_p); print("removed stale", _p)
REGISTRY[:] = [f for f in REGISTRY if f.name != "garch"]       # drop the v1 registration
REGISTRY.append(GarchFamily())
print([f.name for f in REGISTRY])

In [ ]:
import numpy as np, _engine

cls = _engine.HistoricalRef                         # the class StreamFeatures really uses
if not getattr(cls, "_keeps_series", False):
    _orig = cls.__init__
    def _init_keep_series(self, x_hist, *a, **k):
        _orig(self, x_hist, *a, **k)
        x = np.asarray(x_hist, dtype=np.float64)
        self.x_series = x[np.isfinite(x)]
    cls.__init__ = _init_keep_series
    cls._keeps_series = True
HistoricalRef = cls                                 # keep the notebook's name in sync

# verify the way the failing code path sees it
sf = _engine.StreamFeatures(np.random.default_rng(0).standard_normal(500), active_families=set())
print("x_series via StreamFeatures:", len(getattr(sf.h, "x_series", [])))    # expect 500

In [ ]:
test_family(GarchFamily)

In [ ]:
class TemporalFamily(FeatureFamily):
    """Everything about TIME structure in one family, so one CV run answers all three questions.

    A. dependence & spectrum (on e, the AR-whitened residual, serially uncorrelated under no
       break): ACF lags 1-10, PACF 1-3 (Durbin-Levinson), Ljung-Box Q(10), and the share of
       spectral power in low / mid / high bands.
    B. volatility clustering: ACF of e^2 and |e| at lags 1-3, plus an ARCH-LM-style Q on e^2.
    C. persistence (on raw standardized ctx.zr): variance ratios VR(k) at k = 2, 4, 8, and an
       aggregated-variance Hurst estimate.
    Called every step (cost "O1") since it keeps its own buffers; the O(W log W) work runs
    every STRIDE steps and is held in between."""
    name = "temporal"
    channels = ("e",)
    cost = "O1"
    version = 1
    W = 256
    STRIDE = 4
    LAGS = 10

    def names(self):
        out = ["tmp_acf%d" % k for k in range(1, self.LAGS + 1)]
        out += ["tmp_pacf%d" % k for k in (1, 2, 3)]
        out += ["tmp_lbq10", "tmp_band_lo", "tmp_band_mid", "tmp_band_hi"]
        out += ["tmp_acf_sq%d" % k for k in (1, 2, 3)] + ["tmp_acf_abs%d" % k for k in (1, 2, 3)]
        out += ["tmp_archq5"]
        out += ["tmp_vr%d" % k for k in (2, 4, 8)] + ["tmp_hurst"]
        return out

    def init(self, hist):
        self.e = np.zeros(self.W); self.r = np.zeros(self.W)
        self.n = 0
        self._last = [0.0] * len(self.names())

    def _ordered(self, a):
        m = min(self.n, self.W)
        i = (self.n - 1) % self.W
        s = i - m + 1
        return a[s:i + 1] if s >= 0 else np.concatenate([a[s % self.W:], a[:i + 1]])

    @staticmethod
    def _acf(x, K):
        x = x - x.mean()
        d = float(np.dot(x, x))
        if d <= 0:
            return np.zeros(K)
        return np.array([float(np.dot(x[k:], x[:-k])) / d for k in range(1, K + 1)])

    @staticmethod
    def _pacf(rho, K):
        phi = np.zeros((K + 1, K + 1)); out = []
        for k in range(1, K + 1):
            num = rho[k - 1] - sum(phi[k - 1, j] * rho[k - 1 - j] for j in range(1, k))
            den = 1.0 - sum(phi[k - 1, j] * rho[j - 1] for j in range(1, k))
            phi[k, k] = num / den if abs(den) > 1e-12 else 0.0
            for j in range(1, k):
                phi[k, j] = phi[k - 1, j] - phi[k, k] * phi[k - 1, k - j]
            out.append(phi[k, k])
        return out

    def update(self, ctx):
        i = self.n % self.W
        self.e[i] = ctx.vals["e"]; self.r[i] = float(ctx.zr)
        self.n += 1
        if self.n < 48 or (self.n % self.STRIDE):
            return self._last

        e = self._ordered(self.e); r = self._ordered(self.r); m = e.size
        rm = math.sqrt(m)
        acf = self._acf(e, self.LAGS)
        out = [_safe(v * rm) for v in acf]
        out += [_safe(v * rm) for v in self._pacf(acf, 3)]
        out.append(_safe(m * (m + 2) * float(np.sum(acf ** 2 / (m - np.arange(1, self.LAGS + 1))))))

        P = (np.abs(np.fft.rfft(e - e.mean())) ** 2)[1:]
        tot = float(P.sum()) or 1.0
        b = len(P) // 3
        out += [_safe(float(P[:b].sum()) / tot - 1/3), _safe(float(P[b:2*b].sum()) / tot - 1/3),
                _safe(float(P[2*b:].sum()) / tot - 1/3)]

        a2 = self._acf(e * e, 5); a1 = self._acf(np.abs(e), 3)
        out += [_safe(v * rm) for v in a2[:3]] + [_safe(v * rm) for v in a1]
        out.append(_safe(m * float(np.sum(a2 ** 2))))

        v1 = float(np.var(r)) or 1e-12
        for k in (2, 4, 8):
            s = np.convolve(r, np.ones(k), "valid")[::k]
            out.append(_safe((float(np.var(s)) / (k * v1) - 1.0) * math.sqrt(len(s))))
        ks, vs = [], []
        for k in (2, 4, 8, 16):
            nb = m // k
            if nb >= 4:
                blk = r[:nb * k].reshape(nb, k).mean(axis=1)
                vk = float(np.var(blk))
                if vk > 0:
                    ks.append(math.log(k)); vs.append(math.log(vk))
        out.append(_safe(1.0 + 0.5 * np.polyfit(ks, vs, 1)[0]) if len(ks) >= 3 else 0.5)
        self._last = out
        return out


if not any(f.name == "temporal" for f in REGISTRY):
    REGISTRY.append(TemporalFamily())
print([f.name for f in REGISTRY])

In [ ]:
test_family(TemporalFamily)

In [ ]:
import numpy as np, lightgbm as lgb, gc, json

# --- the four columns, from the cached temporal extraction
Z = np.load(f"{CACHE}/new_temporal.npz")
tnames = [str(s) for s in Z["names"]]
KEEP4 = ["tmp_acf_sq2", "tmp_acf_abs2", "tmp_acf_sq1", "tmp_archq5"]
X4 = np.ascontiguousarray(Z["X"][:, [tnames.index(n) for n in KEEP4]])
del Z; gc.collect()
print(X4.shape, KEEP4)

# --- production baseline: the 73 features the 14 shipped families emit
fam_of = {nm: f.name for f in REGISTRY for nm in f.names()}
PROD = [i for i, n in enumerate(BASE_NAMES) if n in fam_of and fam_of[n] not in
        {"mood", "ansari", "iqr", "fligner", "temporal", "garch", "rawmu", "dist", "dep", "long",
         "dratio", "rlen", "glr2", "ctm", "newma", "bocpd2"}]
print(len(PROD), "production features")          # must be 73
assert len(PROD) == 73

def cv_prod_plus(label, extra=None, extra_names=(), seeds=(0, 1, 2)):
    aucs = []
    names = [BASE_NAMES[i] for i in PROD] + list(extra_names)
    for fi, F in enumerate(FOLD_IDX):
        for swap in (False, True):
            si, ei = (F["sc"], F["st"]) if swap else (F["st"], F["sc"])
            mk = lambda rows: np.ascontiguousarray(
                np.asarray(X_all[rows])[:, PROD] if extra is None
                else np.hstack([np.asarray(X_all[rows])[:, PROD], extra[rows]]))
            Xtr, Xs, Xe = mk(F["tr"]), mk(si), mk(ei)
            pr = []
            for s in seeds:
                p = dict(LGB_PARAMS); p.update(objective="binary", metric="binary_logloss",
                                               seed=s, bagging_seed=s + 100, feature_fraction_seed=s + 200)
                dtr = lgb.Dataset(Xtr, label=y_all[F["tr"]],
                                  weight=STRIDE_ALL[F["tr"]].astype(np.float32), feature_name=names)
                b = lgb.train(p, dtr, num_boost_round=NUM_ROUNDS,
                              valid_sets=[lgb.Dataset(Xs, label=y_all[si], reference=dtr)],
                              callbacks=[lgb.early_stopping(100, verbose=False)])
                pr.append(b.predict(Xe, num_iteration=b.best_iteration)); del dtr, b; gc.collect()
            aucs.append(ts_auc(y_all[ei], t_all[ei], np.mean(pr, axis=0)))
            print(f"  [{label}] fold {fi} swap={swap}: {aucs[-1]:.5f}")
            del Xtr, Xs, Xe, pr; gc.collect()
    CMP[label] = np.array(aucs); print(f"[{label}] mean {CMP[label].mean():.5f}")

FOLD_IDX = build_folds(11); CMP.clear()
cv_prod_plus("prod73")
cv_prod_plus("prod73 + arch4", X4, KEEP4)
cmp_table("prod73")

In [ ]:
# =====================================================================
#  NEGATIVE AUGMENTATION
#  Split long historical segments into pseudo-history + pseudo-online.
#  The historical segment is pre-break by construction, so every pseudo-online
#  point is a guaranteed negative, including hard "irregular but unbroken" ones.
#  Run the cells in order; each one checks the assumption the next one relies on.
# =====================================================================

# ---------- CELL A: inspect the data format and length distributions ----------
import numpy as np, pandas as pd, lightgbm as lgb, gc, time

items = list(train_data)
print("item layout:", type(items[0]), len(items[0]), [type(v).__name__ for v in items[0]])
LH = np.array([len(it[1]) for it in items])
LO = np.array([len(it[2]) for it in items])
print("history length  p0/p5/p25/p50/p75/p95/p100:", np.percentile(LH, [0, 5, 25, 50, 75, 95, 100]).astype(int))
print("online  length  p0/p5/p25/p50/p75/p95/p100:", np.percentile(LO, [0, 5, 25, 50, 75, 95, 100]).astype(int))
taus = [it[3] for it in items]
print("tau types:", pd.Series([type(v).__name__ for v in taus]).value_counts().to_dict())
print("first 12 taus:", taus[:12])
# the no-break sentinel is whatever unbroken series carry (None / NaN / -1 ...)
# confirm against the cache: series whose labels are all zero
_g0 = pd.Series(y_all).groupby(g_all).max()
_nobreak_ids = set(_g0.index[_g0 == 0])
_ex = next(it for it in items if it[0] in _nobreak_ids)
NOBREAK = _ex[3]
print("no-break sentinel (from an all-zero series):", repr(NOBREAK), "| unbroken series:", len(_nobreak_ids))


# ---------- CELL B: build augmented negative series ----------
PROD_FAMS = {"z2", "e2", "tail", "time", "ftest", "bartlett", "levene",
             "lmom", "sigtuk", "wass", "energy", "mmd", "entropy", "bocpd"}
MIN_HIST = int(np.percentile(LH, 5))        # pseudo-history must look like a REAL history
MIN_ONLINE = 32
AUG_ID_OFFSET = 10_000_000                  # keeps ids integer and disjoint from real ids

def make_aug(items, seed=0):
    rng = np.random.default_rng(seed)
    aug, src = [], []
    for sid, xh, xo, tau in items:
        H = len(xh)
        h_len = int(rng.choice(LH))                     # target history length ~ real distribution
        o_len = int(rng.choice(LO))                     # target online length  ~ real distribution
        if h_len < MIN_HIST or H < h_len + MIN_ONLINE:
            continue
        o_len = min(o_len, H - h_len)
        if o_len < MIN_ONLINE:
            continue
        aug.append((AUG_ID_OFFSET + int(sid), np.asarray(xh[:h_len]), np.asarray(xh[h_len:h_len + o_len]), NOBREAK))
        src.append(int(sid))
    return aug, np.array(src)

AUG_ITEMS, AUG_SRC = make_aug(items)
ah = np.array([len(a[1]) for a in AUG_ITEMS]); ao = np.array([len(a[2]) for a in AUG_ITEMS])
print(f"augmented series: {len(AUG_ITEMS)} from {len(items)} real")
print("aug history p5/p50/p95:", np.percentile(ah, [5, 50, 95]).astype(int), "| real:", np.percentile(LH, [5, 50, 95]).astype(int))
print("aug online  p5/p50/p95:", np.percentile(ao, [5, 50, 95]).astype(int), "| real:", np.percentile(LO, [5, 50, 95]).astype(int))


# ---------- CELL C: extract features for the augmented series ----------
PROD_NAMES = [BASE_NAMES[i] for i in PROD]                  # the 73, in cache column order
t0 = time.time()
Xa, ya, ta, wa, ga = build_dataset(AUG_ITEMS, subsample=True,
                                   active_families=PROD_FAMS, backend="fork")
got = active_feature_names(PROD_FAMS)
assert Xa.shape[1] == len(got) == 73, (Xa.shape, len(got))
Xa = np.ascontiguousarray(Xa[:, [got.index(n) for n in PROD_NAMES]])   # reorder to PROD order
assert ya.max() == 0, "augmented rows must all be negative -- check the NOBREAK sentinel"
assert np.isfinite(Xa).all()
ga_src = ga.astype(np.int64) - AUG_ID_OFFSET                           # map back to the source series
np.savez(f"{CACHE}/aug_neg.npz", X=Xa, t=ta, w=wa, src=ga_src)
print(f"augmented rows {Xa.shape} in {(time.time()-t0)/60:.1f} min | all negative: {ya.max() == 0}")


# ---------- CELL D: can a model tell augmented negatives from real ones? ----------
# If it can, the tree will learn "augmented => negative" instead of "no break => negative".
F = FOLD_IDX[0]
real_neg = F["tr"][y_all[F["tr"]] == 0]
rng = np.random.default_rng(0)
rn = np.sort(rng.choice(real_neg, size=min(200_000, real_neg.size), replace=False))
an = rng.choice(Xa.shape[0], size=min(200_000, Xa.shape[0]), replace=False)
Xd = np.vstack([np.asarray(X_all[rn])[:, PROD], Xa[an]]).astype(np.float32)
yd = np.r_[np.zeros(rn.size), np.ones(an.size)]
td = np.r_[t_all[rn], ta[an]]
perm = rng.permutation(yd.size); cut = int(0.7 * yd.size)
tr_i, te_i = perm[:cut], perm[cut:]
b = lgb.train(dict(objective="binary", learning_rate=0.1, num_leaves=31, verbosity=-1),
              lgb.Dataset(Xd[tr_i], yd[tr_i], feature_name=PROD_NAMES), num_boost_round=200)
from sklearn.metrics import roc_auc_score
pd_ = b.predict(Xd[te_i])
print(f"real-vs-augmented AUC: {roc_auc_score(yd[te_i], pd_):.4f}   (0.50 = indistinguishable)")
print(f"  ... within t (TS-AUC): {ts_auc(yd[te_i].astype(np.int8), td[te_i], pd_):.4f}")
imp = pd.Series(b.feature_importance('gain'), index=PROD_NAMES).sort_values(ascending=False)
print("features that give augmented rows away:\n", (imp / imp.sum()).head(8).round(3).to_string())
del Xd, yd, td, b; gc.collect()
# READ: < 0.55 -> safe to use. 0.55-0.65 -> usable with a small weight. > 0.65 -> fix the
# giveaway first (usually history length or AR-fit quality) before trusting any CV result.


# ---------- CELL E: CV -- augmented rows added to TRAINING folds only ----------
def cv_aug(label, w_aug=0.0, seeds=(0, 1, 2)):
    """Score is always on REAL series. Augmented rows come only from series in the training fold,
    so no information from a validation series (not even its history) enters training."""
    aucs = []
    for fi, F in enumerate(FOLD_IDX):
        tr_series = np.unique(g_all[F["tr"]])
        amask = np.isin(ga_src, tr_series) if w_aug > 0 else np.zeros(Xa.shape[0], bool)
        for swap in (False, True):
            si, ei = (F["sc"], F["st"]) if swap else (F["st"], F["sc"])
            Xr = np.asarray(X_all[F["tr"]])[:, PROD]
            Xtr = np.ascontiguousarray(np.vstack([Xr, Xa[amask]]) if amask.any() else Xr)
            ytr = np.r_[y_all[F["tr"]], np.zeros(int(amask.sum()), np.int8)]
            wtr = np.r_[STRIDE_ALL[F["tr"]].astype(np.float32), (wa[amask] * w_aug).astype(np.float32)]
            Xs = np.ascontiguousarray(np.asarray(X_all[si])[:, PROD])
            Xe = np.ascontiguousarray(np.asarray(X_all[ei])[:, PROD])
            pr = []
            for s in seeds:
                p = dict(LGB_PARAMS); p.update(objective="binary", metric="binary_logloss",
                                               seed=s, bagging_seed=s + 100, feature_fraction_seed=s + 200)
                dtr = lgb.Dataset(Xtr, label=ytr, weight=wtr, feature_name=PROD_NAMES)
                bb = lgb.train(p, dtr, num_boost_round=NUM_ROUNDS,
                               valid_sets=[lgb.Dataset(Xs, label=y_all[si], reference=dtr)],
                               callbacks=[lgb.early_stopping(100, verbose=False)])
                pr.append(bb.predict(Xe, num_iteration=bb.best_iteration)); del dtr, bb; gc.collect()
            aucs.append(ts_auc(y_all[ei], t_all[ei], np.mean(pr, axis=0)))
            print(f"  [{label}] fold {fi} swap={swap}: {aucs[-1]:.5f} | +{int(amask.sum()):,} aug rows")
            del Xr, Xtr, Xs, Xe, pr; gc.collect()
    CMP[label] = np.array(aucs); print(f"[{label}] mean {CMP[label].mean():.5f}")

FOLD_IDX = build_folds(11); CMP.clear()
cv_aug("prod73 (no aug)", 0.0)
cv_aug("aug w=0.25", 0.25)
cv_aug("aug w=0.5", 0.5)
cv_aug("aug w=1.0", 1.0)
cmp_table("prod73 (no aug)")

In [ ]:
# =====================================================================
#  TEMPORAL STACKING  (single cell)
#
#  Outer: your 3 folds (FOLD_IDX, seed 11). Their st+sc halves are evaluation-only.
#  Inner: 5 folds BY SERIES inside each outer-training set -> out-of-fold stage-1 logits.
#  Stage 1 on outer-validation series = average of the 5 inner models (= what ships).
#  Stage 2 = model on causal features of the stage-1 score trajectory.
#
#  Four methods, all trained on outer-training series only, scored on both halves:
#    base      single full LightGBM, 3 seeds          (today's model)
#    fold5     average of the 5 inner models           (fold-averaging alone)
#    s2_logit  fold5 -> logistic regression stage 2
#    s2_lgbm   fold5 -> monotone-constrained LightGBM stage 2
# =====================================================================
import numpy as np, pandas as pd, lightgbm as lgb, gc, time, json, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

K_INNER = 5
ES_FRAC = 0.10              # share of training SERIES held out for early stopping (never OOF rows)
SEEDS_BASE = (0, 1, 2)
PRED_CHUNK = 400_000
assert X_all.shape[1] == len(BASE_NAMES) and len(PROD) == 73, (X_all.shape, len(PROD))
PNAMES = [BASE_NAMES[i] for i in PROD]
STR = STRIDE_ALL.astype(np.float32)
SUB = (t_all % STRIDE_ALL) == 0                     # rows the stage-1 training subsample keeps

# ---------- helpers --------------------------------------------------------
def rows_of(series, mask=None):
    m = np.isin(g_all, series)
    if mask is not None:
        m &= mask
    return np.flatnonzero(m)

def fit_lgb(tr_rows, es_rows, seed, params=None):
    p = dict(LGB_PARAMS); p.update(params or {})
    p.update(objective="binary", metric="binary_logloss",
             seed=seed, bagging_seed=seed + 100, feature_fraction_seed=seed + 200)
    Xtr = np.ascontiguousarray(np.asarray(X_all[tr_rows])[:, PROD])
    Xes = np.ascontiguousarray(np.asarray(X_all[es_rows])[:, PROD])
    dtr = lgb.Dataset(Xtr, label=y_all[tr_rows], weight=STR[tr_rows], feature_name=PNAMES)
    b = lgb.train(p, dtr, num_boost_round=NUM_ROUNDS,
                  valid_sets=[lgb.Dataset(Xes, label=y_all[es_rows], weight=STR[es_rows], reference=dtr)],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    del Xtr, Xes, dtr; gc.collect()
    return b

def predict_logit(b, rows):
    out = np.empty(rows.size, dtype=np.float64)
    for i in range(0, rows.size, PRED_CHUNK):
        r = rows[i:i + PRED_CHUNK]
        out[i:i + r.size] = b.predict(np.asarray(X_all[r])[:, PROD], num_iteration=b.best_iteration, raw_score=True)
    return out

def split_es(series, seed):
    rng = np.random.default_rng(seed)
    s = rng.permutation(series)
    n_es = max(1, int(ES_FRAC * s.size))
    return s[n_es:], s[:n_es]

STRIDE_TAB = np.array([_stride_for(int(x)) for x in range(int(t_all.max()) + 1)])

def trajectory_features(rows, s):
    """Causal features of the score path. rows/s aligned; returns (F, names) in the SAME order."""
    g, t = g_all[rows], t_all[rows].astype(np.int64)
    o = np.lexsort((t, g))
    g, t, s = g[o], t[o], s[o].copy()
    new = np.r_[True, g[1:] != g[:-1]]
    # mimic inference: score only on stride steps, hold in between
    keep = (t % STRIDE_TAB[t]) == 0
    s[~keep] = np.nan
    s = pd.Series(s).groupby(g).ffill().fillna(0.0).to_numpy()

    df = pd.DataFrame({"g": g, "s": s})
    gb = df.groupby("g").s
    feats = {"s": s}
    for a in (0.3, 0.1, 0.03):
        feats[f"ew{a}"] = gb.transform(lambda x, a=a: x.ewm(alpha=a, adjust=False).mean()).to_numpy()
    feats["slope"] = feats["ew0.3"] - feats["ew0.03"]
    feats["runmax"] = gb.cummax().to_numpy()
    feats["drawdown"] = s - feats["runmax"]
    feats["runmean"] = (gb.cumsum() / (gb.cumcount() + 1)).to_numpy()
    feats["lag4"] = gb.shift(4).fillna(0.0).to_numpy()
    feats["lag16"] = gb.shift(16).fillna(0.0).to_numpy()

    # CUSUM and persistence against the null quantiles (fitted on training negatives only)
    c50 = np.zeros(s.size); c90 = np.zeros(s.size)
    tfirst = np.full(s.size, -1.0); frac = np.zeros(s.size)
    C1 = C2 = 0.0; first = -1; above = 0; k = 0
    for i in range(s.size):
        if new[i]:
            C1 = C2 = 0.0; first = -1; above = 0; k = 0
        v = s[i]; k += 1
        C1 = max(0.0, C1 + v - Q50); C2 = max(0.0, C2 + v - Q90)
        if v > Q90:
            above += 1
            if first < 0:
                first = t[i]
        c50[i] = C1; c90[i] = C2; frac[i] = above / k
        tfirst[i] = (t[i] - first) if first >= 0 else -1.0
    feats["cusum50"] = c50; feats["cusum90"] = c90; feats["frac90"] = frac
    feats["since90"] = np.where(tfirst >= 0, np.log1p(np.maximum(tfirst, 0)), -1.0)
    feats["t_log"] = np.log1p(t.astype(np.float64))

    names = list(feats)
    Fm = np.column_stack([feats[n] for n in names]).astype(np.float32)
    inv = np.empty_like(o); inv[o] = np.arange(o.size)
    return Fm[inv], names

# monotone +1 where more evidence can only mean "more likely broken"; 0 where direction is ambiguous
MONO = {"s": 1, "ew0.3": 1, "ew0.1": 1, "ew0.03": 1, "slope": 0, "runmax": 1, "drawdown": 0,
        "runmean": 1, "lag4": 1, "lag16": 1, "cusum50": 1, "cusum90": 1, "frac90": 1,
        "since90": 1, "t_log": 0}
S2_LGB = dict(objective="binary", metric="binary_logloss", learning_rate=0.05, num_leaves=15,
              min_data_in_leaf=2000, lambda_l2=50.0, feature_fraction=0.9, bagging_fraction=0.8,
              bagging_freq=1, verbosity=-1, seed=0, num_threads=8)

# ---------- main loop over outer folds -------------------------------------
RES = {k: [] for k in ("base", "fold5", "s2_logit", "s2_lgbm")}
T0 = time.time()
for fi, F in enumerate(FOLD_IDX):
    tf = time.time()
    tr_series = np.unique(g_all[F["tr"]])
    ev_rows = np.concatenate([F["st"], F["sc"]])
    half = np.r_[np.zeros(F["st"].size, bool), np.ones(F["sc"].size, bool)]   # False=st, True=sc

    # --- base: today's model (3 seeds), early-stopped on held-out TRAINING series
    fit_s, es_s = split_es(tr_series, 1000 + fi)
    b_tr, b_es = rows_of(fit_s, SUB), rows_of(es_s, SUB)
    p_base = np.zeros(ev_rows.size)
    for sd in SEEDS_BASE:
        b = fit_lgb(b_tr, b_es, sd); p_base += predict_logit(b, ev_rows) / len(SEEDS_BASE); del b; gc.collect()

    # --- inner 5-fold by series: OOF logits on all training rows, and 5-avg on evaluation rows
    rng = np.random.default_rng(2000 + fi)
    parts = np.array_split(rng.permutation(tr_series), K_INNER)
    tr_full = rows_of(tr_series)                                   # full resolution, all training series
    oof = np.full(tr_full.size, np.nan)
    p_fold5 = np.zeros(ev_rows.size)
    for k in range(K_INNER):
        hold = parts[k]
        rest = np.concatenate([parts[j] for j in range(K_INNER) if j != k])
        fit_s, es_s = split_es(rest, 3000 + 10 * fi + k)
        b = fit_lgb(rows_of(fit_s, SUB), rows_of(es_s, SUB), seed=k)
        hm = np.isin(g_all[tr_full], hold)
        oof[hm] = predict_logit(b, tr_full[hm])
        p_fold5 += predict_logit(b, ev_rows) / K_INNER
        print(f"  fold {fi} inner {k}: trees {b.best_iteration} | {time.time()-tf:.0f}s")
        del b; gc.collect()
    assert np.isfinite(oof).all()

    # --- null quantiles from TRAINING negatives only
    neg = y_all[tr_full] == 0
    Q50, Q90 = np.quantile(oof[neg], [0.5, 0.9])

    # --- stage-2 features
    F_tr, fnames = trajectory_features(tr_full, oof)
    F_ev, _ = trajectory_features(ev_rows, p_fold5)

    # stage-2 training rows: subsampled like stage 1, split by series for LGBM early stopping
    sub_tr = SUB[tr_full]
    fit_s, es_s = split_es(tr_series, 4000 + fi)
    m_fit = sub_tr & np.isin(g_all[tr_full], fit_s)
    m_es = sub_tr & np.isin(g_all[tr_full], es_s)
    ytr_full = y_all[tr_full]; wtr_full = STR[tr_full]

    # logistic stage 2
    sc = StandardScaler().fit(F_tr[sub_tr])
    lr = LogisticRegression(max_iter=2000, C=1.0).fit(sc.transform(F_tr[sub_tr]), ytr_full[sub_tr],
                                                      sample_weight=wtr_full[sub_tr])
    p_s2l = lr.decision_function(sc.transform(F_ev))

    # monotone LightGBM stage 2
    p2 = dict(S2_LGB); p2["monotone_constraints"] = [MONO[n] for n in fnames]
    d2 = lgb.Dataset(F_tr[m_fit], label=ytr_full[m_fit], weight=wtr_full[m_fit], feature_name=fnames)
    b2 = lgb.train(p2, d2, num_boost_round=3000,
                   valid_sets=[lgb.Dataset(F_tr[m_es], label=ytr_full[m_es], weight=wtr_full[m_es], reference=d2)],
                   callbacks=[lgb.early_stopping(100, verbose=False)])
    p_s2g = b2.predict(F_ev, num_iteration=b2.best_iteration, raw_score=True)

    # --- score both halves
    for name, p in (("base", p_base), ("fold5", p_fold5), ("s2_logit", p_s2l), ("s2_lgbm", p_s2g)):
        for h in (False, True):
            m = half == h
            RES[name].append(ts_auc(y_all[ev_rows[m]], t_all[ev_rows[m]], p[m]))
    print(f"fold {fi} | " + " | ".join(f"{k} {np.mean(v[-2:]):.5f}" for k, v in RES.items())
          + f" | s2 trees {b2.best_iteration} | {(time.time()-tf)/60:.1f} min")
    imp = pd.Series(b2.feature_importance("gain"), index=fnames).sort_values(ascending=False)
    print("   stage-2 top features:", ", ".join(f"{n} {v/imp.sum():.2f}" for n, v in imp.head(5).items()))
    del F_tr, F_ev, oof, p_base, p_fold5, p_s2l, p_s2g, b2, d2, tr_full; gc.collect()

# ---------- verdict ---------------------------------------------------------
base = np.array(RES["base"])
rows = []
for name, v in RES.items():
    v = np.array(v); d = v - base
    se = d.std(ddof=1) / np.sqrt(len(d)) if name != "base" else 0.0
    rows.append(dict(method=name, mean=v.mean(), delta=d.mean(), se=se,
                     sigma=(d.mean() / se if se else 0.0), better=f"{int((d > 0).sum())}/{len(d)}"))
T = pd.DataFrame(rows).sort_values("mean", ascending=False).round(5)
print("\n" + T.to_string(index=False))
f5 = np.array(RES["fold5"])
for name in ("s2_logit", "s2_lgbm"):
    d = np.array(RES[name]) - f5; se = d.std(ddof=1) / np.sqrt(len(d))
    print(f"{name} vs fold5 (the pure stacking gain): {d.mean():+.5f} | se {se:.5f} | {int((d>0).sum())}/{len(d)}")
print(f"\ntotal {(time.time()-T0)/60:.1f} min")

In [ ]:
def run_fold5_confirm(fold_seed=7, k_inner=5, seeds_base=(0, 1, 2, 3, 4)):
    """base with 5 seeds vs fold5: equal member counts, so any gain is data diversity, not count."""
    global FOLD_IDX
    FOLD_IDX = build_folds(fold_seed)
    res = {"base5seeds": [], "fold5": []}
    for fi, F in enumerate(FOLD_IDX):
        tf = time.time()
        tr_series = np.unique(g_all[F["tr"]])
        ev_rows = np.concatenate([F["st"], F["sc"]])
        half = np.r_[np.zeros(F["st"].size, bool), np.ones(F["sc"].size, bool)]

        fit_s, es_s = split_es(tr_series, 1000 + fi)
        b_tr, b_es = rows_of(fit_s, SUB), rows_of(es_s, SUB)
        p_base = np.zeros(ev_rows.size)
        for sd in seeds_base:
            b = fit_lgb(b_tr, b_es, sd); p_base += predict_logit(b, ev_rows) / len(seeds_base)
            del b; gc.collect()

        parts = np.array_split(np.random.default_rng(2000 + fi).permutation(tr_series), k_inner)
        p_f = np.zeros(ev_rows.size)
        for k in range(k_inner):
            rest = np.concatenate([parts[j] for j in range(k_inner) if j != k])
            fs, es = split_es(rest, 3000 + 10 * fi + k)
            b = fit_lgb(rows_of(fs, SUB), rows_of(es, SUB), seed=k)
            p_f += predict_logit(b, ev_rows) / k_inner
            del b; gc.collect()

        for name, p in (("base5seeds", p_base), ("fold5", p_f)):
            for h in (False, True):
                m = half == h
                res[name].append(ts_auc(y_all[ev_rows[m]], t_all[ev_rows[m]], p[m]))
        print(f"fold {fi}: base5 {np.mean(res['base5seeds'][-2:]):.5f} | fold5 {np.mean(res['fold5'][-2:]):.5f}"
              f" | {(time.time()-tf)/60:.1f} min")
    d = np.array(res["fold5"]) - np.array(res["base5seeds"])
    se = d.std(ddof=1) / np.sqrt(len(d))
    print(f"\nfold5 vs base (5 seeds): {d.mean():+.5f} | se {se:.5f} | {d.mean()/se:.1f} sigma | {int((d>0).sum())}/6")
    return res

R7 = run_fold5_confirm(7)

In [ ]:
class ContextFamily(FeatureFamily):
    """What the HISTORICAL segment looks like -- constant for the whole series.

    Carries no break evidence by itself (so univariate TS-AUC sits near 0.5 by design). Its job is
    to let the trees learn how much to trust every other column for this kind of series: a variance
    ratio of 1.5 against a long, clean, well-fitted history is strong evidence; the same 1.5 against
    a short, heavy-tailed, badly-fitted one is noise. Culled on univariate AUC originally and never
    CV-tested -- the same mistake that nearly lost bocpd_runlen_log.

    Reads hist.context, computed once in HistoricalRef.__init__: length, skew, kurtosis, ACF(1,2),
    AR(3) coefficients, residual-sd / sd (how much the AR fit explains), sd / MAD (tail weight),
    P(|z|>2), P(|z|>3), and the ACF of the PIT-normal series."""
    name = "h"
    channels = ()
    cost = "O1"
    version = 2

    def names(self):
        return ["h_n", "h_logn", "h_skew", "h_kurt", "h_acf1", "h_acf2",
                "h_ar1", "h_ar2", "h_ar3", "h_rsd_ratio", "h_sd_mad",
                "h_p2", "h_p3", "h_zacf1"]

    def init(self, hist):
        ctx = getattr(hist, "context", None)
        assert ctx is not None and len(ctx) == 14, \
            "HistoricalRef.context missing or changed -- check HistoricalRef.__init__"
        self.ctx = [_safe(float(v)) for v in ctx]

    def update(self, ctx):
        return self.ctx


if not any(f.name == "h" for f in REGISTRY):
    REGISTRY.append(ContextFamily())
print([f.name for f in REGISTRY])

In [ ]:
test_family(ContextFamily)

In [ ]:
import numpy as np, lightgbm as lgb, gc

Z = np.load(f"{CACHE}/new_garch.npz")
gnames = [str(s) for s in Z["names"]]
XG = np.asarray(Z["X"], dtype=np.float32); del Z; gc.collect()
print(XG.shape, gnames)

def g_cols(drop=()):
    keep = [n for n in gnames if n not in set(drop)]
    return np.ascontiguousarray(XG[:, [gnames.index(n) for n in keep]]), keep

fam_of = {nm: f.name for f in REGISTRY for nm in f.names()}
PROD = [i for i, n in enumerate(BASE_NAMES) if n in fam_of and fam_of[n] in
        {"z2", "e2", "tail", "time", "ftest", "bartlett", "levene", "lmom",
         "sigtuk", "wass", "energy", "mmd", "entropy", "bocpd"}]
assert len(PROD) == 73, len(PROD)

def cv_prod_plus(label, extra=None, extra_names=(), seeds=(0, 1, 2)):
    aucs = []
    names = [BASE_NAMES[i] for i in PROD] + list(extra_names)
    for fi, F in enumerate(FOLD_IDX):
        for swap in (False, True):
            si, ei = (F["sc"], F["st"]) if swap else (F["st"], F["sc"])
            mk = lambda rows: np.ascontiguousarray(
                np.asarray(X_all[rows])[:, PROD] if extra is None
                else np.hstack([np.asarray(X_all[rows])[:, PROD], extra[rows]]))
            Xtr, Xs, Xe = mk(F["tr"]), mk(si), mk(ei)
            pr = []
            for s in seeds:
                p = dict(LGB_PARAMS); p.update(objective="binary", metric="binary_logloss",
                                               seed=s, bagging_seed=s + 100, feature_fraction_seed=s + 200)
                dtr = lgb.Dataset(Xtr, label=y_all[F["tr"]],
                                  weight=STRIDE_ALL[F["tr"]].astype(np.float32), feature_name=names)
                b = lgb.train(p, dtr, num_boost_round=NUM_ROUNDS,
                              valid_sets=[lgb.Dataset(Xs, label=y_all[si], reference=dtr)],
                              callbacks=[lgb.early_stopping(100, verbose=False)])
                pr.append(b.predict(Xe, num_iteration=b.best_iteration)); del dtr, b; gc.collect()
            aucs.append(ts_auc(y_all[ei], t_all[ei], np.mean(pr, axis=0)))
            print(f"  [{label}] fold {fi} swap={swap}: {aucs[-1]:.5f}")
            del Xtr, Xs, Xe, pr; gc.collect()
    CMP[label] = np.array(aucs); print(f"[{label}] mean {CMP[label].mean():.5f}")

FOLD_IDX = build_folds(7); CMP.clear()          # fresh split: independent of the harness run
cv_prod_plus("prod73")

In [ ]:
X10, n10 = g_cols()
cv_prod_plus("+garch (all 10)", X10, n10)

In [ ]:
X8, n8 = g_cols(drop=("garch_logh", "garch_persist"))
cv_prod_plus("+garch (-logh,-persist)", X8, n8)

In [ ]:
X7, n7 = g_cols(drop=("garch_persist"))
cv_prod_plus("+garch (-persist)", X7, n7)
cmp_table("prod73")

In [ ]:
keep_family("garch")

In [ ]:
# =====================================================================
#  Four conditional models of the HISTORY, each asking "does the online data still follow
#  the dynamics this model learned?"  All need HistoricalRef.x_series (the ordered history).
#    argarch  GARCH on the AR(3) residuals      -> joint mean + variance dynamics
#    tgarch   GARCH with Student-t innovations  -> proper (constant) kurtosis
#    gjr      GJR-GARCH                          -> asymmetric (leverage) volatility response
#    caviar   CAViaR conditional quantiles       -> dynamic tails, tested by exceedance rates
# =====================================================================
from scipy.special import stdtr as _stdtr, ndtri as _ndtri, gammaln as _gln

_SQ2 = 0.7071067811865476


def _history(hist):
    xs = getattr(hist, "x_series", None)
    assert xs is not None and xs.size >= 50, "HistoricalRef.x_series missing -- run the patch cell"
    return np.asarray(xs, dtype=np.float64)


def _ar_resid(xs, mu, ar):
    """AR(p) residuals of the centred history, using the AR coefficients fitted in HistoricalRef."""
    c = xs - mu
    p = len(ar)
    r = c.copy()
    for k in range(1, p + 1):
        r[p:] -= ar[k - 1] * c[p - k:-k]
    return r[p:]


def _garch_grid(x2, v, extra=None, nll=None):
    """Grid + local refinement over (alpha, beta[, extra]) with variance targeting."""
    best, bp = None, None
    extras = extra if extra is not None else (None,)
    for ex in extras:
        for al in (0.03, 0.08, 0.15, 0.25):
            for be in (0.0, 0.50, 0.75, 0.88, 0.94):
                f = nll(al, be, ex, x2, v)
                if np.isfinite(f) and (best is None or f < best):
                    best, bp = f, (al, be, ex)
    al, be, ex = bp
    for step in (0.04, 0.015):
        for da in (-step, 0.0, step):
            for db in (-step, 0.0, step):
                a2, b2 = al + da, be + db
                if a2 < 0 or b2 < 0:
                    continue
                f = nll(a2, b2, ex, x2, v)
                if f < best:
                    best, al, be = f, a2, b2
    return al, be, ex


class _CondBase(FeatureFamily):
    """Shared plumbing: a ring buffer per derived stream, window/EWMA/expanding aggregates."""
    channels = ("z",)
    cost = "O1"
    version = 1
    WINS = (32, 128, 512)
    STREAMS = ()          # names of the derived per-step streams, each ~ mean 0 under the null

    def _setup_buffers(self):
        self.cap = max(self.WINS)
        self.buf = {s: np.zeros(self.cap) for s in self.STREAMS}
        self.acc = {s: 0.0 for s in self.STREAMS}
        self.ew = {s: 0.0 for s in self.STREAMS}
        self.n = 0

    def _push(self, vals):
        i = self.n % self.cap
        self.n += 1
        for s, v in vals.items():
            self.buf[s][i] = v
            self.acc[s] += v
            self.ew[s] += 0.02 * (v - self.ew[s])

    def _win(self, a, W):
        m = W if self.n > W else self.n
        i = (self.n - 1) % self.cap
        s = i - m + 1
        return a[s:i + 1] if s >= 0 else np.concatenate([a[s % self.cap:], a[:i + 1]])

    def _aggregate(self):
        out = []
        for s in self.STREAMS:
            for W in self.WINS:
                w = self._win(self.buf[s], W)
                out.append(_safe(float(w.mean()) * math.sqrt(w.size)) if w.size >= 8 else 0.0)
            out.append(_safe(self.ew[s] * math.sqrt(min(self.n, 100.0))))
            out.append(_safe(self.acc[s] / self.n * math.sqrt(self.n)))
        return out

    def _agg_names(self):
        return [f"{self.name}_{s}_{k}" for s in self.STREAMS
                for k in [f"w{W}" for W in self.WINS] + ["ew", "exp"]]


# ---------------------------------------------------------------- 1. AR-GARCH
class ARGarchFamily(_CondBase):
    """GARCH on the AR(3) residuals instead of the raw series: the mean dynamics are removed first,
    so the volatility model sees genuine innovations. Streams: nu^2 - 1 (scale surprise) and
    log(eps^2/h) (realized vs expected variance)."""
    name = "argarch"
    STREAMS = ("nu2", "gap")

    @staticmethod
    def _nll(al, be, _ex, x2, v):
        om = v * (1.0 - al - be)
        if om <= 1e-12 or al + be >= 0.99:
            return 1e18
        h, s = v, 0.0
        for e2 in x2:
            s += math.log(h) + e2 / h
            h = om + al * e2 + be * h
        return s

    def names(self):
        return self._agg_names() + [f"{self.name}_logh"]

    def init(self, hist):
        xs = _history(hist)
        self.mu, self.sd = float(hist.mu), (float(hist.sd) if hist.sd > 0 else 1.0)
        self.ar = [float(a) for a in hist.ar_l]
        r = _ar_resid(xs, self.mu, self.ar)
        v = max(float(np.var(r)), 1e-10)
        al, be, _ = _garch_grid((r ** 2)[-600:], v, nll=self._nll)
        self.om, self.al, self.be, self.v0 = v * (1 - al - be), al, be, v
        h = v
        for e in r:
            h = self.om + self.al * e * e + self.be * h
        self.h = max(h, 1e-12)
        self.lags = list((xs[-len(self.ar):] - self.mu)[::-1])    # most recent first
        self._setup_buffers()

    def update(self, ctx):
        c = float(ctx.zr) * self.sd
        eps = c - sum(a * l for a, l in zip(self.ar, self.lags))
        self.lags = [c] + self.lags[:-1]
        h = max(self.h, 1e-12)
        nu2 = min(eps * eps / h, 100.0)
        self.h = self.om + self.al * eps * eps + self.be * h
        self._push({"nu2": (nu2 - 1.0) * _SQ2,
                    "gap": math.log(max(eps * eps, 1e-12) / h)})
        return self._aggregate() + [_safe(math.log(self.h / self.v0))]


# ---------------------------------------------------------------- 2. Student-t GARCH
class TGarchFamily(_CondBase):
    """GARCH with Student-t innovations: the degrees of freedom are estimated with alpha/beta, so the
    innovations are standardized by the RIGHT distribution. Each standardized innovation is mapped
    through the t CDF and back to a normal score, so under the null the stream is exactly N(0,1)
    even for fat-tailed data. Streams: z^2 - 1 (scale) and 1[|z|>2] - P(|Z|>2) (tail rate)."""
    name = "tgarch"
    STREAMS = ("z2", "tail2")
    NUS = (4.0, 6.0, 10.0, 30.0)
    P2 = 0.04550026389635842

    @staticmethod
    def _nll(al, be, nu, x2, v):
        om = v * (1.0 - al - be)
        if om <= 1e-12 or al + be >= 0.99:
            return 1e18
        c = _gln((nu + 1) / 2) - _gln(nu / 2) - 0.5 * math.log(math.pi * (nu - 2))
        h, s = v, 0.0
        for e2 in x2:
            s -= c - 0.5 * math.log(h) - (nu + 1) / 2 * math.log1p(e2 / ((nu - 2) * h))
            h = om + al * e2 + be * h
        return s

    def names(self):
        return self._agg_names() + [f"{self.name}_nu"]

    def init(self, hist):
        xs = _history(hist)
        self.mu, self.sd = float(hist.mu), (float(hist.sd) if hist.sd > 0 else 1.0)
        eps = xs - self.mu
        v = max(float(np.var(eps)), 1e-10)
        al, be, nu = _garch_grid((eps ** 2)[-600:], v, extra=self.NUS, nll=self._nll)
        self.om, self.al, self.be, self.nu, self.v0 = v * (1 - al - be), al, be, nu, v
        self.tscale = math.sqrt(nu / (nu - 2.0))
        h = v
        for e in eps:
            h = self.om + self.al * e * e + self.be * h
        self.h = max(h, 1e-12)
        self._setup_buffers()

    def update(self, ctx):
        eps = float(ctx.zr) * self.sd
        h = max(self.h, 1e-12)
        s = eps / math.sqrt(h) * self.tscale
        u = min(max(float(_stdtr(self.nu, s)), 1e-12), 1 - 1e-12)
        z = float(_ndtri(u))
        self.h = self.om + self.al * eps * eps + self.be * h
        self._push({"z2": (z * z - 1.0) * _SQ2,
                    "tail2": (1.0 if abs(z) > 2.0 else 0.0) - self.P2})
        return self._aggregate() + [_safe(math.log(self.nu))]


# ---------------------------------------------------------------- 3. GJR-GARCH
class GJRGarchFamily(_CondBase):
    """GJR-GARCH: negative shocks can raise volatility more than positive ones
    (h_t = omega + (alpha + gamma*1[eps<0]) eps^2 + beta h). gamma shapes the filter but is not
    emitted; the asymmetry stream carried no signal. Stream: nu^2 - 1, the scale surprise relative
    to the asymmetric volatility model."""
    name = "gjr"
    version = 2
    STREAMS = ("nu2",)
    GAMMAS = (0.0, 0.05, 0.12, 0.25)

    @staticmethod
    def _nll(al, be, ga, x, v):
        om = v * (1.0 - al - ga / 2.0 - be)
        if om <= 1e-12 or al + ga / 2.0 + be >= 0.99:
            return 1e18
        h, s = v, 0.0
        for e in x:
            e2 = e * e
            s += math.log(h) + e2 / h
            h = om + (al + (ga if e < 0 else 0.0)) * e2 + be * h
        return s

    def names(self):
        return self._agg_names()

    def init(self, hist):
        xs = _history(hist)
        self.mu, self.sd = float(hist.mu), (float(hist.sd) if hist.sd > 0 else 1.0)
        eps = xs - self.mu
        v = max(float(np.var(eps)), 1e-10)
        al, be, ga = _garch_grid(eps[-600:], v, extra=self.GAMMAS, nll=self._nll)
        self.al, self.be, self.ga, self.v0 = al, be, ga, v
        self.om = v * (1 - al - ga / 2 - be)
        h = v
        for e in eps:
            h = self.om + (al + (ga if e < 0 else 0.0)) * e * e + be * h
        self.h = max(h, 1e-12)
        self._setup_buffers()

    def update(self, ctx):
        eps = float(ctx.zr) * self.sd
        h = max(self.h, 1e-12)
        nu2 = min(eps * eps / h, 100.0)
        self.h = self.om + (self.al + (self.ga if eps < 0 else 0.0)) * eps * eps + self.be * h
        self._push({"nu2": (nu2 - 1.0) * _SQ2})
        return self._aggregate()

# ---------------------------------------------------------------- 4. CAViaR
class CaviarFamily(_CondBase):
    """CAViaR (Engle & Manganelli 2004), symmetric absolute-value form, on |eps|:
        q_t = b0 + b1*q_{t-1} + b2*|eps_{t-1}|
    one model per quantile level theta, fit on the history by minimizing the pinball (quantile) loss.
    Under the null, P(|eps_t| > q_t) = 1 - theta at EVERY step regardless of volatility clustering,
    so the exceedance rate is a model-based test of the tails. Streams: standardized hit indicator
    per theta, plus log(|eps|/q) at the higher level (how far past the quantile)."""
    name = "caviar"
    THETAS = (0.90, 0.975)
    STREAMS = ("hit90", "hit975", "excess")

    @staticmethod
    def _loss(b1, b2, a, theta, qbar, mabs):
        b0 = qbar * (1.0 - b1) - b2 * mabs
        if b0 <= 0:
            return 1e18, b0
        q, s = qbar, 0.0
        for x in a:
            u = x - q
            s += (theta - (1.0 if u < 0 else 0.0)) * u
            q = b0 + b1 * q + b2 * x
        return s, b0

    def _fit(self, a, theta):
        qbar = float(np.quantile(a, theta)); mabs = float(a.mean())
        best, bp = None, None
        for b1 in (0.5, 0.7, 0.8, 0.9, 0.95):
            for b2 in (0.03, 0.08, 0.15, 0.25, 0.4):
                f, b0 = self._loss(b1, b2, a[-600:], theta, qbar, mabs)
                if f < 1e17 and (best is None or f < best):
                    best, bp = f, (b0, b1, b2)
        if bp is None:                                   # static quantile fallback
            bp = (qbar, 0.0, 0.0)
        return bp, qbar

    def names(self):
        return self._agg_names()

    def init(self, hist):
        xs = _history(hist)
        self.mu, self.sd = float(hist.mu), (float(hist.sd) if hist.sd > 0 else 1.0)
        a = np.abs(xs - self.mu)
        self.m = []
        for th in self.THETAS:
            (b0, b1, b2), qbar = self._fit(a, th)
            q = qbar
            for x in a:
                q = b0 + b1 * q + b2 * x
            self.m.append([b0, b1, b2, max(q, 1e-12), th])
        self._setup_buffers()

    def update(self, ctx):
        x = abs(float(ctx.zr) * self.sd)
        vals = {}
        for key, mdl in zip(("hit90", "hit975"), self.m):
            b0, b1, b2, q, th = mdl
            p = 1.0 - th
            vals[key] = ((1.0 if x > q else 0.0) - p) / math.sqrt(p * (1.0 - p))
            mdl[3] = max(b0 + b1 * q + b2 * x, 1e-12)
        vals["excess"] = math.log(max(x, 1e-12) / self.m[1][3]) if x > 0 else 0.0
        self._push(vals)
        return self._aggregate()

In [ ]:
test_family(ARGarchFamily, force=True)

In [ ]:
test_family(TGarchFamily, force=True)

In [ ]:
keep_family("tgarch")

In [ ]:
test_family(GJRGarchFamily, force=True)

In [ ]:
test_family(CaviarFamily, force=True)

In [ ]:
import numpy as np, gc, os, time

def load_fam(name, drop=()):
    Z = np.load(f"{CACHE}/new_{name}.npz")
    nm = [str(s) for s in Z["names"]]
    keep = [i for i, n in enumerate(nm) if n not in set(drop)]
    X = np.ascontiguousarray(np.asarray(Z["X"], dtype=np.float32)[:, keep]); del Z; gc.collect()
    return X, [nm[i] for i in keep]

def ensure_fam(cls, drop=()):
    """Load the cached extraction, or extract once (fork backend, row-aligned to the cache)."""
    p = f"{CACHE}/new_{cls.name}.npz"
    if not os.path.exists(p):
        REGISTRY[:] = [f for f in REGISTRY if f.name != cls.name]
        REGISTRY.append(cls())
        names = active_feature_names({cls.name})
        t0 = time.time()
        X, yn, tn, _, gn = build_dataset(list(train_data), subsample=False,
                                         active_families={cls.name}, backend="fork")
        assert X.shape[1] == len(names), (X.shape, len(names))
        assert np.array_equal(yn, y_all) and np.array_equal(tn, t_all) and np.array_equal(gn, g_all)
        np.savez(p, X=X, names=np.array(names))
        print(f"extracted {cls.name} {X.shape} in {(time.time()-t0)/60:.1f} min"); del X; gc.collect()
    return load_fam(cls.name, drop)

assert getattr(_engine.HistoricalRef, "_keeps_series", False), "run the HistoricalRef patch cell first"

XG, NG = load_fam("garch", drop=("garch_persist",))
XT, NT = load_fam("tgarch")
XA, NA = ensure_fam(ARGarchFamily)
XJ, NJ = ensure_fam(GJRGarchFamily, drop=[n for n in load_fam("gjr")[1]
                                          if "asym" in n or n.endswith("_gamma")]
                    if os.path.exists(f"{CACHE}/new_gjr.npz") else ())
CUR = np.hstack([XG, XT]); NCUR = NG + NT
print("current extra:", CUR.shape[1], "| argarch:", len(NA), "| gjr:", len(NJ), NJ)
assert len(PROD) == 73

In [ ]:
FOLD_IDX = build_folds(7); CMP.clear()
cv_prod_plus("curr", CUR, NCUR)

In [ ]:
cv_prod_plus("curr + gjr", np.hstack([CUR, XJ]), NCUR + NJ)

In [ ]:
cv_prod_plus("curr + argarch", np.hstack([CUR, XA]), NCUR + NA)

In [ ]:
cv_prod_plus("curr + gjr + argarch", np.hstack([CUR, XJ, XA]), NCUR + NJ + NA)

In [ ]:
cmp_table("curr")

In [ ]:
from scipy.special import digamma as _digamma


class GarchScoreFamily(_CondBase):
    """'GARCH the other way round': instead of asking whether the online data is surprising under
    the history's parameters, ask which way the online data wants the parameters to MOVE.

    A Student-t GARCH(1,1) is fit on the ordered history (same fit as `tgarch`). For every new
    observation we evaluate the SCORE -- the gradient of its log-likelihood w.r.t. each parameter --
    at theta_hist = (omega, alpha, beta, nu). Under no break the scores average to zero; after a
    break their running sums drift, and each component points at a specific parameter:
        omega  volatility level    alpha  shock propagation    beta  persistence    nu  tail weight
    The Lagrange-multiplier / Nyblom-Hansen stability test, run sequentially. Scores are standardized
    by their mean and covariance over the HISTORY, plus a joint chi-square. O(1) per step."""
    name = "gscore"
    version = 1
    WINS = (128, 512)
    COMP = ("om", "al", "be", "nu")
    STREAMS = COMP
    NUS = (4.0, 6.0, 10.0, 30.0)

    def names(self):
        return self._agg_names() + [f"{self.name}_chi2_w128", f"{self.name}_chi2_exp"]

    def _score(self, eps):
        h = max(self.h, 1e-12)
        nu = self.nu
        q = eps * eps / ((nu - 2.0) * h)
        dl_dh = -0.5 / h + 0.5 * (nu + 1.0) * q / ((1.0 + q) * h)
        g = np.array([dl_dh * self.dh[0], dl_dh * self.dh[1], dl_dh * self.dh[2],
                      0.5 * _digamma((nu + 1) / 2) - 0.5 * _digamma(nu / 2) - 0.5 / (nu - 2.0)
                      - 0.5 * math.log1p(q) + 0.5 * (nu + 1.0) * q / ((1.0 + q) * (nu - 2.0))])
        e2 = eps * eps
        self.dh = np.array([1.0 + self.be * self.dh[0],
                            e2 + self.be * self.dh[1],
                            h + self.be * self.dh[2]])
        self.h = self.om + self.al * e2 + self.be * h
        return g

    def init(self, hist):
        xs = _history(hist)
        self.mu, self.sd = float(hist.mu), (float(hist.sd) if hist.sd > 0 else 1.0)
        eps = xs - self.mu
        v = max(float(np.var(eps)), 1e-10)
        al, be, nu = _garch_grid((eps ** 2)[-600:], v, extra=self.NUS, nll=TGarchFamily._nll)
        self.al, self.be, self.nu = al, be, nu
        self.om = v * (1 - al - be)
        self.h, self.dh = v, np.zeros(3)
        G = np.empty((eps.size, 4))
        for i, e in enumerate(eps):
            G[i] = self._score(e)
        G = G[min(50, eps.size // 4):]
        self.gm = G.mean(axis=0)
        self.gs = np.maximum(G.std(axis=0), 1e-12)
        Z = (G - self.gm) / self.gs
        self.Cinv = np.linalg.inv(np.cov(Z.T) + 1e-3 * np.eye(4))
        self._setup_buffers()

    def _joint(self, W):
        zs = np.stack([self._win(self.buf[c], W) for c in self.COMP], axis=1)
        m = zs.shape[0]
        if m < 8:
            return 0.0
        zbar = zs.mean(axis=0)
        return float(m * zbar @ self.Cinv @ zbar)

    def update(self, ctx):
        eps = float(ctx.zr) * self.sd
        z = np.clip((self._score(eps) - self.gm) / self.gs, -20.0, 20.0)
        self._push(dict(zip(self.COMP, z.tolist())))
        acc = np.array([self.acc[c] for c in self.COMP]) / self.n
        chi_exp = float(self.n * acc @ self.Cinv @ acc)
        return self._aggregate() + [_safe(self._joint(128)), _safe(chi_exp)]

In [ ]:
for f in ("garch", "tgarch", "argarch", "gjr"):
    if f not in KEPT and f in EXTRA:
        keep_family(f)
print(KEPT)
test_family(GarchScoreFamily, force=True)

In [ ]:
def _stack(rows, extra_arrays):
    """Production base columns (73) + extra family columns, for the given rows."""
    parts = [np.asarray(X_all[rows])[:, PROD]]
    for a in extra_arrays:
        parts.append(a[rows])
    return np.ascontiguousarray(np.hstack(parts)) if len(parts) > 1 else np.ascontiguousarray(parts[0])

print(_stack(FOLD_IDX[0]["st"][:5], kept_arrays).shape[1], "columns")    # expect 109

In [ ]:
import numpy as np, gc

Z = np.load(f"{CACHE}/new_gscore.npz")
gn = [str(s) for s in Z["names"]]
GX = np.asarray(Z["X"], dtype=np.float32); del Z; gc.collect()

def pick(prefixes):
    cols = [n for n in gn if any(n.startswith(p) for p in prefixes)]
    return np.ascontiguousarray(GX[:, [gn.index(n) for n in cols]]), cols

XA, NA = pick(["gscore_chi2", "gscore_nu"])                     # 6 columns
XB, NB = pick(["gscore_chi2", "gscore_nu", "gscore_om"])        # 10 columns
print(len(NA), NA); print(len(NB), NB)

kept_arrays = [EXTRA[k][0] for k in KEPT]
kept_names = [n for k in KEPT for n in EXTRA[k][1]]
PROD_NAMES = [BASE_NAMES[i] for i in PROD]
print("baseline:", len(PROD_NAMES) + len(kept_names), "columns | kept:", KEPT)   # expect 109
assert len(PROD_NAMES) + len(kept_names) == 109

In [ ]:
SEEDS = (0, 1, 2)            # 3 seeds: the effects here are small enough that noise matters
FOLD_IDX = build_folds(11)
for k in [k for k in CMP if k.startswith("base") or k.startswith("+gscore")]:
    CMP.pop(k)
_cv("base109", kept_arrays, PROD_NAMES + kept_names, seeds=SEEDS)

In [ ]:
_cv("+gscore chi2+nu", kept_arrays + [XA], PROD_NAMES + kept_names + NA, seeds=SEEDS)

In [ ]:
_cv("+gscore chi2+nu+om", kept_arrays + [XB], PROD_NAMES + kept_names + NB, seeds=SEEDS)

In [ ]:
cmp_table("base109")

In [17]:
# =====================================================================
#  Five families from the structural-break literature, all built on the lesson GARCH taught:
#  MODEL THE HISTORY, then measure how the online data departs from that model.
#  They need _CondBase, _history (from the volatility-families cell) and HistoricalRef.x_series.
#
#    ordpat  ordinal-pattern distribution vs the history's (Bandt & Pompe 2002; Sinn, Ghodsi &
#            Keller 2012; Unakafov & Keller 2018) -- dynamics, invariant to any monotone transform
#    wavvar  Haar wavelet variance per time scale, relative to the HISTORY's own wavelet spectrum
#            (Percival & Walden 2000; Whitcher, Guttorp & Percival 2000)
#    recres  recursive residuals of the history's AR model + CUSUM / CUSUM-of-squares
#            (Brown, Durbin & Evans 1975; Ploberger & Kraemer 1992) and a sequential Wald distance
#    hmm     2-state Gaussian hidden Markov model fit on the history (Hamilton 1989):
#            predictive surprise and regime occupancy
#    evt     peaks-over-threshold / generalized Pareto tail model of the history
#            (Pickands 1975; Balkema & de Haan 1974): exceedance rate, excess likelihood, excess PIT
# =====================================================================
import itertools
from scipy.special import ndtri as _ndtri_l3


def _std_history(hist):
    xs = _history(hist)
    sd = float(hist.sd) if hist.sd > 0 else 1.0
    return (xs - float(hist.mu)) / sd


# ------------------------------------------------------------------ 1. ordinal patterns
class OrdinalPatternFamily(_CondBase):
    """Each window of d consecutive values is reduced to the permutation that sorts it (its
    'ordinal pattern'). The pattern distribution encodes the dynamics -- trends, oscillation,
    dependence -- and is unchanged by any monotone transform, so it ignores scale entirely.
    The history's pattern probabilities are the reference; streams are the per-step surprise
    -log p_hist(pattern) - H_hist for d = 3 and 4 (mean zero under no break), the monotone-pattern
    rate minus its historical rate, plus a KL divergence of the last 128 patterns (d = 3)."""
    name = "ordpat"
    version = 1
    STREAMS = ("s3", "s4", "mono")
    KL_W = 128

    def names(self):
        return self._agg_names() + ["ordpat_kl3_w128"]

    @staticmethod
    def _table(d):
        return {p: i for i, p in enumerate(itertools.permutations(range(d)))}

    @staticmethod
    def _code(vals, tab):
        return tab[tuple(sorted(range(len(vals)), key=lambda i: vals[i]))]

    def init(self, hist):
        z = _std_history(hist)
        self.t3, self.t4 = self._table(3), self._table(4)
        c3 = np.ones(6); c4 = np.ones(24)
        for i in range(3, z.size):
            c4[self._code(z[i - 3:i + 1], self.t4)] += 1
            c3[self._code(z[i - 2:i + 1], self.t3)] += 1
        self.p3, self.p4 = c3 / c3.sum(), c4 / c4.sum()
        self.lp3, self.lp4 = -np.log(self.p3), -np.log(self.p4)
        self.h3, self.h4 = float(self.p3 @ self.lp3), float(self.p4 @ self.lp4)
        self.mono = {self.t3[(0, 1, 2)], self.t3[(2, 1, 0)]}
        self.pmono = float(sum(self.p3[i] for i in self.mono))
        self.last = list(z[-3:])
        self.kbuf = np.zeros(self.KL_W, dtype=np.int64); self.kcnt = np.zeros(6); self.kn = 0
        self._setup_buffers()

    def update(self, ctx):
        v = float(ctx.zr)
        w4 = self.last + [v]
        self.last = w4[1:]
        c3 = self._code(w4[1:], self.t3); c4 = self._code(w4, self.t4)
        i = self.kn % self.KL_W
        if self.kn >= self.KL_W:
            self.kcnt[self.kbuf[i]] -= 1
        self.kbuf[i] = c3; self.kcnt[c3] += 1; self.kn += 1
        self._push({"s3": self.lp3[c3] - self.h3, "s4": self.lp4[c4] - self.h4,
                    "mono": (1.0 if c3 in self.mono else 0.0) - self.pmono})
        m = min(self.kn, self.KL_W)
        q = (self.kcnt + 0.5) / (m + 3.0)
        return self._aggregate() + [_safe(float(m * np.sum(q * np.log(q / self.p3))))]


# ------------------------------------------------------------------ 2. wavelet variance by scale
class WaveletVarianceFamily(_CondBase):
    """Haar wavelet coefficients at scales L = 1, 2, 4, 8:
        d_L(t) = (sum of the last L values - sum of the L before) / sqrt(2L)
    Their variance at each scale is the series' energy in that frequency band. The HISTORY's
    wavelet variance per scale is the reference, so each stream d_L^2 / v_hist(L) - 1 is zero-mean
    under no break whatever the history's own spectrum looks like -- the 'relative to the history's
    own model' step the plain spectral features lacked."""
    name = "wavvar"
    version = 1
    SCALES = (1, 2, 4, 8)
    STREAMS = tuple(f"L{L}" for L in SCALES)

    def names(self):
        return self._agg_names()

    def init(self, hist):
        z = _std_history(hist)
        c = np.concatenate([[0.0], np.cumsum(z)])
        self.vh = {}
        for L in self.SCALES:
            if z.size < 4 * L:
                self.vh[L] = 1.0
                continue
            s_now = c[2 * L:] - c[L:-L]
            s_prev = c[L:-L] - c[:-2 * L]
            d = (s_now - s_prev) / math.sqrt(2 * L)
            self.vh[L] = max(float(np.mean(d * d)), 1e-8)
        self.win = list(z[-2 * max(self.SCALES):])
        self._setup_buffers()

    def update(self, ctx):
        self.win = self.win[1:] + [float(ctx.zr)]
        vals = {}
        for L in self.SCALES:
            a = sum(self.win[-L:]); b = sum(self.win[-2 * L:-L])
            d = (a - b) / math.sqrt(2 * L)
            vals[f"L{L}"] = d * d / self.vh[L] - 1.0
        self._push(vals)
        return self._aggregate()


# ------------------------------------------------------------------ 3. recursive residuals
class RecursiveResidualFamily(_CondBase):
    """The history's AR(3) regression (with intercept) is fit by OLS, then updated online by
    recursive least squares with forgetting factor LAM. The one-step recursive residual
        w_t = (z_t - x_t' beta_{t-1}) / sqrt(LAM + x_t' P_{t-1} x_t) / sigma_hist
    is iid N(0,1)-like while the regression holds (Brown-Durbin-Evans). Streams: w (its CUSUM
    detects drift in any coefficient, including the intercept) and w^2 - 1 (CUSUM of squares:
    variance). Plus a sequential Wald distance between the tracked coefficients and the
    history's, scaled by the history's design matrix -- how far the dynamics have moved."""
    name = "recres"
    version = 1
    STREAMS = ("w", "w2")
    P_AR = 3
    LAM = 0.995

    def names(self):
        return self._agg_names() + ["recres_wald"]

    def init(self, hist):
        z = _std_history(hist)
        p = self.P_AR
        X = np.column_stack([np.ones(z.size - p)] + [z[p - k:z.size - k] for k in range(1, p + 1)])
        y = z[p:]
        XtX = X.T @ X + 1e-6 * np.eye(p + 1)
        self.beta_h = np.linalg.solve(XtX, X.T @ y)
        r = y - X @ self.beta_h
        self.sig = max(float(np.std(r)), 1e-6)
        self.beta = self.beta_h.copy()
        self.P = np.linalg.inv(XtX) * (X.shape[0] * (1.0 - self.LAM))   # as if the history were
        self.M = XtX / X.shape[0] / (1.0 - self.LAM)                      # an LAM-weighted sample
        self.lags = list(z[-p:][::-1])
        self._setup_buffers()

    def update(self, ctx):
        z = float(ctx.zr)
        x = np.array([1.0] + self.lags)
        Px = self.P @ x
        f = self.LAM + float(x @ Px)
        e = z - float(x @ self.beta)
        w = e / math.sqrt(max(f, 1e-12)) / self.sig
        k = Px / f
        self.beta = self.beta + k * e
        self.P = (self.P - np.outer(k, Px)) / self.LAM
        self.lags = [z] + self.lags[:-1]
        w = max(min(w, 20.0), -20.0)
        self._push({"w": w, "w2": (w * w - 1.0) * 0.7071067811865476})
        db = self.beta - self.beta_h
        return self._aggregate() + [_safe(float(db @ self.M @ db) / (self.sig ** 2))]


# ------------------------------------------------------------------ 4. hidden Markov regimes
class HMMRegimeFamily(_CondBase):
    """A 2-state Gaussian HMM (Hamilton 1989) is fit on the history by EM (scaled forward-backward).
    Online, the forward filter gives the predictive density of each new point and the filtered
    probability of the high-variance regime. Streams: predictive surprise
    -log p(z_t | past) minus its historical average, and high-regime occupancy minus its
    historical average. Unlike GARCH, the volatility moves by SWITCHING between two levels, so a
    break that changes how often or how long the process visits each regime shows up here."""
    name = "hmm"
    version = 1
    STREAMS = ("surp", "hi")
    EM_ITERS = 12
    FIT_LAST = 800

    def names(self):
        return self._agg_names() + ["hmm_p_hi"]

    @staticmethod
    def _dens(x, mu, var):
        return np.exp(-0.5 * (x - mu) ** 2 / var) / np.sqrt(2 * np.pi * var)

    def _fit(self, z):
        v = max(float(np.var(z)), 1e-6)
        mu = np.array([float(np.mean(z)), float(np.mean(z))])
        var = np.array([0.5 * v, 2.0 * v])
        A = np.array([[0.97, 0.03], [0.03, 0.97]])
        pi = np.array([0.5, 0.5])
        T = z.size
        for _ in range(self.EM_ITERS):
            B = np.stack([self._dens(z, mu[0], var[0]), self._dens(z, mu[1], var[1])], axis=1) + 1e-300
            al = np.empty((T, 2)); c = np.empty(T)
            a = pi * B[0]; c[0] = a.sum(); al[0] = a / c[0]
            for t in range(1, T):
                a = (al[t - 1] @ A) * B[t]; c[t] = a.sum(); al[t] = a / c[t]
            be = np.empty((T, 2)); be[-1] = 1.0
            for t in range(T - 2, -1, -1):
                be[t] = (A @ (B[t + 1] * be[t + 1])) / c[t + 1]
            g = al * be; g /= g.sum(axis=1, keepdims=True)
            xi = (al[:-1, :, None] * A[None] * (B[1:] * be[1:])[:, None, :]) / c[1:, None, None]
            A = xi.sum(axis=0); A /= A.sum(axis=1, keepdims=True)
            pi = g[0]
            wsum = g.sum(axis=0) + 1e-12
            mu = (g * z[:, None]).sum(axis=0) / wsum
            var = np.maximum((g * (z[:, None] - mu) ** 2).sum(axis=0) / wsum, 1e-4 * v)
        if var[0] > var[1]:                                     # state 1 = high variance
            mu, var, A = mu[::-1], var[::-1], A[::-1, ::-1]
        return mu, var, A

    def init(self, hist):
        z = _std_history(hist)[-self.FIT_LAST:]
        self.mu, self.var, self.A = self._fit(z)
        al = np.array([0.5, 0.5]); surp, hi = [], []
        for x in z:                                             # historical averages of the streams
            pr = al @ self.A
            b = self._dens(x, self.mu, self.var) + 1e-300
            p = float(pr @ b); al = pr * b / p
            surp.append(-math.log(p)); hi.append(al[1])
        self.al = al
        self.surp0, self.hi0 = float(np.mean(surp)), float(np.mean(hi))
        self._setup_buffers()

    def update(self, ctx):
        x = float(ctx.zr)
        pr = self.al @ self.A
        b = self._dens(x, self.mu, self.var) + 1e-300
        p = float(pr @ b)
        self.al = pr * b / p
        self._push({"surp": -math.log(max(p, 1e-300)) - self.surp0, "hi": float(self.al[1]) - self.hi0})
        return self._aggregate() + [_safe(float(self.al[1]))]


# ------------------------------------------------------------------ 5. extreme-value tail model
class EVTTailFamily(_CondBase):
    """Peaks over threshold: u = 95th percentile of |z| in the history; the excesses |z| - u are
    fit with a generalized Pareto distribution (method of moments). Streams, all zero-mean under
    no break: the standardized exceedance rate; for exceedances, the GPD negative log-likelihood of
    the excess minus its historical average (how implausible the SIZE of the extreme is); and the
    normal score of the excess's GPD CDF (whether extremes land deeper in the tail). The existing
    tail family counts exceedances of fixed thresholds; this models the tail's SHAPE."""
    name = "evt"
    version = 1
    STREAMS = ("rate", "nll", "pit")
    Q = 0.95

    def names(self):
        return self._agg_names() + ["evt_xi"]

    def _gpd_nll(self, y):
        if abs(self.xi) < 1e-6:
            return math.log(self.sg) + y / self.sg
        a = 1.0 + self.xi * y / self.sg
        if a <= 1e-12:
            return 50.0
        return math.log(self.sg) + (1.0 / self.xi + 1.0) * math.log(a)

    def _gpd_cdf(self, y):
        if abs(self.xi) < 1e-6:
            return 1.0 - math.exp(-y / self.sg)
        a = 1.0 + self.xi * y / self.sg
        return 1.0 if a <= 0 else 1.0 - a ** (-1.0 / self.xi)

    def init(self, hist):
        az = np.abs(_std_history(hist))
        self.u = float(np.quantile(az, self.Q))
        ex = az[az > self.u] - self.u
        if ex.size >= 20:
            m, v = float(ex.mean()), float(ex.var())
            r = m * m / max(v, 1e-12)
            self.xi = float(np.clip(0.5 * (1.0 - r), -0.45, 0.8))
            self.sg = max(0.5 * m * (1.0 + r), 1e-6)
        else:
            self.xi, self.sg = 0.1, max(float(ex.mean()) if ex.size else 0.3, 1e-6)
        self.p = max(float((az > self.u).mean()), 1e-3)
        nll_h = [self._gpd_nll(y) for y in ex] if ex.size else [0.0]
        self.nll0 = float(np.mean(nll_h))
        self._setup_buffers()

    def update(self, ctx):
        a = abs(float(ctx.zr))
        hit = a > self.u
        vals = {"rate": ((1.0 if hit else 0.0) - self.p) / math.sqrt(self.p * (1.0 - self.p)),
                "nll": 0.0, "pit": 0.0}
        if hit:
            y = a - self.u
            vals["nll"] = min(self._gpd_nll(y) - self.nll0, 50.0)
            F = min(max(self._gpd_cdf(y), 1e-9), 1 - 1e-9)
            vals["pit"] = float(_ndtri_l3(F))
        self._push(vals)
        return self._aggregate() + [_safe(self.xi)]

In [18]:
test_family(RecursiveResidualFamily, force=True)

== recres: 11 features -> ['recres_w_w32', 'recres_w_w128', 'recres_w_w512', 'recres_w_ew', 'recres_w_exp', 'recres_w2_w32', 'recres_w2_w128', 'recres_w2_w512', 'recres_w2_ew', 'recres_w2_exp', 'recres_wald']
   smoke test OK
build_dataset: fork Pool over 10000 series (chunksize 8)
   extracted (5036517, 11) in 0.3 min


KeyboardInterrupt: 

In [ ]:
test_family(WaveletVarianceFamily, force=True)

In [ ]:
test_family(OrdinalPatternFamily, force=True)

In [ ]:
test_family(EVTTailFamily, force=True)

In [ ]:
test_family(HMMRegimeFamily, force=True)

In [19]:
# %% [markdown]
# ## Bucket 3 pruning: keep-one-in ablation
# Bucket 3 = centered spread of `e`: ftest, bartlett, levene, sigtuk, lmom_lscale (each at w32/w64/w256).
#
# * `noB3`  = the shipped 151 minus all 15 bucket-3 columns (136)
# * keep-one = noB3 + one test (all three windows), for each of the five tests
# * forward  = add a second (third...) test only if it clears `delta > 2*se` and 5/6 over the current set
# * final    = selected set vs full151 with the pruning rule `delta > -2*se`
#
# Prereqs (a fresh kernel is safest, so REGISTRY holds only the shipped families):
# section 0 (import _engine), section 1 (cache), section 2 (build_folds, CMP),
# and the ONE-CALL FAMILY TEST cell (defines `_cv` and `_stack`).
# Every finished run is written to cache/b3_results.json, so a crash or restart
# resumes where it stopped: re-run cell 1, then the cell you were on.

# %% [1] setup: rebuild the shipped 151 as PROD (cache, minus bucket 3) + fixed extra arrays
import os, json, gc, time
import numpy as np, pandas as pd

SHIPPED = ["z2", "e2", "tail", "time", "ftest", "bartlett", "levene", "lmom", "sigtuk",
           "wass", "energy", "mmd", "entropy", "bocpd",
           "garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]
EXTRA_FAMS = ["garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]   # live in cache/new_<fam>.npz

B3_TESTS = {
    "ftest":    [f"ftest_e_w{W}"       for W in (32, 64, 256)],
    "bartlett": [f"bartlett_e_w{W}"    for W in (32, 64, 256)],
    "levene":   [f"levene_e_w{W}"      for W in (32, 64, 256)],
    "sigtuk":   [f"sigtuk_e_w{W}"      for W in (32, 64, 256)],
    "lscale":   [f"lmom_lscale_e_w{W}" for W in (32, 64, 256)],
}
B3_ALL = [n for v in B3_TESTS.values() for n in v]

# the shipped feature names, straight from the engine's family classes
FAM_NAMES = {f.name: list(f.names()) for f in REGISTRY if f.name in SHIPPED}
assert set(FAM_NAMES) == set(SHIPPED), f"REGISTRY lacks {set(SHIPPED) - set(FAM_NAMES)} -- re-run section 0"
SHIP_NAMES = [n for fam in SHIPPED for n in FAM_NAMES[fam]]
assert len(SHIP_NAMES) == len(set(SHIP_NAMES)) == 151, len(SHIP_NAMES)
assert set(B3_ALL) <= set(SHIP_NAMES), set(B3_ALL) - set(SHIP_NAMES)

# 85-column cache: the 14 original families live here (mood/ansari/iqr/fligner are simply not selected)
X_all = np.load(f"{CACHE}/X_all.npy", mmap_mode="r")
BASE_NAMES = json.load(open(f"{CACHE}/names.json"))
assert len(BASE_NAMES) == X_all.shape[1] and X_all.shape[0] == y_all.size
cache_names = [n for fam in SHIPPED if fam not in EXTRA_FAMS for n in FAM_NAMES[fam]]
_miss = [n for n in cache_names if n not in BASE_NAMES]
assert not _miss, f"not in the 85-col cache: {_miss}"
assert len(cache_names) == 73, len(cache_names)

# _stack() reads the global PROD: fix it to the cached shipped columns MINUS bucket 3.
# Bucket 3 travels as extra arrays instead, so every config is just a different list of arrays.
PROD = [BASE_NAMES.index(n) for n in cache_names if n not in B3_ALL]
PROD_NAMES = [BASE_NAMES[i] for i in PROD]
assert len(PROD) == 58, len(PROD)


def load_shipped(fam):
    """The shipped columns of one extra family, in engine order (drops garch_persist, gjr asym, ...)."""
    want = FAM_NAMES[fam]
    with np.load(f"{CACHE}/new_{fam}.npz") as Z:
        nm = [str(s) for s in Z["names"]]
        miss = [n for n in want if n not in nm]
        assert not miss, f"{fam}: cached extraction lacks {miss} -- re-extract it with force=True"
        X = np.asarray(Z["X"], dtype=np.float32)
    X = np.ascontiguousarray(X[:, [nm.index(n) for n in want]])
    assert X.shape[0] == y_all.size and np.isfinite(X).all(), fam
    return X, want


REST_ARRAYS, REST_NAMES = [], []
for fam in EXTRA_FAMS:
    _X, _nm = load_shipped(fam)
    REST_ARRAYS.append(_X); REST_NAMES += _nm
    print(f"  {fam:8s} {_X.shape[1]:2d} cols")
del _X; gc.collect()

# bucket-3 columns out of the mmap in ONE pass, then one small array per test (~60 MB each)
_blk = np.asarray(X_all[:, [BASE_NAMES.index(n) for n in B3_ALL]], dtype=np.float32)
B3_X = {t: np.ascontiguousarray(_blk[:, [B3_ALL.index(n) for n in cols]]) for t, cols in B3_TESTS.items()}
del _blk; gc.collect()

assert len(PROD_NAMES) + len(REST_NAMES) + len(B3_ALL) == 151
assert set(PROD_NAMES + REST_NAMES + B3_ALL) == set(SHIP_NAMES)

FOLD_SEED, SEEDS = 11, (0, 1, 2)
FOLD_IDX = build_folds(FOLD_SEED)
RES_PATH = f"{CACHE}/b3_results.json"
RES = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}
print(f"PROD {len(PROD)} | extra {len(REST_NAMES)} | bucket 3 {len(B3_ALL)} | cached runs: {list(RES)}")


def _label(tests):
    if not tests:
        return "noB3"
    if set(tests) == set(B3_TESTS):
        return "full151"
    return "keep[" + "+".join(sorted(tests)) + "]"


def run(tests):
    """CV of noB3 + the listed bucket-3 tests (all three windows each). Resumes from RES_PATH."""
    tests = list(tests)
    label = _label(tests)
    r = RES.get(label)
    if r is not None:
        assert (r["fold_seed"] == FOLD_SEED and r["seeds"] == list(SEEDS)
                and sorted(r["tests"]) == sorted(tests)), \
            f"{label}: cached with different settings -- delete it from {RES_PATH}"
        CMP[label] = np.array(r["aucs"])
        print(f"[{label}] cached | mean {CMP[label].mean():.5f}")
        return label
    arrays = REST_ARRAYS + [B3_X[t] for t in tests]
    names = PROD_NAMES + REST_NAMES + [n for t in tests for n in B3_TESTS[t]]
    t0 = time.time()
    aucs = _cv(label, arrays, names, seeds=SEEDS)
    RES[label] = dict(tests=tests, aucs=[float(a) for a in aucs], n_feats=len(names),
                      fold_seed=FOLD_SEED, seeds=list(SEEDS))
    with open(RES_PATH + ".tmp", "w") as f:
        json.dump(RES, f, indent=1)
    os.replace(RES_PATH + ".tmp", RES_PATH)
    print(f"   {label}: {len(names)} features | {(time.time() - t0) / 60:.1f} min")
    gc.collect()
    return label


def paired(a, b):
    """delta of config a over config b on identical fold-halves: (mean, se, n_better, n)."""
    d = CMP[a] - CMP[b]
    return float(d.mean()), float(d.std(ddof=1) / np.sqrt(d.size)), int((d > 0).sum()), d.size


def passes_add(a, b):
    """The adoption rule for ADDING: delta > 2*se and better on all but at most one fold-half."""
    m, se, k, n = paired(a, b)
    return m > 2 * se and k >= n - 1


def report(ref, labels):
    rows = []
    for lab in labels:
        m, se, k, n = paired(lab, ref) if lab != ref else (0.0, 0.0, 0, CMP[lab].size)
        rows.append(dict(config=lab, feats=RES[lab]["n_feats"], mean=CMP[lab].mean(), delta=m, se=se,
                         sigma=(m / se if se > 0 else 0.0), better=f"{k}/{n}"))
    df = pd.DataFrame(rows).sort_values("mean", ascending=False).round(5)
    print(f"\n--- vs {ref} ---\n" + df.to_string(index=False))
    return df


# %% [2] reference runs: full151 and noB3 (does the bucket matter at all?)
FULL = run(list(B3_TESTS))
NONE = run([])
assert RES[FULL]["n_feats"] == 151 and RES[NONE]["n_feats"] == 136
m, se, k, n = paired(NONE, FULL)
print(f"\nnoB3 vs full151: {m:+.5f} | se {se:.5f} | {m / se:+.1f} sigma | better {k}/{n}")
print("-> without bucket 3 the model is",
      "NON-INFERIOR (the bucket may be fully redundant)" if m > -2 * se else "measurably WORSE (something in it is needed)")


# %% [3] keep-one-in: noB3 + each test on its own
ONE = {t: run([t]) for t in B3_TESTS}
report(NONE, [NONE] + list(ONE.values()))

best = max(ONE, key=lambda t: CMP[ONE[t]].mean())
tied = [t for t in ONE if t == best or paired(ONE[t], ONE[best])[0] > -2 * paired(ONE[t], ONE[best])[1]]
WINNER = "ftest" if "ftest" in tied else best       # tie-break by design: signed, log-scale
print(f"\nhighest mean: {best} | statistically tied with it: {tied} | winner: {WINNER}")
m, se, k, n = paired(ONE[WINNER], NONE)
print(f"{WINNER} over noB3: {m:+.5f} | se {se:.5f} | {m / se:+.1f} sigma | better {k}/{n}")
SEL = [WINNER] if passes_add(ONE[WINNER], NONE) else []
print("-> first pick:", SEL or "NONE (no single test adds measurably over noB3)")


# %% [4] forward selection on top of the winner (stops at the first step that fails the add rule)
while SEL and len(SEL) < len(B3_TESTS):
    cur = run(SEL)
    cand = {t: run(SEL + [t]) for t in B3_TESTS if t not in SEL}
    report(cur, [cur] + list(cand.values()))
    t_best = max(cand, key=lambda t: CMP[cand[t]].mean())
    m, se, k, n = paired(cand[t_best], cur)
    if passes_add(cand[t_best], cur):
        SEL.append(t_best)
        print(f"-> ADD {t_best}: {m:+.5f} ({m / se:.1f} sigma, {k}/{n})")
    else:
        print(f"-> STOP: best addition {t_best} gives {m:+.5f} ({(m / se if se else 0):.1f} sigma, {k}/{n})")
        break
print("\nselected bucket-3 tests:", SEL or "none")


# %% [5] final verdict: selected set vs full151 (pruning rule: delta > -2*se)
def verdict(sel):
    lab = run(sel)
    m, se, k, n = paired(lab, FULL)
    ok = m > -2 * se
    print(f"{lab} vs full151: {m:+.5f} | se {se:.5f} | {m / se:+.1f} sigma | better {k}/{n} "
          f"-> {'NON-INFERIOR' if ok else 'LOSES'}")
    return ok


ok = verdict(SEL)
if not ok and WINNER not in SEL:            # fallback: the forward pass dropped everything but that lost
    print("fallback: keeping the winner alone")
    SEL = [WINNER]
    ok = verdict(SEL)

dropped = [c for t in B3_TESTS if t not in SEL for c in B3_TESTS[t]]
gone = [fam for fam in SHIPPED if set(FAM_NAMES[fam]) <= set(dropped)]
print("\nVERDICT:", "ADOPT" if ok else "KEEP ALL OF BUCKET 3")
if ok:
    print(f"keep  : {SEL or 'none'}")
    print(f"drop  : {len(dropped)} columns -> {dropped}")
    print(f"whole families removable (these also save extraction time): {gone or 'none'}")
    if any(c.startswith("lmom_") for c in dropped):
        print("note: dropping lscale only removes columns; lmom still runs for lskew/lkurt")

  garch     9 cols
  argarch  11 cols
  tgarch   11 cols
  gjr       5 cols
  recres   11 cols
  wavvar   20 cols
  hmm      11 cols
fold 0: train 1,020,112 | stop 848,058 | score 849,407
fold 1: train 1,026,629 | stop 813,776 | score 845,268
fold 2: train 1,022,269 | stop 833,755 | score 846,253
PROD 58 | extra 78 | bucket 3 15 | cached runs: []
  [full151] fold 0 swap=False: 0.62013
  [full151] fold 0 swap=True: 0.61719
  [full151] fold 1 swap=False: 0.63014
  [full151] fold 1 swap=True: 0.65663
  [full151] fold 2 swap=False: 0.63222
  [full151] fold 2 swap=True: 0.62022
[full151] mean 0.62942
   full151: 151 features | 13.8 min
  [noB3] fold 0 swap=False: 0.62114
  [noB3] fold 0 swap=True: 0.61779
  [noB3] fold 1 swap=False: 0.62985
  [noB3] fold 1 swap=True: 0.65491
  [noB3] fold 2 swap=False: 0.63193
  [noB3] fold 2 swap=True: 0.62139
[noB3] mean 0.62950
   noB3: 136 features | 14.2 min

noB3 vs full151: +0.00008 | se 0.00044 | +0.2 sigma | better 3/6
-> without bucket 3 the model

KeyboardInterrupt: 

In [20]:
# %% [markdown]
# ## Bucket 5 pruning: keep-one-in ablation on the 136-feature base
# Bucket 3 is gone for good. Base = `base136` (the shipped 151 minus the 15 bucket-3 columns).
#
# Bucket 5 = conditional scale surprise, linear -- one stream per volatility model, 5 aggregations each:
#   garch_nu2 | argarch_nu2 | gjr_nu2 | tgarch_z2   x   w32, w128, w512, ew, exp
#
# * `noB5`   = base136 minus all 20 (116)
# * keep-one = noB5 + one model's 5 columns, for each of the four models
# * forward  = add another model only if it clears `delta > 2*se` and 5/6 over the current set
# * final    = selected set vs base136 with the pruning rule `delta > -2*se`
#
# Column order is kept identical to base136 in every config (dropped columns are simply skipped),
# so `base136` here is the exact matrix `noB3` was in the bucket-3 run and its cached result is reused.
#
# Prereqs (fresh kernel is safest): section 0, section 1, section 2, and the ONE-CALL FAMILY TEST
# cell (defines `_cv` / `_stack`). Results persist to cache/b5_results.json; after a crash re-run
# cell 1, then the cell you were on.

# %% [1] setup: base136 = PROD (cache, minus bucket 3) + the seven extra families, order preserved
import os, json, gc, time
import numpy as np, pandas as pd

SHIPPED = ["z2", "e2", "tail", "time", "ftest", "bartlett", "levene", "lmom", "sigtuk",
           "wass", "energy", "mmd", "entropy", "bocpd",
           "garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]
EXTRA_FAMS = ["garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]   # cache/new_<fam>.npz

B3_ALL = [f"{s}_e_w{W}" for s in ("ftest", "bartlett", "levene", "sigtuk", "lmom_lscale")
          for W in (32, 64, 256)]                                       # removed permanently

AGG = ("w32", "w128", "w512", "ew", "exp")
B5_TESTS = {
    "garch":   [f"garch_nu2_{a}" if a != "ew" else "garch_nu2_ew0.02" for a in AGG],
    "argarch": [f"argarch_nu2_{a}" for a in AGG],
    "gjr":     [f"gjr_nu2_{a}" for a in AGG],
    "tgarch":  [f"tgarch_z2_{a}" for a in AGG],
}
B5_ALL = [n for v in B5_TESTS.values() for n in v]
PREFER = ["garch", "argarch", "tgarch", "gjr"]    # tie-break order; gjr last: dropping it removes a family

FAM_NAMES = {f.name: list(f.names()) for f in REGISTRY if f.name in SHIPPED}
assert set(FAM_NAMES) == set(SHIPPED), f"REGISTRY lacks {set(SHIPPED) - set(FAM_NAMES)} -- re-run section 0"
SHIP_NAMES = [n for fam in SHIPPED for n in FAM_NAMES[fam]]
assert len(SHIP_NAMES) == len(set(SHIP_NAMES)) == 151, len(SHIP_NAMES)
for grp in (B3_ALL, B5_ALL):
    assert set(grp) <= set(SHIP_NAMES), set(grp) - set(SHIP_NAMES)

X_all = np.load(f"{CACHE}/X_all.npy", mmap_mode="r")
BASE_NAMES = json.load(open(f"{CACHE}/names.json"))
assert len(BASE_NAMES) == X_all.shape[1] and X_all.shape[0] == y_all.size
cache_names = [n for fam in SHIPPED if fam not in EXTRA_FAMS for n in FAM_NAMES[fam]]
assert not [n for n in cache_names if n not in BASE_NAMES]
assert not set(B5_ALL) & set(cache_names), "bucket 5 must live entirely in the extra families"

PROD = [BASE_NAMES.index(n) for n in cache_names if n not in B3_ALL]   # _stack() reads this global
PROD_NAMES = [BASE_NAMES[i] for i in PROD]
assert len(PROD) == 58, len(PROD)


def load_shipped(fam):
    want = FAM_NAMES[fam]
    with np.load(f"{CACHE}/new_{fam}.npz") as Z:
        nm = [str(s) for s in Z["names"]]
        miss = [n for n in want if n not in nm]
        assert not miss, f"{fam}: cached extraction lacks {miss} -- re-extract it with force=True"
        X = np.asarray(Z["X"], dtype=np.float32)
    X = np.ascontiguousarray(X[:, [nm.index(n) for n in want]])
    assert X.shape[0] == y_all.size and np.isfinite(X).all(), fam
    return X, want


FAM_X = {}
for fam in EXTRA_FAMS:
    FAM_X[fam] = load_shipped(fam)
    print(f"  {fam:8s} {FAM_X[fam][0].shape[1]:2d} cols")
gc.collect()


class ColSel:
    """A column subset of a family array WITHOUT copying all rows: _stack only calls a[rows] and
    _cv only reads a.shape, so the selection happens per slice, not on the full 5M rows."""
    def __init__(self, X, cols):
        self.X, self.cols = X, np.asarray(cols, dtype=np.int64)
        self.shape = (X.shape[0], self.cols.size)

    def __getitem__(self, rows):
        return np.asarray(self.X[rows])[:, self.cols]


def config(drop):
    """Extra arrays + full name list for base136 minus `drop`, in base136 column order."""
    drop = set(drop)
    arrays, names = [], []
    for fam in EXTRA_FAMS:
        X, nm = FAM_X[fam]
        keep = [i for i, n in enumerate(nm) if n not in drop]
        if not keep:
            continue
        arrays.append(X if len(keep) == len(nm) else ColSel(X, keep))
        names += [nm[i] for i in keep]
    return arrays, PROD_NAMES + names


_a, _n = config([])
assert len(_n) == 136 and set(_n) == set(SHIP_NAMES) - set(B3_ALL)
del _a, _n

FOLD_SEED, SEEDS = 11, (0, 1, 2)
FOLD_IDX = build_folds(FOLD_SEED)
RES_PATH = f"{CACHE}/b5_results.json"
RES = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}

# base136 is bit-for-bit the bucket-3 run's noB3 (same columns, same order, folds, seeds): reuse it
_b3 = f"{CACHE}/b3_results.json"
if "base136" not in RES and os.path.exists(_b3):
    r = json.load(open(_b3)).get("noB3")
    if r and r["fold_seed"] == FOLD_SEED and r["seeds"] == list(SEEDS) and r["n_feats"] == 136:
        RES["base136"] = dict(tests=list(B5_TESTS), aucs=r["aucs"], n_feats=136,
                              fold_seed=FOLD_SEED, seeds=list(SEEDS), note="reused from b3 noB3")
        print("base136: reusing the bucket-3 noB3 result")
print(f"PROD {len(PROD)} | extra families {len(EXTRA_FAMS)} | bucket 5 {len(B5_ALL)} | cached runs: {list(RES)}")


def _label(tests):
    if not tests:
        return "noB5"
    if set(tests) == set(B5_TESTS):
        return "base136"
    return "keep5[" + "+".join(sorted(tests)) + "]"


def run(tests):
    """CV of noB5 + the listed models' bucket-5 columns. Resumes from RES_PATH."""
    tests = list(tests)
    label = _label(tests)
    r = RES.get(label)
    if r is not None:
        assert (r["fold_seed"] == FOLD_SEED and r["seeds"] == list(SEEDS)
                and sorted(r["tests"]) == sorted(tests)), \
            f"{label}: cached with different settings -- delete it from {RES_PATH}"
        CMP[label] = np.array(r["aucs"])
        print(f"[{label}] cached | mean {CMP[label].mean():.5f}")
        return label
    drop = [c for t in B5_TESTS if t not in tests for c in B5_TESTS[t]]
    arrays, names = config(drop)
    t0 = time.time()
    aucs = _cv(label, arrays, names, seeds=SEEDS)
    RES[label] = dict(tests=tests, aucs=[float(a) for a in aucs], n_feats=len(names),
                      fold_seed=FOLD_SEED, seeds=list(SEEDS))
    with open(RES_PATH + ".tmp", "w") as f:
        json.dump(RES, f, indent=1)
    os.replace(RES_PATH + ".tmp", RES_PATH)
    print(f"   {label}: {len(names)} features | {(time.time() - t0) / 60:.1f} min")
    gc.collect()
    return label


def paired(a, b):
    d = CMP[a] - CMP[b]
    return float(d.mean()), float(d.std(ddof=1) / np.sqrt(d.size)), int((d > 0).sum()), d.size


def passes_add(a, b):
    m, se, k, n = paired(a, b)
    return m > 2 * se and k >= n - 1


def report(ref, labels):
    rows = []
    for lab in labels:
        m, se, k, n = paired(lab, ref) if lab != ref else (0.0, 0.0, 0, CMP[lab].size)
        rows.append(dict(config=lab, feats=RES[lab]["n_feats"], mean=CMP[lab].mean(), delta=m, se=se,
                         sigma=(m / se if se > 0 else 0.0), better=f"{k}/{n}"))
    df = pd.DataFrame(rows).sort_values("mean", ascending=False).round(5)
    print(f"\n--- vs {ref} ---\n" + df.to_string(index=False))
    return df


# %% [2] reference runs: base136 (reused if cached) and noB5
BASE = run(list(B5_TESTS))
NONE = run([])
assert RES[BASE]["n_feats"] == 136 and RES[NONE]["n_feats"] == 116
m, se, k, n = paired(NONE, BASE)
print(f"\nnoB5 vs base136: {m:+.5f} | se {se:.5f} | {m / se:+.1f} sigma | better {k}/{n}")
print("-> without bucket 5 the model is",
      "NON-INFERIOR (the whole bucket may be redundant)" if m > -2 * se else "measurably WORSE (something in it is needed)")


# %% [3] keep-one-in: noB5 + each model's 5 columns on its own
ONE = {t: run([t]) for t in B5_TESTS}
report(NONE, [NONE] + list(ONE.values()))

best = max(ONE, key=lambda t: CMP[ONE[t]].mean())
tied = []
for t in ONE:
    if t == best:
        tied.append(t); continue
    m, se, _, _ = paired(ONE[t], ONE[best])
    if m > -2 * se:
        tied.append(t)
WINNER = next(t for t in PREFER if t in tied)
print(f"\nhighest mean: {best} | statistically tied with it: {tied} | winner: {WINNER}")
m, se, k, n = paired(ONE[WINNER], NONE)
print(f"{WINNER} over noB5: {m:+.5f} | se {se:.5f} | {m / se:+.1f} sigma | better {k}/{n}")
SEL = [WINNER] if passes_add(ONE[WINNER], NONE) else []
print("-> first pick:", SEL or "NONE (no single model adds measurably over noB5)")


# %% [4] forward selection on top of the winner
while SEL and len(SEL) < len(B5_TESTS):
    cur = run(SEL)
    cand = {t: run(SEL + [t]) for t in B5_TESTS if t not in SEL}
    report(cur, [cur] + list(cand.values()))
    t_best = max(cand, key=lambda t: CMP[cand[t]].mean())
    m, se, k, n = paired(cand[t_best], cur)
    if passes_add(cand[t_best], cur):
        SEL.append(t_best)
        print(f"-> ADD {t_best}: {m:+.5f} ({m / se:.1f} sigma, {k}/{n})")
    else:
        print(f"-> STOP: best addition {t_best} gives {m:+.5f} ({(m / se if se else 0):.1f} sigma, {k}/{n})")
        break
print("\nselected bucket-5 models:", SEL or "none")


# %% [5] final verdict: selected set vs base136 (pruning rule: delta > -2*se)
def verdict(sel):
    lab = run(sel)
    m, se, k, n = paired(lab, BASE)
    ok = m > -2 * se
    print(f"{lab} vs base136: {m:+.5f} | se {se:.5f} | {m / se:+.1f} sigma | better {k}/{n} "
          f"-> {'NON-INFERIOR' if ok else 'LOSES'}")
    return ok


ok = verdict(SEL)
if not ok and "WINNER" in globals() and WINNER not in SEL:
    print("fallback: keeping the winner alone")
    SEL = [WINNER]
    ok = verdict(SEL)

dropped = [c for t in B5_TESTS if t not in SEL for c in B5_TESTS[t]]
remaining = set(SHIP_NAMES) - set(B3_ALL) - set(dropped)
gone = [fam for fam in EXTRA_FAMS if not set(FAM_NAMES[fam]) & remaining]
print("\nVERDICT:", "ADOPT" if ok else "KEEP ALL OF BUCKET 5")
if ok:
    print(f"keep  : {SEL or 'none'}")
    print(f"drop  : {len(dropped)} columns -> {dropped}")
    print(f"whole families removable (save extraction time): {gone or 'none'}")
    print(f"new base: {136 - len(dropped)} features")

  garch     9 cols
  argarch  11 cols
  tgarch   11 cols
  gjr       5 cols
  recres   11 cols
  wavvar   20 cols
  hmm      11 cols
fold 0: train 1,020,112 | stop 848,058 | score 849,407
fold 1: train 1,026,629 | stop 813,776 | score 845,268
fold 2: train 1,022,269 | stop 833,755 | score 846,253
base136: reusing the bucket-3 noB3 result
PROD 58 | extra families 7 | bucket 5 20 | cached runs: ['base136']
[base136] cached | mean 0.62950
  [noB5] fold 0 swap=False: 0.61974
  [noB5] fold 0 swap=True: 0.61690
  [noB5] fold 1 swap=False: 0.62523
  [noB5] fold 1 swap=True: 0.65502
  [noB5] fold 2 swap=False: 0.63198
  [noB5] fold 2 swap=True: 0.61861
[noB5] mean 0.62791
   noB5: 116 features | 13.5 min

noB5 vs base136: -0.00159 | se 0.00075 | -2.1 sigma | better 2/6
-> without bucket 5 the model is measurably WORSE (something in it is needed)
  [keep5[garch]] fold 0 swap=False: 0.62138
  [keep5[garch]] fold 0 swap=True: 0.61777
  [keep5[garch]] fold 1 swap=False: 0.62803
  [keep5[garch]] fo

In [23]:
PAIR = run(["garch", "argarch","tgarch"])                       # 126 features
report(NONE, [NONE, ONE["garch"], ONE["argarch"], PAIR, BASE])
for ref in (ONE["argarch"], ONE["garch"], BASE):
    m, se, k, n = paired(PAIR, ref)
    print(f"pair vs {ref:16s}: {m:+.5f} | se {se:.5f} | {m/se:+.1f} sigma | better {k}/{n}")

  [keep5[argarch+garch+tgarch]] fold 0 swap=False: 0.62132
  [keep5[argarch+garch+tgarch]] fold 0 swap=True: 0.61724
  [keep5[argarch+garch+tgarch]] fold 1 swap=False: 0.63008
  [keep5[argarch+garch+tgarch]] fold 1 swap=True: 0.65659
  [keep5[argarch+garch+tgarch]] fold 2 swap=False: 0.63237
  [keep5[argarch+garch+tgarch]] fold 2 swap=True: 0.62028
[keep5[argarch+garch+tgarch]] mean 0.62965
   keep5[argarch+garch+tgarch]: 131 features | 14.0 min

--- vs noB5 ---
                     config  feats    mean   delta      se   sigma better
keep5[argarch+garch+tgarch]    131 0.62965 0.00173 0.00067 2.57674    6/6
                    base136    136 0.62950 0.00159 0.00075 2.12210    4/6
             keep5[argarch]    121 0.62918 0.00126 0.00079 1.59934    5/6
               keep5[garch]    121 0.62866 0.00075 0.00057 1.31607    4/6
                       noB5    116 0.62791 0.00000 0.00000 0.00000    0/6
pair vs keep5[argarch]  : +0.00047 | se 0.00040 | +1.2 sigma | better 4/6
pair vs keep5[g

In [26]:
# %% [markdown]
# ## Bucket 11 pruning: wavvar (4 scales x 5 aggregations) on the 131-feature base
# Permanently removed: bucket 3 (15 cols) and the gjr family (5 cols). Base = `base131`.
#
# wavvar is a 4 x 5 grid, and its two axes behave differently, so they are pruned differently:
#   aggregations (w32, w128, w512, ew, exp) are SUBSTITUTES -- horizons of the same quantity
#       -> keep-one-in + forward selection, all four scales kept
#   scales (L1, L2, L4, L8) are COMPLEMENTS -- spectral shape needs at least two bands
#       -> backward elimination on the chosen aggregations (leave-one-scale-out, repeat)
#
# Acceptance rules
#   add a column group   : delta > 2*se and better on >= n-1 fold-halves (as before)
#   drop a column group  : delta > -2*se AND delta > -MARGIN, both vs the current config AND vs base131
#                          (the fixed margin stops a noisy run from passing a real loss)
#
# Prereqs (fresh kernel is safest): section 0, 1, 2 and the ONE-CALL FAMILY TEST cell (_cv / _stack).
# Results persist to cache/b11_results.json; after a crash re-run cell 1, then the cell you were on.

# %% [1] setup: base131 = PROD (cache minus bucket 3) + extra families minus gjr, order preserved
import os, json, gc, time
import numpy as np, pandas as pd

SHIPPED = ["z2", "e2", "tail", "time", "ftest", "bartlett", "levene", "lmom", "sigtuk",
           "wass", "energy", "mmd", "entropy", "bocpd",
           "garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]
EXTRA_FAMS = ["garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]   # same order as b3/b5 runs

B3_ALL = [f"{s}_e_w{W}" for s in ("ftest", "bartlett", "levene", "sigtuk", "lmom_lscale")
          for W in (32, 64, 256)]
SCALES = ("L1", "L2", "L4", "L8")
AGGS = ("w32", "w128", "w512", "ew", "exp")
W_ALL = [f"wavvar_{s}_{a}" for s in SCALES for a in AGGS]
MARGIN = 0.0005          # largest loss we accept when dropping, whatever the se

FAM_NAMES = {f.name: list(f.names()) for f in REGISTRY if f.name in SHIPPED}
assert set(FAM_NAMES) == set(SHIPPED), f"REGISTRY lacks {set(SHIPPED) - set(FAM_NAMES)} -- re-run section 0"
SHIP_NAMES = [n for fam in SHIPPED for n in FAM_NAMES[fam]]
assert len(SHIP_NAMES) == len(set(SHIP_NAMES)) == 151, len(SHIP_NAMES)
assert set(W_ALL) == set(FAM_NAMES["wavvar"]), "wavvar names differ from the 4x5 grid"
DROP_FOREVER = set(B3_ALL) | set(FAM_NAMES["gjr"])

X_all = np.load(f"{CACHE}/X_all.npy", mmap_mode="r")
BASE_NAMES = json.load(open(f"{CACHE}/names.json"))
assert len(BASE_NAMES) == X_all.shape[1] and X_all.shape[0] == y_all.size
cache_names = [n for fam in SHIPPED if fam not in EXTRA_FAMS for n in FAM_NAMES[fam]]
assert not [n for n in cache_names if n not in BASE_NAMES]

PROD = [BASE_NAMES.index(n) for n in cache_names if n not in DROP_FOREVER]   # _stack() reads this
PROD_NAMES = [BASE_NAMES[i] for i in PROD]
assert len(PROD) == 58, len(PROD)


def load_shipped(fam):
    want = FAM_NAMES[fam]
    with np.load(f"{CACHE}/new_{fam}.npz") as Z:
        nm = [str(s) for s in Z["names"]]
        miss = [n for n in want if n not in nm]
        assert not miss, f"{fam}: cached extraction lacks {miss} -- re-extract it with force=True"
        X = np.asarray(Z["X"], dtype=np.float32)
    X = np.ascontiguousarray(X[:, [nm.index(n) for n in want]])
    assert X.shape[0] == y_all.size and np.isfinite(X).all(), fam
    return X, want


FAM_X = {}
for fam in EXTRA_FAMS:
    if fam == "gjr":
        continue                                   # dropped for good: never loaded
    FAM_X[fam] = load_shipped(fam)
    print(f"  {fam:8s} {FAM_X[fam][0].shape[1]:2d} cols")
gc.collect()


class ColSel:
    """Column subset of a family array, applied per row-slice (no full 5M-row copy)."""
    def __init__(self, X, cols):
        self.X, self.cols = X, np.asarray(cols, dtype=np.int64)
        self.shape = (X.shape[0], self.cols.size)

    def __getitem__(self, rows):
        return np.asarray(self.X[rows])[:, self.cols]


def config(drop):
    drop = set(drop) | DROP_FOREVER
    arrays, names = [], []
    for fam in EXTRA_FAMS:
        if fam not in FAM_X:
            continue
        X, nm = FAM_X[fam]
        keep = [i for i, n in enumerate(nm) if n not in drop]
        if not keep:
            continue
        arrays.append(X if len(keep) == len(nm) else ColSel(X, keep))
        names += [nm[i] for i in keep]
    return arrays, PROD_NAMES + names


_a, _n = config([])
assert len(_n) == 131 and set(_n) == set(SHIP_NAMES) - DROP_FOREVER
del _a, _n

FOLD_SEED, SEEDS = 11, (0, 1, 2)
FOLD_IDX = build_folds(FOLD_SEED)
RES_PATH = f"{CACHE}/b11_results.json"
RES = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}


def _import(src_file, src_label, dst_label, n_feats):
    """Reuse a result from an earlier bucket run if it is the identical matrix with the same folds/seeds."""
    p = f"{CACHE}/{src_file}"
    if dst_label in RES or not os.path.exists(p):
        return
    r = json.load(open(p)).get(src_label)
    if r and r["fold_seed"] == FOLD_SEED and r["seeds"] == list(SEEDS) and r["n_feats"] == n_feats:
        RES[dst_label] = dict(aggs=list(AGGS), scales=list(SCALES), aucs=r["aucs"], n_feats=n_feats,
                              fold_seed=FOLD_SEED, seeds=list(SEEDS), note=f"reused {src_file}:{src_label}")
        print(f"{dst_label}: reusing {src_file}:{src_label}")


_import("b5_results.json", "base136", "base136", 136)                       # for the gjr confirmation
_import("b5_results.json", "keep5[argarch+garch+tgarch]", "base131", 131)   # = base136 minus gjr
print(f"PROD {len(PROD)} | wavvar grid {len(W_ALL)} | cached runs: {list(RES)}")


def _canon(aggs, scales):
    return [a for a in AGGS if a in aggs], [s for s in SCALES if s in scales]


def _label(aggs, scales):
    aggs, scales = _canon(aggs, scales)
    if not aggs or not scales:
        return "noW"
    if len(aggs) == len(AGGS) and len(scales) == len(SCALES):
        return "base131"
    return f"W[{'+'.join(aggs)}|{'+'.join(scales)}]"


def run(aggs, scales=SCALES):
    """CV of base131 with wavvar restricted to the grid aggs x scales. Resumes from RES_PATH."""
    aggs, scales = _canon(aggs, scales)
    label = _label(aggs, scales)
    if label == "noW":
        aggs, scales = [], []
    r = RES.get(label)
    if r is not None:
        assert r["fold_seed"] == FOLD_SEED and r["seeds"] == list(SEEDS), \
            f"{label}: cached with different folds/seeds -- delete it from {RES_PATH}"
        CMP[label] = np.array(r["aucs"])
        print(f"[{label}] cached | mean {CMP[label].mean():.5f}")
        return label
    keep = {f"wavvar_{s}_{a}" for s in scales for a in aggs}
    arrays, names = config([c for c in W_ALL if c not in keep])
    t0 = time.time()
    aucs = _cv(label, arrays, names, seeds=SEEDS)
    RES[label] = dict(aggs=aggs, scales=scales, aucs=[float(a) for a in aucs], n_feats=len(names),
                      fold_seed=FOLD_SEED, seeds=list(SEEDS))
    with open(RES_PATH + ".tmp", "w") as f:
        json.dump(RES, f, indent=1)
    os.replace(RES_PATH + ".tmp", RES_PATH)
    print(f"   {label}: {len(names)} features | {(time.time() - t0) / 60:.1f} min")
    gc.collect()
    return label


def paired(a, b):
    d = CMP[a] - CMP[b]
    return float(d.mean()), float(d.std(ddof=1) / np.sqrt(d.size)), int((d > 0).sum()), d.size


def passes_add(a, b):
    m, se, k, n = paired(a, b)
    return m > 2 * se and k >= n - 1


def passes_drop(a, b):
    m, se, _, _ = paired(a, b)
    return m > -2 * se and m > -MARGIN


def line(a, b):
    m, se, k, n = paired(a, b)
    return f"{a} vs {b}: {m:+.5f} | se {se:.5f} | {(m / se if se else 0):+.1f} sigma | better {k}/{n}"


def report(ref, labels):
    rows = []
    for lab in labels:
        m, se, k, n = paired(lab, ref) if lab != ref else (0.0, 0.0, 0, CMP[lab].size)
        rows.append(dict(config=lab, feats=RES[lab]["n_feats"], mean=CMP[lab].mean(), delta=m, se=se,
                         sigma=(m / se if se > 0 else 0.0), better=f"{k}/{n}"))
    df = pd.DataFrame(rows).sort_values("mean", ascending=False).round(5)
    print(f"\n--- vs {ref} ---\n" + df.to_string(index=False))
    return df


# How often are the window aggregates identical? _win() uses min(n, W) points, so for t <= W a
# window aggregate equals the expanding one: w512 == exp on every row with t <= 512.
_t = np.asarray(t_all)
print("rows with t <= 32 / 128 / 512:", [f"{np.mean(_t <= W):.1%}" for W in (32, 128, 512)],
      "-> share of rows where w32 / w128 / w512 are identical to exp")
del _t


# %% [2] reference runs: base131 (confirms the gjr drop) and noW (does wavvar matter at all?)
BASE = run(AGGS, SCALES)
NONE = run([])
assert RES[BASE]["n_feats"] == 131 and RES[NONE]["n_feats"] == 111
if "base136" in RES:
    CMP["base136"] = np.array(RES["base136"]["aucs"])
    print("gjr drop :", line(BASE, "base136"),
          "-> OK" if passes_drop(BASE, "base136") else "-> CHECK: dropping gjr looks like a loss")
print("wavvar   :", line(NONE, BASE))
W_NEEDED = not passes_drop(NONE, BASE)
print("-> without wavvar the model is", "measurably WORSE: prune inside the grid" if W_NEEDED
      else "NON-INFERIOR: the whole family can go (skip to cell 5 with SEL_A = [], SEL_S = [])")


# %% [3] aggregations: keep-one-in (all four scales), then forward selection
ONE = {a: run([a]) for a in AGGS}
report(NONE, [NONE] + list(ONE.values()))

best = max(ONE, key=lambda a: CMP[ONE[a]].mean())            # tie-break: best point estimate
print("\nbest single aggregation:", line(ONE[best], NONE))
SEL_A = [best] if passes_add(ONE[best], NONE) else []

while SEL_A and len(SEL_A) < len(AGGS):
    cur = run(SEL_A)
    cand = {a: run(SEL_A + [a]) for a in AGGS if a not in SEL_A}
    report(cur, [cur] + list(cand.values()))
    a_best = max(cand, key=lambda a: CMP[cand[a]].mean())
    if passes_add(cand[a_best], cur):
        SEL_A.append(a_best); print("-> ADD", a_best, "|", line(cand[a_best], cur))
    else:
        print("-> STOP |", line(cand[a_best], cur)); break

if not SEL_A:
    print("no aggregation adds measurably over noW")
    if W_NEEDED:                       # wavvar matters jointly: start from the best single one anyway
        SEL_A = [best]

# the add rule is strict, so the selection can stop short of base131: keep adding the best
# remaining aggregation (one step at a time) until the config is non-inferior to base131
while SEL_A and len(SEL_A) < len(AGGS) and not passes_drop(run(SEL_A), BASE):
    print("still loses vs base131:", line(run(SEL_A), BASE))
    cand = {a: run(SEL_A + [a]) for a in AGGS if a not in SEL_A}
    a_best = max(cand, key=lambda a: CMP[cand[a]].mean())
    SEL_A.append(a_best); print("-> ADD (to reach base131)", a_best)
SEL_A = _canon(SEL_A, SCALES)[0]
print("\naggregations kept:", SEL_A or "none")


# %% [4] scales: backward elimination on the kept aggregations
SEL_S = list(SCALES) if SEL_A else []
while len(SEL_S) > 1:
    cur = run(SEL_A, SEL_S)
    cand = {s: run(SEL_A, [x for x in SEL_S if x != s]) for s in SEL_S}
    report(cur, [cur] + list(cand.values()))
    s_best = max(cand, key=lambda s: CMP[cand[s]].mean())   # the scale whose removal hurts least
    if passes_drop(cand[s_best], cur) and passes_drop(cand[s_best], BASE):
        SEL_S.remove(s_best); print("-> DROP", s_best, "|", line(cand[s_best], BASE))
    else:
        print("-> STOP | cheapest removal", s_best)
        print("   ", line(cand[s_best], cur)); print("   ", line(cand[s_best], BASE)); break
print("\nscales kept:", SEL_S or "none")


# %% [5] final verdict vs base131
FINAL = run(SEL_A, SEL_S)
print(line(FINAL, BASE))
ok = passes_drop(FINAL, BASE)
kept = [f"wavvar_{s}_{a}" for s in SEL_S for a in SEL_A]
dropped = [c for c in W_ALL if c not in kept]
print("\nVERDICT:", "ADOPT" if ok else "KEEP ALL OF WAVVAR")
if ok:
    print(f"keep  : {len(kept)} -> {kept}")
    print(f"drop  : {len(dropped)} columns")
    print("whole family removable (saves extraction time):", "wavvar" if not kept else "no")
    print(f"new base: {131 - len(dropped)} features")

  garch     9 cols
  argarch  11 cols
  tgarch   11 cols
  recres   11 cols
  wavvar   20 cols
  hmm      11 cols
fold 0: train 1,020,112 | stop 848,058 | score 849,407
fold 1: train 1,026,629 | stop 813,776 | score 845,268
fold 2: train 1,022,269 | stop 833,755 | score 846,253
PROD 58 | wavvar grid 20 | cached runs: ['base136', 'base131', 'noW', 'W[w32|L1+L2+L4+L8]']
rows with t <= 32 / 128 / 512: ['6.5%', '24.1%', '76.3%'] -> share of rows where w32 / w128 / w512 are identical to exp
[base131] cached | mean 0.62965
[noW] cached | mean 0.62928
gjr drop : base131 vs base136: +0.00015 | se 0.00039 | +0.4 sigma | better 4/6 -> OK
wavvar   : noW vs base131: -0.00037 | se 0.00049 | -0.7 sigma | better 1/6
-> without wavvar the model is NON-INFERIOR: the whole family can go (skip to cell 5 with SEL_A = [], SEL_S = [])
[W[w32|L1+L2+L4+L8]] cached | mean 0.62929
  [W[w128|L1+L2+L4+L8]] fold 0 swap=False: 0.62056
  [W[w128|L1+L2+L4+L8]] fold 0 swap=True: 0.61842
  [W[w128|L1+L2+L4+L8]] fold 1 

KeyboardInterrupt: 

In [27]:
full = np.array(json.load(open(f"{CACHE}/b3_results.json"))["full151"]["aucs"])
d = CMP["noW"] - full
m, se = d.mean(), d.std(ddof=1) / np.sqrt(d.size)
print(f"base111 vs full151: {m:+.5f} | se {se:.5f} | {m/se:+.1f} sigma | better {(d > 0).sum()}/{d.size}")

base111 vs full151: -0.00014 | se 0.00049 | -0.3 sigma | better 2/6


In [33]:
# %% [markdown]
# ## Confirmation: full151 vs base111 on a fresh split, scored as the production blend
# base111 = full151 minus bucket 3 (15), gjr (5) and wavvar (20).
#
# Two questions at once:
#   1. does the pruning hold on a split it was NOT selected on?  (fold seed 23; 0, 7 and 11 are used)
#   2. does it hold for the blend, not just LightGBM?  CatBoost never saw the pruned bank.
#
# Each fold-half trains LightGBM (LGB_SEEDS, averaged) and CatBoost (CB_SEEDS) on the same rows,
# early-stops both on the stop half, and scores lgb / cb / blend on the score half.
# Every finished fold-half is saved to cache/confirm_results.json, so a crash resumes mid-config.
#
# Prereqs: section 0 (engine), 1 (cache: X_all, y_all, t_all, g_all), 2 (build_folds, STRIDE_ALL,
# ts_auc, LGB_PARAMS, NUM_ROUNDS). The _cv harness is NOT used here: this cell has its own loop,
# which also stacks each fold-half's matrices once instead of once per seed.

# %% [1] settings -- paste CB_PARAMS and match the blend to submission.ipynb
CB_PARAMS = dict(
    grow_policy="Depthwise",
    learning_rate=0.028354684013833883,
    depth=8,
    l2_leaf_reg=12.76103092455787,
    rsm=0.5758405578310992,
    random_strength=2.5835125442520925,
    subsample=0.7933098848053978,
    min_data_in_leaf=64,
    bootstrap_type="Bernoulli",
)
CAT_WEIGHT = 0.3        # <- paste the CatBoost params from submission.ipynb, e.g.
                          #    dict(iterations=3000, learning_rate=0.05, depth=6, l2_leaf_reg=3, ...)
BLEND_W = 0.7             # weight on LightGBM
BLEND_MODE = "prob"       # "prob": w*p_lgb + (1-w)*p_cb   |   "logit": the same on the logit scale
                          #  -> use whichever submission.ipynb actually does

FOLD_SEED = 23
LGB_SEEDS = (0, 1, 2)
CB_SEEDS = (0,)
PATIENCE = 100            # early stopping for both models, on the stop half
MARGIN = 0.0005

assert CB_PARAMS is not None, "paste the CatBoost params from submission.ipynb into CB_PARAMS"
assert BLEND_MODE in ("prob", "logit")


# %% [2] setup: both banks from the same arrays
import os, json, gc, time
import numpy as np, pandas as pd, lightgbm as lgb
from catboost import CatBoostClassifier, Pool

SHIPPED = ["z2", "e2", "tail", "time", "ftest", "bartlett", "levene", "lmom", "sigtuk",
           "wass", "energy", "mmd", "entropy", "bocpd",
           "garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]
EXTRA_FAMS = ["garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]
B3_ALL = [f"{s}_e_w{W}" for s in ("ftest", "bartlett", "levene", "sigtuk", "lmom_lscale")
          for W in (32, 64, 256)]
DROPPED_FAMS = {"gjr", "wavvar"}

FAM_NAMES = {f.name: list(f.names()) for f in REGISTRY if f.name in SHIPPED}
assert set(FAM_NAMES) == set(SHIPPED), f"REGISTRY lacks {set(SHIPPED) - set(FAM_NAMES)} -- re-run section 0"
SHIP_NAMES = [n for fam in SHIPPED for n in FAM_NAMES[fam]]
assert len(SHIP_NAMES) == len(set(SHIP_NAMES)) == 151, len(SHIP_NAMES)

X_all = np.load(f"{CACHE}/X_all.npy", mmap_mode="r")
BASE_NAMES = json.load(open(f"{CACHE}/names.json"))
assert len(BASE_NAMES) == X_all.shape[1] and X_all.shape[0] == y_all.size
cache_names = [n for fam in SHIPPED if fam not in EXTRA_FAMS for n in FAM_NAMES[fam]]
assert not [n for n in cache_names if n not in BASE_NAMES]


def load_shipped(fam):
    want = FAM_NAMES[fam]
    with np.load(f"{CACHE}/new_{fam}.npz") as Z:
        nm = [str(s) for s in Z["names"]]
        miss = [n for n in want if n not in nm]
        assert not miss, f"{fam}: cached extraction lacks {miss} -- re-extract it with force=True"
        X = np.asarray(Z["X"], dtype=np.float32)
    X = np.ascontiguousarray(X[:, [nm.index(n) for n in want]])
    assert X.shape[0] == y_all.size and np.isfinite(X).all(), fam
    return X, want


FAM_X = {fam: load_shipped(fam) for fam in EXTRA_FAMS}
gc.collect()


def bank(prod_names, fams):
    prod = [BASE_NAMES.index(n) for n in prod_names]
    arrays = [FAM_X[f][0] for f in fams]
    names = list(prod_names) + [n for f in fams for n in FAM_X[f][1]]
    return dict(prod=prod, arrays=arrays, names=names)


BANKS = {
    "full151": bank(cache_names, EXTRA_FAMS),
    "base111": bank([n for n in cache_names if n not in B3_ALL],
                    [f for f in EXTRA_FAMS if f not in DROPPED_FAMS]),
}
assert len(BANKS["full151"]["names"]) == 151 and set(BANKS["full151"]["names"]) == set(SHIP_NAMES)
assert len(BANKS["base111"]["names"]) == 111
print({k: len(v["names"]) for k, v in BANKS.items()})

FOLDS = build_folds(FOLD_SEED)
RES_PATH = f"{CACHE}/confirm_results.json"
RES = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}
SETTINGS = json.loads(json.dumps(dict(      # JSON round-trip so it compares equal to the saved copy
    fold_seed=FOLD_SEED, lgb_seeds=list(LGB_SEEDS), cb_seeds=list(CB_SEEDS),
    blend_w=BLEND_W, blend_mode=BLEND_MODE, patience=PATIENCE, cb_params=CB_PARAMS)))


def _stack(rows, B):
    parts = [np.asarray(X_all[rows])[:, B["prod"]]] + [a[rows] for a in B["arrays"]]
    return np.ascontiguousarray(np.hstack(parts), dtype=np.float32)


def _logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))


def blend(p_lgb, p_cb):
    if BLEND_MODE == "prob":
        return BLEND_W * p_lgb + (1 - BLEND_W) * p_cb
    return BLEND_W * _logit(p_lgb) + (1 - BLEND_W) * _logit(p_cb)   # monotone: TS-AUC needs no sigmoid


def cv_blend(label):
    """6 fold-halves of LGB + CB on bank `label`; resumes from the last saved fold-half."""
    B = BANKS[label]
    r = RES.get(label)
    if r is None or r["settings"] != SETTINGS:
        if r is not None:
            print(f"[{label}] settings changed -- starting this config over")
        r = RES[label] = dict(settings=SETTINGS, n_feats=len(B["names"]), lgb=[], cb=[], blend=[],
                              lgb_trees=[], cb_trees=[])
    k = 0
    for fi, F in enumerate(FOLDS):
        for swap in (False, True):
            if k < len(r["blend"]):
                k += 1
                continue
            t0 = time.time()
            si, ei = (F["sc"], F["st"]) if swap else (F["st"], F["sc"])
            Xtr, Xs, Xe = _stack(F["tr"], B), _stack(si, B), _stack(ei, B)
            ytr, w = y_all[F["tr"]], STRIDE_ALL[F["tr"]].astype(np.float32)

            pl, it_l = [], []
            for s in LGB_SEEDS:
                p = dict(LGB_PARAMS)
                p.update(objective="binary", metric="binary_logloss",
                         seed=s, bagging_seed=s + 100, feature_fraction_seed=s + 200)
                dtr = lgb.Dataset(Xtr, label=ytr, weight=w, feature_name=B["names"])
                dva = lgb.Dataset(Xs, label=y_all[si], reference=dtr)
                b = lgb.train(p, dtr, num_boost_round=NUM_ROUNDS, valid_sets=[dva],
                              callbacks=[lgb.early_stopping(PATIENCE, verbose=False)])
                pl.append(b.predict(Xe, num_iteration=b.best_iteration)); it_l.append(b.best_iteration)
                del dtr, dva, b; gc.collect()

            pc, it_c = [], []
            ptr, pva = Pool(Xtr, label=ytr, weight=w), Pool(Xs, label=y_all[si])
            for s in CB_SEEDS:
                p = dict(CB_PARAMS)
                p.update(loss_function="Logloss", eval_metric="Logloss", random_seed=s,
                         od_type="Iter", od_wait=PATIENCE, use_best_model=True, verbose=0)
                for key in ("early_stopping_rounds", "logging_level", "verbose_eval", "silent",
                            "random_state", "objective", "od_pval"):
                    p.pop(key, None)
                m = CatBoostClassifier(**p)
                m.fit(ptr, eval_set=pva)
                pc.append(m.predict_proba(Xe)[:, 1]); it_c.append(m.get_best_iteration())
                del m; gc.collect()
            del ptr, pva

            p_l, p_c = np.mean(pl, axis=0), np.mean(pc, axis=0)
            ye, te = y_all[ei], t_all[ei]
            r["lgb"].append(float(ts_auc(ye, te, p_l)))
            r["cb"].append(float(ts_auc(ye, te, p_c)))
            r["blend"].append(float(ts_auc(ye, te, blend(p_l, p_c))))
            r["lgb_trees"].append(int(np.mean(it_l))); r["cb_trees"].append(int(np.mean(it_c)))
            with open(RES_PATH + ".tmp", "w") as f:
                json.dump(RES, f, indent=1)
            os.replace(RES_PATH + ".tmp", RES_PATH)
            print(f"  [{label}] fold {fi} swap={swap}: lgb {r['lgb'][-1]:.5f} | cb {r['cb'][-1]:.5f} | "
                  f"blend {r['blend'][-1]:.5f} | trees {r['lgb_trees'][-1]}/{r['cb_trees'][-1]} | "
                  f"{(time.time() - t0) / 60:.1f} min")
            del Xtr, Xs, Xe, pl, pc; gc.collect()
            k += 1
    print(f"[{label}] {r['n_feats']} feats | lgb {np.mean(r['lgb']):.5f} | cb {np.mean(r['cb']):.5f} | "
          f"blend {np.mean(r['blend']):.5f}")
    return r


# %% [3] run both configs (resumable: re-run this cell after a crash)
for label in ("full151", "base111"):
    cv_blend(label)


# %% [4] verdict
def paired(d):
    d = np.asarray(d)
    return d.mean(), d.std(ddof=1) / np.sqrt(d.size), int((d > 0).sum()), d.size


A, Bb = RES["base111"], RES["full151"]
assert len(A["blend"]) == len(Bb["blend"]) == 2 * len(FOLDS), "a config is unfinished -- re-run cell 3"
rows = []
for model in ("lgb", "cb", "blend"):
    m, se, k, n = paired(np.array(A[model]) - np.array(Bb[model]))
    rows.append(dict(model=model, full151=np.mean(Bb[model]), base111=np.mean(A[model]), delta=m, se=se,
                     sigma=m / se if se else 0.0, better=f"{k}/{n}",
                     verdict="NON-INFERIOR" if (m > -2 * se and m > -MARGIN) else "LOSES"))
print(f"\nbase111 vs full151 | fold seed {FOLD_SEED} | blend {BLEND_W}/{1 - BLEND_W:.1f} ({BLEND_MODE})")
print(pd.DataFrame(rows).round(5).to_string(index=False))

# pooled LightGBM evidence: this split + the seed-11 runs the pruning was chosen on (12 fold-halves)
try:
    full11 = np.array(json.load(open(f"{CACHE}/b3_results.json"))["full151"]["aucs"])
    base11 = np.array(json.load(open(f"{CACHE}/b11_results.json"))["noW"]["aucs"])
    d = np.r_[base11 - full11, np.array(A["lgb"]) - np.array(Bb["lgb"])]
    m, se, k, n = paired(d)
    print(f"\nLightGBM pooled over seeds 11 + {FOLD_SEED}: {m:+.5f} | se {se:.5f} | {m / se:+.1f} sigma | better {k}/{n}")
except (FileNotFoundError, KeyError) as e:
    print("pooled LightGBM line skipped:", e)

bl = rows[-1]
print("\nFINAL:", "SHIP base111" if bl["verdict"] == "NON-INFERIOR" else
      "the blend loses on the fresh split -- do not ship base111 yet; look at which model moved (lgb or cb)")

{'full151': 151, 'base111': 111}
fold 0: train 1,026,362 | stop 808,924 | score 855,141
fold 1: train 1,017,639 | stop 834,000 | score 866,518
fold 2: train 1,025,009 | stop 842,221 | score 829,713
  [full151] fold 0 swap=False: lgb 0.62414 | cb 0.62591 | blend 0.62556 | trees 285/399 | 5.0 min
  [full151] fold 0 swap=True: lgb 0.62150 | cb 0.61959 | blend 0.62167 | trees 253/313 | 4.7 min
  [full151] fold 1 swap=False: lgb 0.61869 | cb 0.61813 | blend 0.61942 | trees 196/240 | 4.1 min
  [full151] fold 1 swap=True: lgb 0.62735 | cb 0.62500 | blend 0.62741 | trees 171/236 | 3.6 min
  [full151] fold 2 swap=False: lgb 0.62551 | cb 0.62491 | blend 0.62668 | trees 367/436 | 6.0 min
  [full151] fold 2 swap=True: lgb 0.62919 | cb 0.63113 | blend 0.63063 | trees 351/386 | 5.9 min
[full151] 151 feats | lgb 0.62440 | cb 0.62411 | blend 0.62523
  [base111] fold 0 swap=False: lgb 0.62489 | cb 0.62800 | blend 0.62676 | trees 326/379 | 4.6 min
  [base111] fold 0 swap=True: lgb 0.62361 | cb 0.62266 |

In [ ]:
# %% [markdown]
# ## lit5: eight families for properties the bank has never measured
# | family  | property                                   | streams (x 5 aggregates)        | feats |
# |---------|--------------------------------------------|---------------------------------|-------|
# | irrev   | time irreversibility (rise/fall asymmetry) | d1, d4, rr1, ord                | 20    |
# | jump    | jumps vs continuous variation              | jshare, jflag3, jflag4          | 15    |
# | nlmean  | nonlinear mean dynamics (LM streams)       | quad, bil, cub, thr             | 20    |
# | lvlvar  | variance tied to the level                 | bp, wh                          | 10    |
# | xclust  | clustering of extremes (rate-adjusted)     | c1, c5, cu, cl                  | 20    |
# | aggkurt | tail weight vs aggregation horizon         | q1, q4, q16, qr                 | 20    |
# | period  | narrowband periodicity (sliding DFT)       | g, pk                           | 10    |
# | recproc | recording process: ties / repeats / grid   | tie, rep, fine                  | 15    |
#
# Every stream is computed on the history with the SAME recursion as online, and emitted as
# (value - history mean) / history sd, so each is ~ mean 0 / sd 1 under "no break" whatever the
# series looks like. The shared `_CondBase` then aggregates each stream as w32 / w128 / w512 / ew / exp
# (window mean x sqrt(size)), exactly like garch / recres / hmm. All O(1) per step, no RNG.
#
# `r` below is the history-AR(3) one-step innovation of the raw standardized series, in units of its
# own history sd -- NOT the engine's `e` channel, which is rank-based (PIT) and caps every value at
# the history's range, so it cannot see a jump bigger than anything in the history.
#
# Needs from the engine: FeatureFamily, _CondBase, _history, _safe (the research notebook's
# vol-helper import cell provides them). Test with the base111 `test_family` (one call per family,
# cells below). The ties family (`recproc`) is only worth its CV run if the pre-check says the
# data has ties at all.

# %% [1] lit5 families
import math
import numpy as np

_BV = math.pi / 2.0        # (pi/2) E|Z||Z'| = 1 for iid N(0,1): bipower is unbiased for variance
_ZCAP = 20.0               # standardized stream values are clipped here (rare-event streams)


def _zr_hist(hist):
    xs = _history(hist)
    return (xs - float(hist.mu)) / (float(hist.sd) if hist.sd > 0 else 1.0)


class _Lit5Base(_CondBase):
    """History-calibrated streams + an AR(3) innovation in raw (not rank) units."""
    channels = ("z",)
    cost = "O1"
    version = 1
    SD_FLOOR = 0.05
    WARM = 50               # history steps skipped before calibration (recursive states warming up)

    def names(self):
        return self._agg_names()

    # ---- raw-unit AR innovation, same recursion as HistoricalRef's residual ----
    def _setup_innov(self, hist):
        zh = _zr_hist(hist)
        self.ar = [float(a) for a in hist.ar_l]
        p = len(self.ar)
        r = zh[p:].copy()
        for k in range(1, p + 1):
            r -= self.ar[k - 1] * zh[p - k:zh.size - k]
        s = float(r.std(ddof=1)) if r.size > 2 else 1.0
        self.rs = s if s > 1e-12 else 1.0
        self.lagz = [float(v) for v in zh[-p:][::-1]]           # most recent first
        return zh, r / self.rs                                   # r[j] belongs to time p + j

    def _innov(self, zr):
        e = zr
        for a, l in zip(self.ar, self.lagz):
            e -= a * l
        self.lagz = [zr] + self.lagz[:-1]
        return e / self.rs

    # ---- calibration ----
    def _calibrate(self, S):
        self.mu_s, self.sd_s = {}, {}
        for k in self.STREAMS:
            v = np.asarray(S.get(k, ()), dtype=np.float64)
            v = v[np.isfinite(v)]
            m, s = (float(v.mean()), float(v.std())) if v.size >= 20 else (0.0, 1.0)
            self.mu_s[k] = m
            self.sd_s[k] = max(s, self.SD_FLOOR)

    def _emit(self, raw):
        vals = {}
        for k in self.STREAMS:
            z = (raw[k] - self.mu_s[k]) / self.sd_s[k]
            vals[k] = _safe(min(max(z, -_ZCAP), _ZCAP))
        self._push(vals)
        return self._aggregate()


# ------------------------------------------------------------------ 1. time irreversibility
class IrreversibilityFamily(_Lit5Base):
    """Is the process the same played backwards? (Ramsey & Rothman 1996; Lacasa et al. 2012)
    d1, d4 : cubed k-step change of the (clipped) standardized level, in history-sd units --
             skewness of increments, i.e. sharp rises vs slow falls. Not lskew (skew of LEVELS).
    rr1    : Ramsey-Rothman bicovariance  z_t^2 z_{t-1} - z_t z_{t-1}^2  (zero for reversible series)
    ord    : 1[up-up] - 1[down-down] over consecutive triples (ordpat's `mono` pools the two)."""
    name = "irrev"
    STREAMS = ("d1", "d4", "rr1", "ord")
    C = 4.0

    def init(self, hist):
        z = np.clip(_zr_hist(hist), -self.C, self.C)
        d1 = z[1:] - z[:-1]
        d4 = z[4:] - z[:-4]
        self.s1 = max(float(d1.std()), 1e-6)
        self.s4 = max(float(d4.std()), 1e-6)
        a, b, c = z[:-2], z[1:-1], z[2:]
        self._calibrate({
            "d1": (d1 / self.s1) ** 3,
            "d4": (d4 / self.s4) ** 3,
            "rr1": z[1:] ** 2 * z[:-1] - z[1:] * z[:-1] ** 2,
            "ord": ((a < b) & (b < c)).astype(float) - ((a > b) & (b > c)).astype(float),
        })
        self.zb = [float(v) for v in z[-4:]]                 # oldest first: z_{t-4} .. z_{t-1}
        self._setup_buffers()

    def update(self, ctx):
        z = min(max(float(ctx.zr), -self.C), self.C)
        zb = self.zb
        z1, z2, z4 = zb[-1], zb[-2], zb[-4]
        up = 1.0 if (z2 < z1 < z) else 0.0
        dn = 1.0 if (z2 > z1 > z) else 0.0
        raw = {"d1": ((z - z1) / self.s1) ** 3,
               "d4": ((z - z4) / self.s4) ** 3,
               "rr1": z * z * z1 - z * z1 * z1,
               "ord": up - dn}
        self.zb = zb[1:] + [z]
        return self._emit(raw)


# ------------------------------------------------------------------ 2. jumps vs continuous variation
class JumpVariationFamily(_Lit5Base):
    """How variance arrives: many small moves or a few jumps. (Barndorff-Nielsen & Shephard 2004,
    2006; Lee & Mykland 2008). BV_loc = EWMA of the bipower term (pi/2)|r_t||r_{t-1}|, which is
    robust to isolated jumps, lagged one step, so every stream is SCALE-FREE:
    jshare : (r_t^2 - (pi/2)|r_t||r_{t-1}|) / BV_loc -- realized minus bipower = the jump part
    jflag3 : 1[r_t^2 > 9 BV_loc]    jflag4 : 1[r_t^2 > 16 BV_loc]   (jump indicators, 3 and 4 sigma)
    A pure volatility rise moves r^2 and BV_loc together and leaves all three near zero."""
    name = "jump"
    STREAMS = ("jshare", "jflag3", "jflag4")
    A = 0.05
    RCAP = 50.0

    def init(self, hist):
        _, r = self._setup_innov(hist)
        r = np.clip(r, -self.RCAP, self.RCAP)
        loc = float(np.mean(_BV * np.abs(r[1:31]) * np.abs(r[:30]))) if r.size > 31 else 1.0
        loc = max(loc, 1e-6)
        S = {k: [] for k in self.STREAMS}
        prev = float(r[0]) if r.size else 0.0
        for j in range(1, r.size):
            x = float(r[j])
            b = _BV * abs(x) * abs(prev)
            L = max(loc, 1e-6)
            if j >= self.WARM:
                S["jshare"].append((x * x - b) / L)
                S["jflag3"].append(1.0 if x * x > 9.0 * L else 0.0)
                S["jflag4"].append(1.0 if x * x > 16.0 * L else 0.0)
            loc += self.A * (b - loc)
            prev = x
        self._calibrate(S)
        self.loc, self.prev = max(loc, 1e-6), prev
        self._setup_buffers()

    def update(self, ctx):
        x = min(max(self._innov(float(ctx.zr)), -self.RCAP), self.RCAP)
        b = _BV * abs(x) * abs(self.prev)
        L = max(self.loc, 1e-6)
        raw = {"jshare": (x * x - b) / L,
               "jflag3": 1.0 if x * x > 9.0 * L else 0.0,
               "jflag4": 1.0 if x * x > 16.0 * L else 0.0}
        self.loc += self.A * (b - self.loc)
        self.prev = x
        return self._emit(raw)


# ------------------------------------------------------------------ 3. nonlinear mean dynamics
class NonlinearMeanFamily(_Lit5Base):
    """Does the next value start depending NONLINEARLY on the past? Lagrange-multiplier streams:
    under the history's linear AR(3) the innovation r_t is uncorrelated with any function of the
    lags, so each product averages to its history value; a new nonlinearity makes it drift.
    quad : r_t (z_{t-1}^2 - m2)          Keenan (1985), quadratic
    bil  : r_t z_{t-1} z_{t-2}            bilinear / cross-product
    cub  : r_t z_{t-1}^3                  Luukkonen, Saikkonen & Terasvirta (1988), smooth transition
    thr  : r_t sign(z_{t-1} - median)     Tsay (1989), threshold AR
    Signed, so the aggregates are CUSUM-like. recres only tracks LINEAR coefficient drift."""
    name = "nlmean"
    STREAMS = ("quad", "bil", "cub", "thr")
    ZC, RC = 5.0, 6.0

    def init(self, hist):
        zh, r = self._setup_innov(hist)
        p = len(self.ar)
        zc = np.clip(zh, -self.ZC, self.ZC)
        rc = np.clip(r, -self.RC, self.RC)
        z1, z2 = zc[p - 1:zc.size - 1], zc[p - 2:zc.size - 2]
        self.m2 = float(np.mean(zc * zc))
        self.med = float(np.median(zc))
        self._calibrate({"quad": rc * (z1 * z1 - self.m2), "bil": rc * z1 * z2,
                         "cub": rc * z1 ** 3, "thr": rc * np.sign(z1 - self.med)})
        self.z1, self.z2 = float(zc[-1]), float(zc[-2])
        self._setup_buffers()

    def update(self, ctx):
        zr = float(ctx.zr)
        x = min(max(self._innov(zr), -self.RC), self.RC)
        z1, z2 = self.z1, self.z2
        s = 1.0 if z1 > self.med else (-1.0 if z1 < self.med else 0.0)
        raw = {"quad": x * (z1 * z1 - self.m2), "bil": x * z1 * z2, "cub": x * z1 ** 3, "thr": x * s}
        self.z2, self.z1 = z1, min(max(zr, -self.ZC), self.ZC)
        return self._emit(raw)


# ------------------------------------------------------------------ 4. variance tied to the level
class LevelVarianceFamily(_Lit5Base):
    """Is volatility a function of WHERE the series is (not of past shocks, as in GARCH)?
    Breusch & Pagan (1979) / White (1980) score streams on the squared innovation, against a
    short EWMA of the level (lagged): L = EWMA_0.25 of the clipped standardized level.
    bp : (min(r_t^2, 36) - 1) (L_{t-1} - mean L)      wh : (min(r_t^2, 36) - 1) (L_{t-1}^2 - mean L^2)
    (Chan, Karolyi, Longstaff & Sanders 1992 for level-dependent volatility.)"""
    name = "lvlvar"
    STREAMS = ("bp", "wh")
    A, ZC = 0.25, 5.0

    def init(self, hist):
        zh, r = self._setup_innov(hist)
        p = len(self.ar)
        zc = np.clip(zh, -self.ZC, self.ZC)
        L = np.empty(zc.size)
        l = float(zc[0])
        for i in range(zc.size):
            l += self.A * (zc[i] - l)
            L[i] = l
        Ll = L[p - 1:L.size - 1]                     # level up to t-1, aligned with r (time p + j)
        self.mL, self.mL2 = float(Ll.mean()), float(np.mean(Ll * Ll))
        q = np.minimum(r * r, 36.0) - 1.0
        k = min(self.WARM, max(q.size - 20, 0))
        self._calibrate({"bp": (q * (Ll - self.mL))[k:], "wh": (q * (Ll * Ll - self.mL2))[k:]})
        self.L = l
        self._setup_buffers()

    def update(self, ctx):
        zr = float(ctx.zr)
        x = self._innov(zr)
        q = min(x * x, 36.0) - 1.0
        L = self.L
        raw = {"bp": q * (L - self.mL), "wh": q * (L * L - self.mL2)}
        self.L += self.A * (min(max(zr, -self.ZC), self.ZC) - L)
        return self._emit(raw)


# ------------------------------------------------------------------ 5. clustering of extremes
class ExtremeClusterFamily(_Lit5Base):
    """Do extremes arrive in bursts, HOLDING THEIR RATE FIXED? (Christoffersen 1998 independence
    test; Ferro & Segers 2003). Hits: I = 1[|r| > q95 of history], Iu = 1[r > q97.5], Il = 1[r < q2.5].
    p_loc = EWMA_0.02 of the hit indicator (lagged), so a plain rise in the hit RATE cancels:
    c1 : I_{t-1} (I_t - p_loc)            hit right after a hit, beyond the current rate
    c5 : any(I_{t-5..t-1}) (I_t - p_loc)  hit within 5 steps of a hit
    cu / cl : the c1 stream for the upper / lower tail separately (asymmetric clustering).
    tail / evt / caviar measure how OFTEN and how BIG; none measures whether extremes cluster."""
    name = "xclust"
    STREAMS = ("c1", "c5", "cu", "cl")
    A = 0.02

    def init(self, hist):
        _, r = self._setup_innov(hist)
        if r.size < 40:
            self.q, self.qu, self.ql = 2.0, 2.0, -2.0
        else:
            self.q = float(np.quantile(np.abs(r), 0.95))
            self.qu, self.ql = float(np.quantile(r, 0.975)), float(np.quantile(r, 0.025))
        self.p, self.pu, self.pl = 0.05, 0.025, 0.025
        self.last = [0.0] * 5                              # I_{t-5} .. I_{t-1}
        self.lu = self.ll = 0.0
        S = {k: [] for k in self.STREAMS}
        for j in range(r.size):
            raw = self._step(float(r[j]))
            if j >= self.WARM:
                for k in self.STREAMS:
                    S[k].append(raw[k])
        self._calibrate(S)
        self._setup_buffers()

    def _step(self, x):
        I = 1.0 if abs(x) > self.q else 0.0
        Iu = 1.0 if x > self.qu else 0.0
        Il = 1.0 if x < self.ql else 0.0
        raw = {"c1": self.last[-1] * (I - self.p),
               "c5": (1.0 if max(self.last) > 0.0 else 0.0) * (I - self.p),
               "cu": self.lu * (Iu - self.pu),
               "cl": self.ll * (Il - self.pl)}
        self.p += self.A * (I - self.p)
        self.pu += self.A * (Iu - self.pu)
        self.pl += self.A * (Il - self.pl)
        self.last = self.last[1:] + [I]
        self.lu, self.ll = Iu, Il
        return raw

    def update(self, ctx):
        return self._emit(self._step(self._innov(float(ctx.zr))))


# ------------------------------------------------------------------ 6. tail weight vs aggregation horizon
class AggKurtosisFamily(_Lit5Base):
    """Do heavy tails survive aggregation? For tau = 1, 4, 16 the tau-step change of the clipped
    level, y = (z_t - z_{t-tau}) / sd_hist(tau), gives a kurtosis-type stream q = y^4 / v^2 with
    v = EWMA_0.02 of y^2 (lagged) -- scale-free, so a volatility change alone does not move it.
    q1 / q4 / q16, and qr = standardized q16 - standardized q1: the CONTRAST (tails persisting across
    horizons = intermittency / multifractality), given as its own column because trees split one
    column at a time. (Kantelhardt et al. 2002; Cont 2001, aggregational Gaussianity.)"""
    name = "aggkurt"
    STREAMS = ("q1", "q4", "q16", "qr")
    TAUS = (1, 4, 16)
    A, ZC, QCAP = 0.02, 8.0, 400.0

    def init(self, hist):
        z = np.clip(_zr_hist(hist), -self.ZC, self.ZC)
        T = max(self.TAUS)
        self.sd = {}
        for tau in self.TAUS:
            d = z[tau:] - z[:-tau] if z.size > tau + 2 else np.ones(3)
            self.sd[tau] = max(float(d.std()), 1e-6)
        self.v = {tau: 1.0 for tau in self.TAUS}
        self.zb = [float(v) for v in z[-T:]]
        Q = {tau: [] for tau in self.TAUS}
        buf = [float(v) for v in z[:T]]
        for i in range(T, z.size):
            q = self._q(buf, float(z[i]))
            if i >= T + self.WARM:
                for tau in self.TAUS:
                    Q[tau].append(q[tau])
            buf = buf[1:] + [float(z[i])]
        S = {"q%d" % tau: Q[tau] for tau in self.TAUS}
        self.STREAMS = ("q1", "q4", "q16")
        self._calibrate(S)                                       # first pass: the three q's
        std = {tau: (np.asarray(Q[tau]) - self.mu_s["q%d" % tau]) / self.sd_s["q%d" % tau]
               for tau in self.TAUS}
        S["qr"] = (std[16] - std[1]) if len(Q[1]) else []
        self.STREAMS = AggKurtosisFamily.STREAMS
        self._calibrate(S)                                       # second pass: all four
        # the init loop warmed self.v; restart the online buffer from the true history tail
        self.zb = [float(v) for v in z[-T:]]
        self._setup_buffers()

    def _q(self, buf, z):
        out = {}
        for tau in self.TAUS:
            y = (z - buf[-tau]) / self.sd[tau]
            v = max(self.v[tau], 1e-4)
            out[tau] = min(y ** 4, self.QCAP) / (v * v)
            self.v[tau] += self.A * (y * y - self.v[tau])
        return out

    def update(self, ctx):
        z = min(max(float(ctx.zr), -self.ZC), self.ZC)
        q = self._q(self.zb, z)
        self.zb = self.zb[1:] + [z]
        raw = {"q%d" % tau: q[tau] for tau in self.TAUS}
        raw["qr"] = ((raw["q16"] - self.mu_s["q16"]) / self.sd_s["q16"]
                     - (raw["q1"] - self.mu_s["q1"]) / self.sd_s["q1"])
        return self._emit(raw)


# ------------------------------------------------------------------ 7. narrowband periodicity
class PeriodicityFamily(_Lit5Base):
    """Does a sharp cycle appear or vanish? Sliding DFT (W = 128) of the AR innovations -- whitened,
    so the broad AR spectrum does not masquerade as a peak -- at the 63 interior Fourier frequencies.
    g  : Fisher's (1929) g = max periodogram share (sum of shares = 1)
    pk : log share at the HISTORY's dominant frequency (does the history's own cycle persist?)
    Band energies (temporal, wavvar) smear a narrow peak; g measures its concentration.
    The DFT is updated in O(K) per step and recomputed exactly every W steps (no drift)."""
    name = "period"
    STREAMS = ("g", "pk")
    W = 128
    RC = 6.0
    MAXWIN = 2048

    def init(self, hist):
        _, r = self._setup_innov(hist)
        x = np.clip(r, -self.RC, self.RC)
        W = self.W
        K = W // 2 - 1
        self.k = np.arange(1, W // 2)
        self.rot = np.exp(2j * np.pi * self.k / W)
        if x.size >= 2 * W + 32:
            M = np.lib.stride_tricks.sliding_window_view(x, W)[-self.MAXWIN:]
            P = np.abs(np.fft.rfft(M, axis=1)[:, 1:W // 2]) ** 2
            sh = P / np.maximum(P.sum(axis=1, keepdims=True), 1e-12)
            # pick the dominant frequency on the EARLIER half of the windows and calibrate pk on the
            # later half: picking and calibrating on the same windows is a winner's curse (on a
            # peak-free history the chosen bin looks stronger than it is, so pk drifts negative)
            h = sh.shape[0] // 2
            self.kstar = int(np.argmax(sh[:h].mean(axis=0)))
            self._calibrate({"g": sh.max(axis=1), "pk": np.log(np.maximum(sh[h:, self.kstar], 1e-12))})
        else:                       # short history: white-noise (exponential ordinates) approximations
            self.kstar = 0
            self.mu_s = {"g": (math.log(K) + 0.5772) / K, "pk": -math.log(K) - 0.5772}
            self.sd_s = {"g": 1.2825 / K, "pk": 1.2825}
        seed = x[-W:] if x.size >= W else np.r_[np.zeros(W - x.size), x]
        self.xb = np.ascontiguousarray(seed, dtype=np.float64)   # ring buffer, oldest at self.pos
        self.pos = 0
        self.X = np.fft.rfft(self.xb)[1:W // 2]
        self.since = 0
        self._setup_buffers()

    def update(self, ctx):
        x = min(max(self._innov(float(ctx.zr)), -self.RC), self.RC)
        old = self.xb[self.pos]
        self.xb[self.pos] = x
        self.pos = (self.pos + 1) % self.W
        self.since += 1
        if self.since >= self.W:                                  # exact refresh
            self.X = np.fft.rfft(np.roll(self.xb, -self.pos))[1:self.W // 2]
            self.since = 0
        else:
            self.X = self.rot * (self.X - old + x)
        P = self.X.real ** 2 + self.X.imag ** 2
        tot = float(P.sum())
        if tot <= 1e-12:
            raw = {"g": self.mu_s["g"], "pk": self.mu_s["pk"]}
        else:
            raw = {"g": float(P.max()) / tot, "pk": math.log(max(float(P[self.kstar]) / tot, 1e-12))}
        return self._emit(raw)


# ------------------------------------------------------------------ 8. recording process
class RecordingProcessFamily(_Lit5Base):
    """Did the way the series is MEASURED change -- rounding / tick size, stale repeated values?
    Every other feature treats values as continuous. Equal raw values give bit-equal zr, so ties
    are exact. (Lesmond, Ogden & Trzcinka 1999 on zero returns; Bandi, Pirino & Reno 2017 on
    staleness.)
    tie  : 1[z_t == z_{t-1}]                    rep : 1[z_t equals any of the previous 5 values]
    fine : 1[0 < |dz| < 1st percentile of the history's nonzero |dz|]   (finer grid than before)
    Dead (constant) on continuous data -- check the tie rate before spending a CV run on it."""
    name = "recproc"
    STREAMS = ("tie", "rep", "fine")

    def init(self, hist):
        z = _zr_hist(hist)
        d = np.abs(np.diff(z))
        nz = d[d > 0]
        self.g1 = float(np.quantile(nz, 0.01)) if nz.size >= 20 else 0.0
        rep = np.zeros(max(z.size - 5, 0), dtype=bool)
        for k in range(1, 6):
            rep |= z[5:] == z[5 - k:z.size - k]
        self._calibrate({"tie": (d == 0).astype(float),
                         "rep": rep.astype(float),
                         "fine": ((d > 0) & (d < self.g1)).astype(float)})
        self.zb = [float(v) for v in z[-5:]]
        self._setup_buffers()

    def update(self, ctx):
        z = float(ctx.zr)
        d = abs(z - self.zb[-1])
        raw = {"tie": 1.0 if d == 0.0 else 0.0,
               "rep": 1.0 if z in self.zb else 0.0,
               "fine": 1.0 if 0.0 < d < self.g1 else 0.0}
        self.zb = self.zb[1:] + [z]
        return self._emit(raw)


LIT5 = (IrreversibilityFamily, JumpVariationFamily, NonlinearMeanFamily, LevelVarianceFamily,
        ExtremeClusterFamily, AggKurtosisFamily, PeriodicityFamily, RecordingProcessFamily)
print("lit5:", {c.name: len(c().names()) for c in LIT5})


# %% [2] pre-check for recproc: is there anything to measure? (reads the raw training series)
_tie = []
for _did, _xh, _xo, _tau in list(train_data)[:2000]:
    _x = np.asarray(_xh, dtype=np.float64)
    _tie.append(float(np.mean(np.diff(_x) == 0)) if _x.size > 1 else 0.0)
_tie = np.array(_tie)
print(f"history tie rate over {_tie.size} series: median {np.median(_tie):.4f} | "
      f"share of series with > 1% ties: {np.mean(_tie > 0.01):.1%}")
print("-> test recproc" if np.mean(_tie > 0.01) > 0.05 else "-> skip recproc: the data is effectively continuous")



# %% [4] jumps vs continuous variation


# %% [5] nonlinear mean dynamics
test_family(NonlinearMeanFamily)

# %% [6] variance tied to the level
test_family(LevelVarianceFamily)

# %% [7] clustering of extremes
test_family(ExtremeClusterFamily)

# %% [8] tail weight vs aggregation horizon
test_family(AggKurtosisFamily)

# %% [9] narrowband periodicity
test_family(PeriodicityFamily)

# %% [10] recording process (only if cell 2 said so)
test_family(RecordingProcessFamily)

lit5: {'irrev': 20, 'jump': 15, 'nlmean': 20, 'lvlvar': 10, 'xclust': 20, 'aggkurt': 20, 'period': 10, 'recproc': 15}
history tie rate over 2000 series: median 0.0000 | share of series with > 1% ties: 0.0%
-> skip recproc: the data is effectively continuous
== jump: 15 features -> ['jump_jshare_w32', 'jump_jshare_w128', 'jump_jshare_w512', 'jump_jshare_ew', 'jump_jshare_exp', 'jump_jflag3_w32', 'jump_jflag3_w128', 'jump_jflag3_w512', 'jump_jflag3_ew', 'jump_jflag3_exp', 'jump_jflag4_w32', 'jump_jflag4_w128', 'jump_jflag4_w512', 'jump_jflag4_ew', 'jump_jflag4_exp']
   smoke test OK
build_dataset: fork Pool over 10000 series (chunksize 8)
   extracted (5036517, 15) in 0.3 min
   reference e2_w256 = 0.56155


,feature,ts_auc_abs,ts_auc_raw,corr_with_e2_w256
9,jump_jflag3_exp,0.5304,0.5304,0.1113
7,jump_jflag3_w512,0.5297,0.5297,0.1052
14,jump_jflag4_exp,0.5228,0.5228,0.0873
12,jump_jflag4_w512,0.5227,0.5227,0.0870
6,jump_jflag3_w128,0.5192,0.5192,0.0449
8,jump_jflag3_ew,0.5191,0.5191,0.0269
4,jump_jshare_exp,0.5177,0.5177,0.0603
2,jump_jshare_w512,0.5170,0.5170,0.0570
13,jump_jflag4_ew,0.5125,0.5125,0.0349
11,jump_jflag4_w128,0.5114,0.5114,0.0508


[base111] cached | mean 0.62928


TypeError: list indices must be integers or slices, not str

In [39]:
def _stack(rows, extra_arrays):
    parts = [np.asarray(X_all[rows])[:, PROD]]
    for a in extra_arrays:
        parts.append(a[rows])
    return np.ascontiguousarray(np.hstack(parts)) if len(parts) > 1 else np.ascontiguousarray(parts[0])

In [40]:
test_family(JumpVariationFamily)

== jump: 15 features -> ['jump_jshare_w32', 'jump_jshare_w128', 'jump_jshare_w512', 'jump_jshare_ew', 'jump_jshare_exp', 'jump_jflag3_w32', 'jump_jflag3_w128', 'jump_jflag3_w512', 'jump_jflag3_ew', 'jump_jflag3_exp', 'jump_jflag4_w32', 'jump_jflag4_w128', 'jump_jflag4_w512', 'jump_jflag4_ew', 'jump_jflag4_exp']
   smoke test OK
   loaded cache (5036517, 15)
   reference e2_w256 = 0.56155


,feature,ts_auc_abs,ts_auc_raw,corr_with_e2_w256
9,jump_jflag3_exp,0.5304,0.5304,0.1113
7,jump_jflag3_w512,0.5297,0.5297,0.1052
14,jump_jflag4_exp,0.5228,0.5228,0.0873
12,jump_jflag4_w512,0.5227,0.5227,0.0870
6,jump_jflag3_w128,0.5192,0.5192,0.0449
8,jump_jflag3_ew,0.5191,0.5191,0.0269
4,jump_jshare_exp,0.5177,0.5177,0.0603
2,jump_jshare_w512,0.5170,0.5170,0.0570
13,jump_jflag4_ew,0.5125,0.5125,0.0349
11,jump_jflag4_w128,0.5114,0.5114,0.0508


[base111] cached | mean 0.62928
  [base111+jump] fold 0 swap=False: 0.61479
  [base111+jump] fold 0 swap=True: 0.61712
  [base111+jump] fold 1 swap=False: 0.62197
  [base111+jump] fold 1 swap=True: 0.65154
  [base111+jump] fold 2 swap=False: 0.64026
  [base111+jump] fold 2 swap=True: 0.62099
[base111+jump] mean 0.62778

jump on base111 (111 feats): delta -0.00150 | se 0.00222 | -0.7 sigma | better 2/6
VERDICT: REJECT -> forget('jump')
(total 803s)


,feature,ts_auc_abs,ts_auc_raw,corr_with_e2_w256
0,jump_jshare_w32,0.500033,0.499967,0.011522
1,jump_jshare_w128,0.507097,0.507097,0.030980
2,jump_jshare_w512,0.517008,0.517008,0.056967
3,jump_jshare_ew,0.505141,0.505141,0.022528
4,jump_jshare_exp,0.517718,0.517718,0.060297
5,jump_jflag3_w32,0.504390,0.504390,0.012330
6,jump_jflag3_w128,0.519181,0.519181,0.044882
7,jump_jflag3_w512,0.529676,0.529676,0.105211
8,jump_jflag3_ew,0.519150,0.519150,0.026947
9,jump_jflag3_exp,0.530403,0.530403,0.111310


In [41]:
# %% [5] nonlinear mean dynamics
test_family(NonlinearMeanFamily)

# %% [6] variance tied to the level
test_family(LevelVarianceFamily)

# %% [7] clustering of extremes
test_family(ExtremeClusterFamily)

# %% [8] tail weight vs aggregation horizon
test_family(AggKurtosisFamily)

# %% [9] narrowband periodicity
test_family(PeriodicityFamily)

# %% [10] recording process (only if cell 2 said so)
test_family(RecordingProcessFamily)

== nlmean: 20 features -> ['nlmean_quad_w32', 'nlmean_quad_w128', 'nlmean_quad_w512', 'nlmean_quad_ew', 'nlmean_quad_exp', 'nlmean_bil_w32', 'nlmean_bil_w128', 'nlmean_bil_w512', 'nlmean_bil_ew', 'nlmean_bil_exp', 'nlmean_cub_w32', 'nlmean_cub_w128', 'nlmean_cub_w512', 'nlmean_cub_ew', 'nlmean_cub_exp', 'nlmean_thr_w32', 'nlmean_thr_w128', 'nlmean_thr_w512', 'nlmean_thr_ew', 'nlmean_thr_exp']
   smoke test OK
build_dataset: fork Pool over 10000 series (chunksize 8)
   extracted (5036517, 20) in 0.6 min
   reference e2_w256 = 0.56155


,feature,ts_auc_abs,ts_auc_raw,corr_with_e2_w256
14,nlmean_cub_exp,0.5103,0.4897,0.1484
16,nlmean_thr_w128,0.5102,0.4898,0.0817
11,nlmean_cub_w128,0.5100,0.4900,0.1223
12,nlmean_cub_w512,0.5099,0.4901,0.1526
13,nlmean_cub_ew,0.5095,0.4905,0.1072
17,nlmean_thr_w512,0.5094,0.4906,0.1004
18,nlmean_thr_ew,0.5094,0.4906,0.0739
19,nlmean_thr_exp,0.5093,0.4907,0.1023
15,nlmean_thr_w32,0.5063,0.4937,0.0454
9,nlmean_bil_exp,0.5054,0.5054,0.0083


[base111] cached | mean 0.62928
  [base111+nlmean] fold 0 swap=False: 0.61953
  [base111+nlmean] fold 0 swap=True: 0.61621
  [base111+nlmean] fold 1 swap=False: 0.62683
  [base111+nlmean] fold 1 swap=True: 0.65238
  [base111+nlmean] fold 2 swap=False: 0.62723
  [base111+nlmean] fold 2 swap=True: 0.61725
[base111+nlmean] mean 0.62657

nlmean on base111 (111 feats): delta -0.00271 | se 0.00057 | -4.8 sigma | better 0/6
VERDICT: REJECT -> forget('nlmean')
(total 888s)
== lvlvar: 10 features -> ['lvlvar_bp_w32', 'lvlvar_bp_w128', 'lvlvar_bp_w512', 'lvlvar_bp_ew', 'lvlvar_bp_exp', 'lvlvar_wh_w32', 'lvlvar_wh_w128', 'lvlvar_wh_w512', 'lvlvar_wh_ew', 'lvlvar_wh_exp']
   smoke test OK
build_dataset: fork Pool over 10000 series (chunksize 8)
   extracted (5036517, 10) in 0.4 min
   reference e2_w256 = 0.56155


,feature,ts_auc_abs,ts_auc_raw,corr_with_e2_w256
7,lvlvar_wh_w512,0.5139,0.5139,0.7059
6,lvlvar_wh_w128,0.5135,0.5135,0.6794
9,lvlvar_wh_exp,0.5134,0.5134,0.6923
8,lvlvar_wh_ew,0.5132,0.5132,0.6518
5,lvlvar_wh_w32,0.5077,0.5077,0.5375
2,lvlvar_bp_w512,0.5054,0.4946,0.0206
4,lvlvar_bp_exp,0.5051,0.4949,0.0186
3,lvlvar_bp_ew,0.5048,0.4952,0.0404
1,lvlvar_bp_w128,0.5044,0.4956,0.0356
0,lvlvar_bp_w32,0.5029,0.4971,0.0348


[base111] cached | mean 0.62928
  [base111+lvlvar] fold 0 swap=False: 0.61925
  [base111+lvlvar] fold 0 swap=True: 0.61605
  [base111+lvlvar] fold 1 swap=False: 0.62761
  [base111+lvlvar] fold 1 swap=True: 0.65237
  [base111+lvlvar] fold 2 swap=False: 0.63424
  [base111+lvlvar] fold 2 swap=True: 0.61662
[base111+lvlvar] mean 0.62769

lvlvar on base111 (111 feats): delta -0.00159 | se 0.00089 | -1.8 sigma | better 1/6
VERDICT: REJECT -> forget('lvlvar')
(total 759s)
== xclust: 20 features -> ['xclust_c1_w32', 'xclust_c1_w128', 'xclust_c1_w512', 'xclust_c1_ew', 'xclust_c1_exp', 'xclust_c5_w32', 'xclust_c5_w128', 'xclust_c5_w512', 'xclust_c5_ew', 'xclust_c5_exp', 'xclust_cu_w32', 'xclust_cu_w128', 'xclust_cu_w512', 'xclust_cu_ew', 'xclust_cu_exp', 'xclust_cl_w32', 'xclust_cl_w128', 'xclust_cl_w512', 'xclust_cl_ew', 'xclust_cl_exp']
   smoke test OK
build_dataset: fork Pool over 10000 series (chunksize 8)
   extracted (5036517, 20) in 0.7 min
   reference e2_w256 = 0.56155


,feature,ts_auc_abs,ts_auc_raw,corr_with_e2_w256
9,xclust_c5_exp,0.5332,0.5332,0.4205
7,xclust_c5_w512,0.5324,0.5324,0.4087
4,xclust_c1_exp,0.5249,0.5249,0.3398
2,xclust_c1_w512,0.5241,0.5241,0.3322
6,xclust_c5_w128,0.5235,0.5235,0.1905
10,xclust_cu_w32,0.5226,0.4774,0.0470
15,xclust_cl_w32,0.5210,0.4790,0.0106
13,xclust_cu_ew,0.5188,0.4812,0.0632
8,xclust_c5_ew,0.5175,0.5175,0.1403
1,xclust_c1_w128,0.5156,0.5156,0.1582


... (6 outputs left out)

   reference e2_w256 = 0.56155


,feature,ts_auc_abs,ts_auc_raw,corr_with_e2_w256
12,recproc_fine_w512,0.5058,0.4942,0.1739
14,recproc_fine_exp,0.5057,0.4943,0.1682
11,recproc_fine_w128,0.5054,0.4946,0.1337
13,recproc_fine_ew,0.5050,0.4950,0.1204
10,recproc_fine_w32,0.5023,0.4977,0.0708
7,recproc_rep_w512,0.5003,0.4997,0.0019
8,recproc_rep_ew,0.5003,0.4997,0.0023
9,recproc_rep_exp,0.5003,0.4997,0.0020
6,recproc_rep_w128,0.5002,0.4998,0.0025
5,recproc_rep_w32,0.5002,0.4998,0.0015


[base111] cached | mean 0.62928
  [base111+recproc] fold 0 swap=False: 0.62114
  [base111+recproc] fold 0 swap=True: 0.61842


KeyboardInterrupt: 

In [42]:
# %% [markdown]
# ## Step 0: is there linear signal the LightGBM + CatBoost blend leaves on the table?
# For each of the 6 fold-halves (fold seed 11, the base111 bank):
#   1. train the production blend on the training series: LightGBM (3 seeds) + CatBoost,
#      both early-stopped on the STOP half, blended 0.7 / 0.3 as in submission.ipynb
#   2. on the STOP half, fit a ridge logistic regression WITH THE BLEND AS A FIXED OFFSET:
#         logit P(y) = logit(p_blend) + b0 + X beta        (X = the 111 features, standardized)
#      beta can only learn what the blend's logit does NOT already contain, and only linearly.
#      lambda is picked by a 2-fold split of the stop half by series.
#   3. on the SCORE half (never used for anything above), compare TS-AUC of
#         logit(p_blend) + X beta     vs     p_blend
# H0: beta adds no within-t ranking information.  Reject (= linear signal left) if the paired
# delta over the 6 fold-halves is > 2*se and better on >= 5/6 -- the usual add rule.
# Also reported: the delta by time bucket (is it an early-t effect?) and the features beta leans
# on, with how many fold-halves agree on the sign.
#
# The blend's tree counts are tuned on the stop half, so p_blend fits that half slightly better
# than unseen data: the probe sees a little LESS leftover structure than exists -> conservative.
#
# Out-of-fold predictions are saved per fold-half (cache/oof_b111_s11_*.npz) and reused on re-runs,
# and cell 5 uses them to re-check the 0.7 / 0.3 blend weight at no extra training cost.
#
# Prereqs: section 0, 1, 2 (X_all / y_all / t_all / g_all, build_folds, ts_auc, STRIDE_ALL,
# LGB_PARAMS, NUM_ROUNDS). Self-contained otherwise; does not use _cv or test_family.

# %% [1] settings
USE_CAT = True                  # False = probe the LightGBM part only (much faster)
CAT_PARAMS = dict(grow_policy="Depthwise", learning_rate=0.028354684013833883, depth=8,
                  l2_leaf_reg=12.76103092455787, rsm=0.5758405578310992,
                  random_strength=2.5835125442520925, subsample=0.7933098848053978,
                  min_data_in_leaf=64, bootstrap_type="Bernoulli")      # = submission.ipynb
CAT_WEIGHT = 0.3                # p = (1 - w) * mean(lgb seeds) + w * catboost   (= submission)
FOLD_SEED = 11
LGB_SEEDS = (0, 1, 2)
PATIENCE_LGB, PATIENCE_CAT = 100, 100
LAMBDAS = (1e-4, 1e-3, 1e-2, 1e-1)
FIT_ROWS = 400_000              # cap on stop-half rows used to fit the probe
BUCKETS = ((0, 64), (64, 256), (256, 10 ** 9))


# %% [2] base111 bank
import os, json, gc, time
import numpy as np, pandas as pd, lightgbm as lgb
from scipy.optimize import minimize
from scipy.special import expit
from catboost import CatBoostClassifier

SHIPPED = ["z2", "e2", "tail", "time", "ftest", "bartlett", "levene", "lmom", "sigtuk",
           "wass", "energy", "mmd", "entropy", "bocpd",
           "garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]
ALL_EXTRA = ["garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]
BANK_EXTRA = ["garch", "argarch", "tgarch", "recres", "hmm"]
B3_ALL = [f"{s}_e_w{W}" for s in ("ftest", "bartlett", "levene", "sigtuk", "lmom_lscale")
          for W in (32, 64, 256)]

_FN = {f.name: list(f.names()) for f in REGISTRY if f.name in SHIPPED}
assert set(_FN) == set(SHIPPED), f"REGISTRY lacks {set(SHIPPED) - set(_FN)} -- re-run section 0"
X_all = np.load(f"{CACHE}/X_all.npy", mmap_mode="r")
BASE_NAMES = json.load(open(f"{CACHE}/names.json"))
_cache_names = [n for fam in SHIPPED if fam not in ALL_EXTRA for n in _FN[fam]]
P_IDX = [BASE_NAMES.index(n) for n in _cache_names if n not in B3_ALL]
assert len(P_IDX) == 58


def _load(fam):
    with np.load(f"{CACHE}/new_{fam}.npz") as Z:
        nm = [str(s) for s in Z["names"]]
        X = np.asarray(Z["X"], dtype=np.float32)
    want = _FN[fam]
    assert not [n for n in want if n not in nm], f"{fam}: cache lacks shipped columns"
    return np.ascontiguousarray(X[:, [nm.index(n) for n in want]]), want


B_ARR, B_NAMES = [], [BASE_NAMES[i] for i in P_IDX]
for fam in BANK_EXTRA:
    a, nm = _load(fam)
    B_ARR.append(a); B_NAMES += nm
assert len(B_NAMES) == 111
gc.collect()


def stack(rows):
    parts = [np.asarray(X_all[rows])[:, P_IDX]] + [a[rows] for a in B_ARR]
    return np.ascontiguousarray(np.hstack(parts), dtype=np.float32)


FOLDS = build_folds(FOLD_SEED)
HALVES = [(fi, swap) for fi in range(len(FOLDS)) for swap in (False, True)]


def halves(fi, swap):
    F = FOLDS[fi]
    return (F["sc"], F["st"]) if swap else (F["st"], F["sc"])      # (stop rows, score rows)


# %% [3] out-of-fold blend predictions (resumable, one file per fold-half)
def oof_path(fi, swap):
    return f"{CACHE}/oof_b111_s{FOLD_SEED}_f{fi}{'b' if swap else 'a'}{'_cat' if USE_CAT else ''}.npz"


for fi, swap in HALVES:
    path = oof_path(fi, swap)
    if os.path.exists(path):
        print(f"fold {fi} swap={swap}: cached"); continue
    t0 = time.time()
    F = FOLDS[fi]
    si, ei = halves(fi, swap)
    Xtr, Xs, Xe = stack(F["tr"]), stack(si), stack(ei)
    ytr, w = y_all[F["tr"]], STRIDE_ALL[F["tr"]].astype(np.float32)
    pl_s, pl_e = [], []
    for s in LGB_SEEDS:
        p = dict(LGB_PARAMS)
        p.update(objective="binary", metric="binary_logloss",
                 seed=s, bagging_seed=s + 100, feature_fraction_seed=s + 200)
        dtr = lgb.Dataset(Xtr, label=ytr, weight=w, feature_name=B_NAMES)
        dva = lgb.Dataset(Xs, label=y_all[si], reference=dtr)
        b = lgb.train(p, dtr, num_boost_round=NUM_ROUNDS, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(PATIENCE_LGB, verbose=False)])
        pl_s.append(b.predict(Xs, num_iteration=b.best_iteration))
        pl_e.append(b.predict(Xe, num_iteration=b.best_iteration))
        del dtr, dva, b; gc.collect()
    out = dict(si=si, ei=ei, lgb_s=np.mean(pl_s, axis=0), lgb_e=np.mean(pl_e, axis=0))
    if USE_CAT:
        cat = CatBoostClassifier(**CAT_PARAMS, iterations=NUM_ROUNDS, early_stopping_rounds=PATIENCE_CAT,
                                 random_seed=0, thread_count=8, verbose=0, allow_writing_files=False)
        cat.fit(Xtr, ytr, sample_weight=w, eval_set=(Xs, y_all[si]))
        out["cat_s"] = cat.predict_proba(Xs)[:, 1]
        out["cat_e"] = cat.predict_proba(Xe)[:, 1]
        del cat
    np.savez_compressed(path, **out)
    del Xtr, Xs, Xe, pl_s, pl_e, out; gc.collect()
    print(f"fold {fi} swap={swap}: saved | {(time.time() - t0) / 60:.1f} min")


def load_oof(fi, swap, w_cat=CAT_WEIGHT):
    Z = dict(np.load(oof_path(fi, swap)))
    if USE_CAT:
        Z["p_s"] = (1 - w_cat) * Z["lgb_s"] + w_cat * Z["cat_s"]
        Z["p_e"] = (1 - w_cat) * Z["lgb_e"] + w_cat * Z["cat_e"]
    else:
        Z["p_s"], Z["p_e"] = Z["lgb_s"], Z["lgb_e"]
    return Z


# %% [4] the probe
def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def fit_offset_ridge(X, y, off, w, lam):
    """argmin  mean_w[ log(1+e^eta) - y eta ] + lam/2 |beta|^2,  eta = off + b0 + X beta."""
    w = w / w.sum()
    d = X.shape[1]

    def f(th):
        eta = off + th[0] + X @ th[1:]
        g = expit(eta) - y
        val = float(w @ (np.logaddexp(0.0, eta) - y * eta)) + 0.5 * lam * float(th[1:] @ th[1:])
        wg = w * g
        return val, np.r_[wg.sum(), X.T @ wg + lam * th[1:]]

    res = minimize(f, np.zeros(d + 1), jac=True, method="L-BFGS-B", options=dict(maxiter=1000))
    return res.x


def tsauc_buckets(y, t, s):
    out = {"all": ts_auc(y, t, s)}
    for lo, hi in BUCKETS:
        m = (t >= lo) & (t < hi)
        out[f"t{lo}-{hi if hi < 10 ** 9 else 'inf'}"] = ts_auc(y[m], t[m], s[m]) if m.sum() > 100 else np.nan
    return out


rows, betas = [], []
for fi, swap in HALVES:
    t0 = time.time()
    Z = load_oof(fi, swap)
    si, ei = Z["si"], Z["ei"]
    # fitting rows: the stop half at training stride, stride-weighted, capped
    keep = (t_all[si] % STRIDE_ALL[si]) == 0
    fit = si[keep]
    off_fit = _logit(Z["p_s"][keep])
    if fit.size > FIT_ROWS:
        pick = np.sort(np.random.default_rng(fi * 2 + swap).choice(fit.size, FIT_ROWS, replace=False))
        fit, off_fit = fit[pick], off_fit[pick]
    Xf = stack(fit).astype(np.float64)
    mu, sd = Xf.mean(axis=0), Xf.std(axis=0)
    sd[sd < 1e-9] = 1.0
    Xf = (Xf - mu) / sd
    yf, wf, tf = y_all[fit].astype(np.float64), STRIDE_ALL[fit].astype(np.float64), t_all[fit]

    # lambda: 2-fold split of the stop half BY SERIES
    part = (g_all[fit] % 2) == 0
    score = {}
    for lam in LAMBDAS:
        s = []
        for a in (part, ~part):
            th = fit_offset_ridge(Xf[a], yf[a], off_fit[a], wf[a], lam)
            b = ~a
            s.append(ts_auc(yf[b], tf[b], off_fit[b] + Xf[b] @ th[1:]))
        score[lam] = np.mean(s)
    lam = max(score, key=score.get)
    th = fit_offset_ridge(Xf, yf, off_fit, wf, lam)
    betas.append(th[1:])
    del Xf; gc.collect()

    # score half: never touched above
    Xe = (stack(ei).astype(np.float64) - mu) / sd
    ye, te = y_all[ei], t_all[ei]
    base = tsauc_buckets(ye, te, Z["p_e"])
    probe = tsauc_buckets(ye, te, _logit(Z["p_e"]) + Xe @ th[1:])
    del Xe; gc.collect()
    rows.append(dict(fold=fi, swap=swap, lam=lam,
                     **{f"base_{k}": v for k, v in base.items()},
                     **{f"probe_{k}": v for k, v in probe.items()}))
    print(f"fold {fi} swap={swap}: lambda {lam:g} | blend {base['all']:.5f} -> +probe {probe['all']:.5f} "
          f"({probe['all'] - base['all']:+.5f}) | {(time.time() - t0) / 60:.1f} min")

R = pd.DataFrame(rows)
print(f"\n=== Step 0 | base111 | fold seed {FOLD_SEED} | blend = {'lgb+cat' if USE_CAT else 'lgb only'}")
for k in ["all"] + [c[5:] for c in R.columns if c.startswith("base_t")]:
    d = (R[f"probe_{k}"] - R[f"base_{k}"]).dropna().to_numpy()
    if d.size < 2:
        continue
    m, se = d.mean(), d.std(ddof=1) / np.sqrt(d.size)
    print(f"  {k:10s}: delta {m:+.5f} | se {se:.5f} | {m / se if se else 0:+.1f} sigma | better {(d > 0).sum()}/{d.size}")
d = (R["probe_all"] - R["base_all"]).to_numpy()
m, se = d.mean(), d.std(ddof=1) / np.sqrt(d.size)
LINEAR_LEFT = m > 2 * se and (d > 0).sum() >= d.size - 1
print("VERDICT:", "LINEAR SIGNAL LEFT -> worth trying fixes 2 and 3 (linear base / linear_tree, contrasts)"
      if LINEAR_LEFT else "NO usable linear signal left -> skip linear stacking; spend the effort on fixes 1 and 4")

B = np.array(betas)
coef = pd.DataFrame({"feature": B_NAMES, "beta_mean": B.mean(axis=0),
                     "sign_agree": [max((B[:, j] > 0).sum(), (B[:, j] < 0).sum()) for j in range(B.shape[1])]})
coef["abs"] = coef.beta_mean.abs()
print("\nfeatures the probe leans on (standardized beta, averaged over fold-halves):")
print(coef.sort_values("abs", ascending=False).head(15).drop(columns="abs").round(4).to_string(index=False))


# %% [5] bonus, free: is 0.7 / 0.3 still the right blend weight on base111?
if USE_CAT:
    GRID = np.round(np.arange(0.0, 1.0001, 0.05), 2)
    res = []
    for fi, swap in HALVES:
        Z = load_oof(fi, swap)
        ys, ts_, ye, te = y_all[Z["si"]], t_all[Z["si"]], y_all[Z["ei"]], t_all[Z["ei"]]
        pick = max(GRID, key=lambda w: ts_auc(ys, ts_, (1 - w) * Z["lgb_s"] + w * Z["cat_s"]))
        res.append(dict(w_best=pick,
                        tuned=ts_auc(ye, te, (1 - pick) * Z["lgb_e"] + pick * Z["cat_e"]),
                        fixed=ts_auc(ye, te, (1 - CAT_WEIGHT) * Z["lgb_e"] + CAT_WEIGHT * Z["cat_e"]),
                        lgb=ts_auc(ye, te, Z["lgb_e"]), cat=ts_auc(ye, te, Z["cat_e"])))
    W = pd.DataFrame(res)
    print(W.round(5).to_string(index=False))
    d = (W.tuned - W.fixed).to_numpy()
    print(f"\ncat weight chosen on the stop half: {W.w_best.tolist()} | tuned vs {CAT_WEIGHT}: "
          f"{d.mean():+.5f} | se {d.std(ddof=1) / np.sqrt(d.size):.5f} | better {(d > 0).sum()}/{d.size}")

fold 0: train 1,020,112 | stop 848,058 | score 849,407
fold 1: train 1,026,629 | stop 813,776 | score 845,268
fold 2: train 1,022,269 | stop 833,755 | score 846,253
fold 0 swap=False: saved | 5.8 min
fold 0 swap=True: saved | 3.9 min
fold 1 swap=False: saved | 6.8 min
fold 1 swap=True: saved | 6.4 min
fold 2 swap=False: saved | 6.2 min
fold 2 swap=True: saved | 5.3 min
fold 0 swap=False: lambda 0.1 | blend 0.62132 -> +probe 0.62072 (-0.00060) | 0.2 min
fold 0 swap=True: lambda 0.1 | blend 0.61958 -> +probe 0.62316 (+0.00358) | 0.2 min
fold 1 swap=False: lambda 0.1 | blend 0.62927 -> +probe 0.62380 (-0.00547) | 0.1 min
fold 1 swap=True: lambda 0.1 | blend 0.65622 -> +probe 0.64853 (-0.00770) | 0.1 min
fold 2 swap=False: lambda 0.1 | blend 0.63254 -> +probe 0.63119 (-0.00135) | 0.2 min
fold 2 swap=True: lambda 0.1 | blend 0.61997 -> +probe 0.61949 (-0.00048) | 0.1 min

=== Step 0 | base111 | fold seed 11 | blend = lgb+cat
  all       : delta -0.00200 | se 0.00164 | -1.2 sigma | better 1/

In [1]:
# %% [markdown]
# ## Blend check + per-t offset, on base111 (fold seed 11)
# Cell A  setup: the base111 bank (reuses test_family's arrays if they are loaded), folds, helpers.
# Cell B  CV: LightGBM (3 seeds) + CatBoost at 0.7/0.3  vs  LightGBM (3 seeds) alone.
#         Uses the out-of-fold predictions saved by the step-0 probe (cache/oof_b111_s11_*_cat.npz)
#         and trains only the fold-halves that are missing. Also pools in the seed-23 confirmation
#         run (cache/confirm_results.json), which already scored lgb and blend on the same bank.
# Cell C  CV: LightGBM with init_score = logit(base rate at t)  vs  plain LightGBM, same folds/seeds.
#         The offset is constant within each t, so it cannot change any within-t ranking by itself;
#         it only changes what the trees spend their splits on. Predictions are saved for later
#         blending (cache/oof_b111_s11_*_off.npz).
# Prereqs: sections 0-2 (X_all, y_all, t_all, g_all, build_folds, ts_auc, STRIDE_ALL, LGB_PARAMS,
# NUM_ROUNDS, REGISTRY, CACHE). Resumable: every fold-half is saved as soon as it finishes.

# %% [A] setup
import os, json, gc, time
import numpy as np, pandas as pd, lightgbm as lgb
from catboost import CatBoostClassifier

FOLD_SEED = 11
LGB_SEEDS = (0, 1, 2)
PATIENCE = 100
CAT_WEIGHT = 0.3
CAT_PARAMS = dict(grow_policy="Depthwise", learning_rate=0.028354684013833883, depth=8,
                  l2_leaf_reg=12.76103092455787, rsm=0.5758405578310992,
                  random_strength=2.5835125442520925, subsample=0.7933098848053978,
                  min_data_in_leaf=64, bootstrap_type="Bernoulli")      # = submission.ipynb
BUCKETS = ((0, 64), (64, 256), (256, 10 ** 9))

_SHIPPED = ["z2", "e2", "tail", "time", "ftest", "bartlett", "levene", "lmom", "sigtuk",
            "wass", "energy", "mmd", "entropy", "bocpd",
            "garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]
_ALL_EXTRA = ["garch", "argarch", "tgarch", "gjr", "recres", "wavvar", "hmm"]
_BANK = ["garch", "argarch", "tgarch", "recres", "hmm"]
_B3 = {f"{s}_e_w{W}" for s in ("ftest", "bartlett", "levene", "sigtuk", "lmom_lscale") for W in (32, 64, 256)}

_FN = {f.name: list(f.names()) for f in REGISTRY if f.name in _SHIPPED}
assert set(_FN) == set(_SHIPPED), f"REGISTRY lacks {set(_SHIPPED) - set(_FN)} -- re-run section 0"
X_all = np.load(f"{CACHE}/X_all.npy", mmap_mode="r")
BASE_NAMES = json.load(open(f"{CACHE}/names.json"))
P111 = [BASE_NAMES.index(n) for fam in _SHIPPED if fam not in _ALL_EXTRA
        for n in _FN[fam] if n not in _B3]
assert len(P111) == 58


def _load(fam):
    ex = globals().get("EXTRA")
    if isinstance(ex, dict) and isinstance(ex.get(fam), tuple) and list(ex[fam][1]) == _FN[fam]:
        return ex[fam]                                        # test_family already holds it: no copy
    with np.load(f"{CACHE}/new_{fam}.npz") as Z:
        nm = [str(s) for s in Z["names"]]
        X = np.asarray(Z["X"], dtype=np.float32)
    want = _FN[fam]
    assert not [n for n in want if n not in nm], f"{fam}: cache lacks shipped columns"
    return np.ascontiguousarray(X[:, [nm.index(n) for n in want]]), want


A111, N111 = [], [BASE_NAMES[i] for i in P111]
for fam in _BANK:
    a, nm = _load(fam)
    A111.append(a); N111 += nm
assert len(N111) == 111
gc.collect()


def stack111(rows):
    parts = [np.asarray(X_all[rows])[:, P111]] + [a[rows] for a in A111]
    return np.ascontiguousarray(np.hstack(parts), dtype=np.float32)


FOLDS = build_folds(FOLD_SEED)
HALVES = [(fi, swap) for fi in range(len(FOLDS)) for swap in (False, True)]


def halves(fi, swap):
    F = FOLDS[fi]
    return (F["sc"], F["st"]) if swap else (F["st"], F["sc"])      # (stop rows, score rows)


def oof_path(fi, swap, kind):          # kind "cat" = step-0 file (lgb + cat), "off" = cell C
    return f"{CACHE}/oof_b111_s{FOLD_SEED}_f{fi}{'b' if swap else 'a'}_{kind}.npz"


def train_lgb(Xtr, ytr, w, Xs, ys, Xe, init_tr=None, init_s=None, init_e=None):
    """3-seed LightGBM exactly as the harness trains it; returns mean scores on stop / score rows."""
    ps, pe, its = [], [], []
    for s in LGB_SEEDS:
        p = dict(LGB_PARAMS)
        p.update(objective="binary", metric="binary_logloss",
                 seed=s, bagging_seed=s + 100, feature_fraction_seed=s + 200)
        dtr = lgb.Dataset(Xtr, label=ytr, weight=w, feature_name=N111, init_score=init_tr)
        dva = lgb.Dataset(Xs, label=ys, reference=dtr, init_score=init_s)
        b = lgb.train(p, dtr, num_boost_round=NUM_ROUNDS, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(PATIENCE, verbose=False)])
        if init_tr is None:
            ps.append(b.predict(Xs, num_iteration=b.best_iteration))
            pe.append(b.predict(Xe, num_iteration=b.best_iteration))
        else:                          # booster output excludes init_score: add it back, then squash
            ps.append(1 / (1 + np.exp(-(b.predict(Xs, num_iteration=b.best_iteration, raw_score=True) + init_s))))
            pe.append(1 / (1 + np.exp(-(b.predict(Xe, num_iteration=b.best_iteration, raw_score=True) + init_e))))
        its.append(b.best_iteration)
        del dtr, dva, b; gc.collect()
    return np.mean(ps, axis=0), np.mean(pe, axis=0), int(np.mean(its))


def paired_line(label, d):
    d = np.asarray(d, dtype=np.float64)
    m, se = d.mean(), d.std(ddof=1) / np.sqrt(d.size)
    return (f"{label:34s} {m:+.5f} | se {se:.5f} | {m / se if se else 0:+.1f} sigma | "
            f"better {(d > 0).sum()}/{d.size}")


def by_bucket(y, t, s):
    out = {"all": ts_auc(y, t, s)}
    for lo, hi in BUCKETS:
        m = (t >= lo) & (t < hi)
        out[f"t{lo}-{hi if hi < 10 ** 9 else 'inf'}"] = ts_auc(y[m], t[m], s[m]) if m.sum() > 100 else np.nan
    return out


print(f"base111 ready: {len(N111)} features | fold seed {FOLD_SEED} | "
      f"{'reusing test_family arrays' if isinstance(globals().get('EXTRA'), dict) else 'loaded from cache'}")


# %% [B] blend vs LightGBM alone
for fi, swap in HALVES:
    path = oof_path(fi, swap, "cat")
    if os.path.exists(path):
        continue
    t0 = time.time()
    F = FOLDS[fi]
    si, ei = halves(fi, swap)
    Xtr, Xs, Xe = stack111(F["tr"]), stack111(si), stack111(ei)
    ytr, w = y_all[F["tr"]], STRIDE_ALL[F["tr"]].astype(np.float32)
    lgb_s, lgb_e, _ = train_lgb(Xtr, ytr, w, Xs, y_all[si], Xe)
    cat = CatBoostClassifier(**CAT_PARAMS, iterations=NUM_ROUNDS, early_stopping_rounds=PATIENCE,
                             random_seed=0, thread_count=8, verbose=0, allow_writing_files=False)
    cat.fit(Xtr, ytr, sample_weight=w, eval_set=(Xs, y_all[si]))
    np.savez_compressed(path, si=si, ei=ei, lgb_s=lgb_s, lgb_e=lgb_e,
                        cat_s=cat.predict_proba(Xs)[:, 1], cat_e=cat.predict_proba(Xe)[:, 1])
    del Xtr, Xs, Xe, cat; gc.collect()
    print(f"fold {fi} swap={swap}: trained + saved | {(time.time() - t0) / 60:.1f} min")

rows = []
for fi, swap in HALVES:
    Z = np.load(oof_path(fi, swap, "cat"))
    ei = Z["ei"]
    ye, te = y_all[ei], t_all[ei]
    blend = (1 - CAT_WEIGHT) * Z["lgb_e"] + CAT_WEIGHT * Z["cat_e"]
    rows.append({**{f"lgb_{k}": v for k, v in by_bucket(ye, te, Z["lgb_e"]).items()},
                 **{f"cat_{k}": v for k, v in by_bucket(ye, te, Z["cat_e"]).items()},
                 **{f"blend_{k}": v for k, v in by_bucket(ye, te, blend).items()}})
RB = pd.DataFrame(rows)
print(f"\n=== fold seed {FOLD_SEED}: means  lgb {RB.lgb_all.mean():.5f} | cat {RB.cat_all.mean():.5f} | "
      f"blend {RB.blend_all.mean():.5f}")
print(paired_line("blend - lgb (all t)", RB.blend_all - RB.lgb_all))
for k in [c[4:] for c in RB.columns if c.startswith("lgb_t")]:
    d = (RB[f"blend_{k}"] - RB[f"lgb_{k}"]).dropna()
    if d.size > 1:
        print(paired_line(f"blend - lgb ({k})", d))
print(paired_line("cat - lgb (all t)", RB.cat_all - RB.lgb_all))

d_all = list(RB.blend_all - RB.lgb_all)
cp = f"{CACHE}/confirm_results.json"
if os.path.exists(cp) and "base111" in json.load(open(cp)):
    C = json.load(open(cp))["base111"]
    d23 = np.subtract(C["blend"], C["lgb"])
    print(paired_line("blend - lgb (seed 23 confirm)", d23))
    d_all += list(d23)
    print(paired_line("blend - lgb POOLED (seeds 11+23)", d_all))
d_all = np.asarray(d_all)
m, se = d_all.mean(), d_all.std(ddof=1) / np.sqrt(d_all.size)
print("\nVERDICT:", "CatBoost EARNS its place in the blend" if (m > 2 * se) else
      "CatBoost adds nothing measurable -> consider CAT_WEIGHT = 0 (saves inference time)" if m < 2 * se and m > -2 * se
      else "the blend is WORSE than LightGBM alone -> drop CatBoost")


# %% [C] per-t base-rate offset (init_score) vs plain LightGBM
def base_rate_fn(t, y, w, nbins=60):
    """Smoothed P(y=1 | t) from training rows: stride-weighted positive share in log(t) bins,
    linearly interpolated between bin centres. Returns a function t -> logit(rate)."""
    lt = np.log1p(t.astype(np.float64))
    edges = np.unique(np.quantile(lt, np.linspace(0, 1, nbins + 1)))
    b = np.clip(np.searchsorted(edges, lt, side="right") - 1, 0, edges.size - 2)
    sw = np.bincount(b, w, minlength=edges.size - 1)
    sy = np.bincount(b, w * y, minlength=edges.size - 1)
    sl = np.bincount(b, w * lt, minlength=edges.size - 1)
    ok = sw > 0
    ctr, rate = sl[ok] / sw[ok], (sy[ok] + 0.5) / (sw[ok] + 1.0)
    rate = np.clip(rate, 1e-4, 1 - 1e-4)
    lg = np.log(rate / (1 - rate))
    return lambda tt: np.interp(np.log1p(np.asarray(tt, dtype=np.float64)), ctr, lg)


rows = []
for fi, swap in HALVES:
    path = oof_path(fi, swap, "off")
    base_path = oof_path(fi, swap, "cat")
    F = FOLDS[fi]
    si, ei = halves(fi, swap)
    if not os.path.exists(path):
        t0 = time.time()
        Xtr, Xs, Xe = stack111(F["tr"]), stack111(si), stack111(ei)
        ytr, w = y_all[F["tr"]], STRIDE_ALL[F["tr"]].astype(np.float64)
        off = base_rate_fn(t_all[F["tr"]], ytr.astype(np.float64), w)
        o_tr, o_s, o_e = off(t_all[F["tr"]]), off(t_all[si]), off(t_all[ei])
        off_s, off_e, it_off = train_lgb(Xtr, ytr, w.astype(np.float32), Xs, y_all[si], Xe, o_tr, o_s, o_e)
        out = dict(si=si, ei=ei, off_s=off_s, off_e=off_e, it_off=it_off)
        if not os.path.exists(base_path):          # no step-0 file: train the plain baseline here too
            b_s, b_e, it_b = train_lgb(Xtr, ytr, w.astype(np.float32), Xs, y_all[si], Xe)
            out.update(lgb_s=b_s, lgb_e=b_e, it_base=it_b)
        np.savez_compressed(path, **out)
        del Xtr, Xs, Xe; gc.collect()
        print(f"fold {fi} swap={swap}: offset model {it_off} trees | {(time.time() - t0) / 60:.1f} min")
    Z = dict(np.load(path))
    base_e = Z["lgb_e"] if "lgb_e" in Z else np.load(base_path)["lgb_e"]
    ye, te = y_all[ei], t_all[ei]
    rows.append({**{f"base_{k}": v for k, v in by_bucket(ye, te, base_e).items()},
                 **{f"off_{k}": v for k, v in by_bucket(ye, te, Z["off_e"]).items()},
                 "trees_off": int(Z["it_off"])})
RC = pd.DataFrame(rows)
print(f"\n=== per-t offset | fold seed {FOLD_SEED} | plain {RC.base_all.mean():.5f} -> offset {RC.off_all.mean():.5f} "
      f"| offset model trees per seed: {RC.trees_off.tolist()}")
print(paired_line("offset - plain (all t)", RC.off_all - RC.base_all))
for k in [c[5:] for c in RC.columns if c.startswith("base_t")]:
    d = (RC[f"off_{k}"] - RC[f"base_{k}"]).dropna()
    if d.size > 1:
        print(paired_line(f"offset - plain ({k})", d))
d = (RC.off_all - RC.base_all).to_numpy()
m, se = d.mean(), d.std(ddof=1) / np.sqrt(d.size)
print("\nVERDICT:", "KEEP the offset" if (m > 2 * se and (d > 0).sum() >= d.size - 1) else "REJECT the offset")

NameError: name 'REGISTRY' is not defined